# SPGER0046 Recurso solar e modelagem fotovoltaica

<details>
<summary>Descrição e método</summary>

**Versão 1.2.2.** Farm Solar Array PVDAQ 2107, Califórnia, abril a junho de 2024. Entradas: POA, temperatura do ar e vento locais. Saída: geração AC calculada, comparada com o medidor. NASA POWER não é usado na cadeia principal. Cada subseção possui título, descrição separada e código/resultados. As descrições usam HTML details fechado por padrão; a renderização e o recolhimento por título variam entre JupyterLab e Colab. O relatório HTML exportado conserva os blocos recolhíveis. A execução local não simula a autenticação Google nem a interface Colab.

</details>

**Como usar:** execute as células na ordem. As saídas ficam em outputs, separadas em tabelas, quadros, graficos, fluxogramas e figuras. Cada figura final possui PDF vetorial e PNG a 600 dpi nos temas claro e escuro. Seis DOCX claros reúnem as categorias e o dossiê técnico para redação. A exportação pessoal ao Drive é opcional e exige autenticação no Colab.

**Sumário:** [1.1](#sec_1_1) · [1.2](#sec_1_2) · [1.3](#sec_1_3) · [1.4](#sec_1_4) · [2.1](#sec_2_1) · [2.2](#sec_2_2) · [2.3](#sec_2_3) · [2.4](#sec_2_4) · [3.1](#sec_3_1) · [3.2](#sec_3_2) · [3.3](#sec_3_3) · [3.4](#sec_3_4) · [3.5](#sec_3_5) · [3.6](#sec_3_6) · [4.1](#sec_4_1) · [4.2](#sec_4_2) · [4.3](#sec_4_3) · [4.4](#sec_4_4) · [4.5](#sec_4_5)

## 0 Preparação e sistema de referência

<details>
<summary>Descrição e método</summary>

O modelo representa uma instalação real. Equipamentos: 2.880 módulos e 24 inversores. P₀ de normalização=893 kWp; soma de placa=892,8 kWp; soma STC CEC=891,648 kWp. O equivalente 20×6 não é cabeamento verificado. Nenhuma lacuna diurna é interpolada. Os arquivos oficiais são baixados apenas se o ZIP local estiver ausente, com conferência SHA-256.

</details>

In [ ]:
from pathlib import Path
import sys,subprocess,importlib.util,importlib.metadata,os,json,base64,io,zipfile,hashlib
if importlib.util.find_spec("packaging") is None:
    subprocess.run([sys.executable,"-m","pip","install","packaging"],check=True)
from packaging.requirements import Requirement
requirements={"numpy":"numpy>=2,<3","pandas":"pandas>=2.2,<3","scipy":"scipy>=1.13,<2",
    "matplotlib":"matplotlib>=3.9,<4","pvlib":"pvlib==0.16.1","python-docx":"python-docx>=1.1,<2","Pillow":"Pillow>=10,<13"}
missing=[]
for package,req in requirements.items():
    try:
        if importlib.metadata.version(package) not in Requirement(req).specifier:missing.append(req)
    except importlib.metadata.PackageNotFoundError:missing.append(req)
if missing:subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
support=base64.b64decode("UEsDBBQAAAAIAKS8UF3hhTSZ0wwAAJIiAAAfAAAAYmlibGlvZ3JhZmlhL2Zvcm1hdGFjYW9fYWJudC5weZ1a63LbuBX+76fg+A/IMc2NnG46lcqdUWQ5UZvaHlvZ7VarciARshhTBANSviTdmb5DH6E/OrMzfYq8SZ+k5xwAJHRxkt1kIpHAOQfnjg9QDg8P+6USlShq/uk/n/4tPT7n6adfVtmce8JTYiHUp1+KecYrby5XnoSh+Vp6C8WL+adfKq+SeTbPap5KrxQ58LxfZ6WIDg8PDxYKGEpeL/Ns5mWrUqrau4TXA/OshH1aF9lcpiLlNddcqZw/WJZTOV+vQL92JqqWXInUEgxWoXdZh97Vq5cDmUvlEIpivYpq8VBb2h9Ok/6b0avz5LJ/1X911b98vU1d1Y+5cMivxz++GSbjHy+HDqV8WOWW5gKeh7nYUhEpoqKyRO+Lg4Ozi/OxF3tsnK1E5Z2Le+9KrnjBDq5HfxvCROfkYDz86zj5YXQ6fp0M/oJDL6JnBwcHqVh4laiThSxqX+i1gu6BB3/A0ZdZ4fE896q5ysraW/BVlmewAi9S8PFK3oE5xVKorAaf1Uvg9lBQRUFCGaWCpYzY6AbWkSrhaZqoS+UHREH0QFSqaJEVqf++8Nl9V53hMAs0TbYwZBkYJwuh9XO5HU+57D0UmxWVULX/LNTUdlnlFXyF+nsTxqt5lo1RfxZ6bNkvqvZN8KruVxlvBuYVWcqmW2pEvK5VNotKWRor2BEuEYSk9JPrNku6q+mFdteAWG0Lx+DvE159QCHVh8GGGJmnO942ooKGCByOdNbdhmnLyYaLfMzLUoA4oAvclRx173jOQFl28jsWmLyby2KR3ayVSFJTiD48mOxDWyoxR1NgEATN60wWVWsIjEQlvxHJfZbWy3iw8k86oEwzvBTZzbKm8T9Evw82+GpZJiuubrIixtdcLGr7DvTPN4lnsq7lyqVXKNlhONlkWAqeCpWkWVVDKxOaQmu2kLLemuoEToiVV6LFLSnYoviN4uWy6nplNM8Ft5WDf/RScbuqQz95Nu0Zgif4Ip5nNwU6Pt7tXtHV6NXr8TZHIx46hlrxGtJIVXWSZ4VIIKFQFNj0bGelHT7iqEo+z4qbuNO6IBN5Gm9l2iJPr6Hb5QIqmgicvILqrhVkFrvsvxqydl21LrbFAFmv6XUw3xJjH9+mroEax6nJx6zDesBiEx3HrC5mCAX2jLVJaUeJpK1OswWYnMYXJ6Oh6pa8wi7i01TIUFMWtBSUYTgV4UyE9Rdj+fecwSr7IOLL2sfWH7gTc9zBInUzi+12Bj0R/gab4q2DNKfdEvb0l3MMZM5C9lKmj94YPOL2mTJuTZwg03QnB1razyfin95ej0dnP/bKraSJvnUk7M3DTnTybYB8WOBuesKYLuK9OVtGuIZIZgIUJWcShx7kCyjLZgyaj7xPoJPVSubxWK3FHk+NsxpyN2SvITtAc6/jPJ84z8/hecBLbHRmVCh4OKNO8Nud+0SJ/ha/bAagt+uozskeT3VOXBm3QpTQtOtlUmBlkZsKWWtX7Rj9tI+/InMGw/Px8AoLi0olNpEARFKJPYAtejM80x3Pca/hmeoimsGu9lSYf014p92dCO6RD4qzqxYoe/2X52NmnbWvhbRjEcIsevR3RYQb6DNqHKDDRFxuhu0KmJrGMuOVSHbpTWuYtuK+omFh6nyu+5SxHtzO9N4XkqAJ6p703SoA2y/2VswxzR58fYPQtUDpXksAvwCUKbRfaBtwXMHw5oARfCeeFkUD1lZm7ytfprCnBUEXeRBeg2tw9woiDc59HNcaK1GvFeWMAV8A/8BrH/Zir0M8upX5I0J64Hy/FhUCfIL/jwT+MWaeuBOFV0tv+VgKBc661QcD2vLaI4CCQq7iCVpiTJh+FuARfQSdAbfPiYNsEsMeOujIjk3bDQr5USjJceGzQkxgZ1w3og+7u8Bgx2UVv9t0V4hHUItX9/qzhxQxHk19oqX3SIcpWt2mmfL1S0VJEIoHiHoib+kt6JFjYFXN7GqFA0atmZS3gEVv/aYwQqyzEI6cM6GMeoA31Q7EsZzXOIngCL9dbJUCsAJ45RtRuxS4ENDQQYDWgaA9tcqwAHE9IPjcCrq8rCEIpJrjG62NHnQmDcqCf/ZQkRUQWAgFVq/rEkRsIWDupbQuQYJtXZtUBk3xy1FV84K6RoiOxteiTOrr244BSsSU2+dia9WsOU5hHe/Dp1T3hE/xwwWoVPnaBBef7nMeElnvQbvL6oznlX+TQXUbT9XyVhRVrAQdGnme+4pN/v7TDz+lyfSIhUQaLnJ+QzRvz0eDi9Oh1lyu63jSFjxJwhrUIjewLw1FubyHwgz0jpoiako5fkj6qOidPgVCzzuOSOmONlTsplnhgANY2/YRaqa0QNAulVVr8AAuhm0rF4Wh+GP8XCOECb3DOWq6UXvMY9E7mRX+Q2QEHLGIkX0PqDesa/3J1zXkSoL1Ufn3xp2K38f+PYbdZ5qgYoEH3Ixh5y5z2EV85vmnYi5gf00DMNGcbEhQPOGwKaisBKKoB6y4MqfOJmAPyjPIO9b7x0/VEdgFnyyEFcnuTT4dlrlUpVS8FjEvHv2KxPD7Jg7UpnU0cj5DQonwjLaubLauUfmQ3YsZ9D+cKCTHeJV3KX8P31KkGZuSdShVLx8AEuu/fMksxmlU6BoPT1parap19NTEATZlJVI3sywMIw9t5BVoYSe2TlKwFDZKyl/CocZ7wBE6x3LMhU3OWQYN22GAiteyYpyZHHemPS20SRUa78LE5mkL1MO8w9nguxPKQy3hZLqvEnZyfkOYY9PWwscn3WnwhEonjkrWsbY1GGlNmvt4o3S03STQCnrSVcPstZ0JphWKOTBh/bfji6tR3xudw6Fu/HYwujjvv4G309Ggf9r3zvt0nzUE7OhWUJXwGbQ9W0G6DLaKS7cqbIY+DUHdHgHUrz2eRyywrqap4DtT4axnvaHHbUaa6nSzPAGkBnsdpC6kNKV9Fxc7Aof64LGAuTbjjDVgPhdVlaSQ3o3++AJljBvfXTDpdp5NKYdvwzuM9j2Ck1Xlk86wBeG+V+GBCRTKak4wieQlTLeuOwu0+aqMV/zBJ/nETk/G1pNnJy+OO8+OOy+0riuwbglqsHe8iCClFuIOv2Cvxi8+U/otw9b7bk0k79Y5zd1I/IJtDr+wz1LlE3sqPkQG+dfqsc3Qx3AVpqBd6WeI4VBXW27HkDE947kF+5j+7H3Uuk1WUDHw9viz1lg8zEVZe/73PF+LoVJShSNIrgd6DGz3YJ0X1Ps9NJiZMNDvDaIAfF6iO5tQINaFEw0kJyRpz1sXt4W8B3y1nuXZnHyNPuR43c0hNuKhpF8lQq8A+KsAE88U0kEGNHfeKN+2JsqAVF8YhQQCzqBwRAhxBDn6JdjcBoGySzJsFX5kOMa6mC508RQylMS6+BkyLYp19ffPwRcqBLWx9fF1RdFUhUvaRTlNeVl7CLb+uiLScnQJtRpCl0IR5AfspxPtg2kEDjG1gEWt9Y08w1njUT1GP90Dgz63B3bTiZTZUSJDfQtgRhNrRS1yT2pAge2ObPowsTwKrlwWfDeUkypKo6kRDc1wLVxCGtCUZoQyrIIzobP3N1ueEUNQIHYZdEomNGEXvo68HFbWbodOqepsDm5g5oF2P7SVegUrlZwLgRcSzU0CTtqgtQLarKQERm8ekat7+E4Ghc11hR6k7YF02/gtwfpA5ms8LnQN6R0IOzLBMnPTYB9jVlVrh69w+PTUfjb8GaDavj6lQZsimmLqoK7//fNf0MOO2eY23dgGQiOdmUCDriMJJg3xaATuOaKx4GCHGZPlqMk+kaMUNxomEp/3OxsV3T3l1tbNaDgcepevL8YX31+8GfdHA+/6cjgY9d+MrsfX3uDi/Gx4NTwfDEPveSdq1Qo9k0dNMVmRFGnKtMtGWcB8beg3FYisoK6N0v58Rz3BWXs8sz+K2rrSif1noue62QVv2MPDSpRhtZ7F5Fo8jMMDFJXPuluWE/UeOzP8zbTUGpGV65ltMjvO2LTJ8Qc2QLwJtNXf3oGiSzoA7lCCMdUST9EyD3qlK7fxbPy0u9sgsqYhbqhKHjzS9jS81IKbF4vu9sWsjVcqsyZapxejrres67LqfvMNTERS3XzTxA8pp06o1iqPffYF8mB7Ka2WGQERG01W8oSGrI7wYlXLqlIWn/57J3JPrGyuwjx5uI+YTZqJTQTX6ttcxcAetYMycLtCkGExidlRK7uRtQerXWTSnB7xRsDO0oWTRS36BwCcb070+so03r2rba/GUP7+FdsKUXFJctW6wNuT2uoLAE1fTutBAiBTGDOdSI8aKDLdcI+9PZOqTm7FY3sM1vPO/9CICro3zj5AKZ+f/fmUhVvoH/A8O3K3djACeH37QzrLbkAEbBVRKmgiiOZwgl6AspDu/wdQSwMEFAAAAAgApLxQXaXmcWUXOQAAuqkAABEAAABzcmMvcHJvamV0b19mdi5wee19XY/cOLbY+/wKrQceqdpq1Ye723a1ZaCnbc84sWc6ttfGbm9BUJVUVRqrpLKk6k/0xUUeAuQ1CZDnvUEQbIAFFtkEAfI4/if7S3I+SIqUVG3P7E0eghi70yWKPCTPOTxfPKTu3Lnz5uS7Z68Hg72DsZXmszC1VnFYbop4FWdV6VpxGs+qIsEX8VmYbsIqyTMrzCIrvlineRFWeXFpFfFZXJSxdfLjkffVV68YQuRaUVwkZ/gjLqtkFVZxRE2Xl+u8WsYVgf24CbMqqZK4BDCrMMmsKIHa2WKTlMtwmsbeVz/k1iopSyizovASIMVWPi3j4oxGU1phEVtJVsXFOk+xE886SlMuyaCHdX4Oo6Na772v7ty589W8yFfWOqyWaTK1ktU6LyrrBB6/Er+XYYmv5GOSy18/lXkmf6+hr3lerOTzpkihjVfEHzcwXVl6laznSRp/JZ+zzWp9aYWlla0VIMAJFMD/1pEqO9P6B8wBriuEvr7EX1Q3rb766qsongPdwiiY5dk8WThFnle98VcW/MOflk/z4mIujatNkdFEPGxYOvTS6ls2g+ivi/ynuMo9rGL3YD4AvYovKqfXEx1GyQJm6ERhFcq+GKrAmweEG+0fcAVvGV+IBrI9gSzzTTGLS+rdBWLN0k0UB1lYhv7zMC1jARnI9RqqW8BgyTwBBsqLZJEgWaeXVVweArAZwKtiCzjKKsJz6/cvTiwicD6fJ7MEav729cvSSuZWFscRcAdywC0IAhC+JXGCU+hDkS3feasPUVI4a+AmWCD+22ITA3tfAMsG+Qd6ZDCrMEvmMGmAZeAaoAPYeQ7MWQbrsyj82IloBBEWsyUsHxwNt6LqwWg4eBCMBqO9QKAiKT3gMh4hzhL5g5t6NLDSEbjEf1WMPBXCovVVrfOkWgblBtB14dhelJ9nOFq7pxphBcnJ3u+T9XP46yhIrmWf2279/sVJ8PTZ85dHb5897SGnXtW94z9YM1YCjYHmCktmDdWluaQ8eMzXceZg61MbnuyJa6E4yDeVfzCgzoq4XINMiNsQ8d8M8Z4hUWQ9QrvTa9UOSxAwlZVCd6JRz/J9i7sm1rMnJMwEb7cr8SLAIfIzIseetDq68s4LqFBWhaPXc+VYe23SwZhB+MxiRxCQq8yLcAVC1Leub77aSjTZokGXz9OkxtwVo0wfrIm+/yuoA0bX33lxFpU4Y2BgXk9tBljFVYjr2VySLTTjvzjt4iDG8Kne7wSArSNeu7PyzEly71uc4YsfFWAUblF8ARI69QcuSHvQlAGMIy41cSHWrS4G6/6FdOXuXTUP1/ohz2KqhfWFRL1F4BDYreKGYOCslMDRoBo0EINVDTrEzG3L1wD7S1bx51evYL0OxjI6lfyl2tVTocUYEJearFF3XlcmHHJdIbJ/Rf9bqdvFpKxAF3EO1YpLhzgLRMV8AQ2TbINMNdyvVedvS2EZnaFll0TrHJ5KWu5vTo5QeZZgwNV1dssqRIMHwZKFBwq0Si+VygQQgAB+fQ8Z/y3QLYrTKnRk9+Jvf8QYKQGd0ARGCNOG32LSaGuSKemzqePJAu+l+OFg7VMbjKykgiWBTCJK8myhiqor3/7t22NQP2HKFX2uJR+DlRRPMNdNivRTXS3iCkiShmA3QiPsE+anV/Z4pmLG2otTO8qSABZPEQbvVyMbxUC29oDGRRFeOjylBH5HSZjNYuqJa1OR7Iuovcwj3y5hYcziwu71jF6oDZAG6VSAKRMsQedfATPAT9mvGKoxIGsHh7MKL5LVZuXAz1le4p8oXoxgBI5ocxVnsE57IKIGZr9hngRQl+C3JgMvBXGqJK24niROeAU9gjUhCkU/4ZrtpYA7rMu5tmGZ8ivB6FkehgH4IGzpf5bdX8dgvEabGRuDCfSF5l94FiZpOE3SpLoElyTLoSn6LSyYAfERrut8zo6C4nVkFecLWb3nVVdogoOVWjnEkEKmMl+cL+MCDCYvKYM0DtfBZRxC7/cPDvA/+1x1AVVHO1B5nfSzHagMrk4+x5q7w3vwuATZsTsc9fqjPW4Qk/U6Gj3yhg93HG8A/x7s34O/w4cHD3cEzRe9XW9wfzR48ABLwIXikuHewXBf1hntUNne4OHeI1kLy7ibKJ5BNwD24NHw4a53/9GjR8NRDR46fDAY7ZvgBwcP9h/q4HFYg0eDBzp4rDc6ePRA1rsv6g33HspqWMRSBFcpDIPRsHMwuFd5jH34UcaA+agPhYCSe3s7mqwxpAVBwvZBmC1IzWkrgnro7+0OH4qlAILHrGGCVWLJYF6oDlVH0ARXGxhbOAsxGWjSkxMDpPbuiXnLcvyN5fJ3PdKe8rzWYPnBShJOE62DrZ5TW6GQDkKtQlLjdh9Mf+ApVlemHMflv4pBGEnUsu7w5VqBx9WasbYG5y2PsAnVCSjSIMR3dbVt8cDC/Aw0qAEyDwZagk37hVClSAe4aISBQM4WsUPjcrFLFxAXf/Tt4T4wmC2wgvB9O43nlYBCfQVyxvRX7090T7WSq9hBQ01vyFOD/35ZIyR9uZmWceUQVTVD62vrDcVoSOJ93MA44sKaxnPU6gJSki3G9Hq+SVMLJQoZM2GSldbTN28t0BkZ67/Sqz0C7AfGSH9RO59y10IlPvF1FPSsbyzj9eN6nr0JcPP6sm2koQmptfGiDZobGL9xel6YdbTQa4MoXeUAIs+SWQA0An7GAJE5fqW/9ZY6iqsrMBxW02SxyTelbxfgS8dA8wwwjyYt6AVZuI2dmhac4Nwz5AqNYuWpTX47vEmKPMPoGvoAITnyHngOUpDUenYLhLpCNwBY7nGxpS29C4Czu5tqwb4to1cVugGQAkVHB93Q0xnZmTP0KTXI4DzY4Sygqja+m0100xm9xhoMmc4scy6BHCtc9BmQG6kQF6UQPEJMPIUhPcfhstb26b9iAYGwAu85CJMiOCarxgFKeAny9RisnwmY8lSdm/Z27496O/v9R7LxORQHq5Ja6g2HzYY7A29v78Fgz2gYJQVMH9aXsql0GKMmDOFubMBSR2qj+VkbX8aQS5CJ4NQLWdXzqJHTa8DzwFdJs9AZKFys8zAQcd5I2ZFf2gk0zFp9SMgad94PuCVjrZ6PF3907gt3aSbrIE4Uj5zWDPD3zFKxSWModa88FC9MUwdMZbAlVVuow4vlPTWh39vZZWcI5o3QLme/ZCqduDSgwWzTzSqjBTXHKQXX2Xgwim4CGOJ7m5ZYhsuI9dgQ+Gm/N9GmUW5W9YIR00G4UC4mTbZ0QFj1pXEZodF3ynA6u6g10GxTFMIxxmjC1lU/80hXcJTGnEo0CwQU6QDhv7McjO1F/HcCFlB0wNEMPRLQTY5GKDGAyY5WJtoK6fa19ZT3Mzi8/G7nhcvB1kx3JNJLuYUSWU+PWSRas2WYZXHqSbrAuN69CMT2iCAKSDDQL6AAnWgGWon5cTuZfunKNF0pJYm2eliqIfghabJYVlQ98tpwwLttGmayTbPDen1pPhnL4+jUkNGukrlulxB12zLMNRbtBAZagThor2789w1MpEtU8Yu23Kgl5/KyJP03BTpEpRi6Vw/cm8bVeQxKbPf+wLUwnIUgxVzUS35lDKc5H70uOCUCUD1F/f3DR/dBZOwMvcF+rx4qjAdANcgnHxma7hnTIMzpKe0aeT/l6Ny1406iP3RW0ssgn6KgQlLeTh63SzK53fSf6IJzqdicRRNatNxhp+ZxO5WA22C/Zge45nqoG/ba3FNP9NTG3gMsINzik/SDKorD2/HFGnhWIwLaXvY4AfVVU4K766lBpXFHdecWyvUkBMVPTdoH4RzxLnhW9N+oo4YhnAh8IeqCrRRxZLDXNwIhWAUQo/fMFIalmhd1VxrXylUpe/vh6M1RsCnjyB6T88p7F7PqghCIVqY9jly7Rrs9rn9r5MWuzlxbOrv2WLm9driJksoeE1lcsctpj9F5ttHxtcf1DmA9EZpLHm3S2B7r8fS6bl9umFK1PJjFs67Y+o0KmSvvOykpct8K8MOshaL5nqZonfz4/tlrMBbjYnFpgXS1NlmCm3txaPWtIfEioInxgVon0404ETgRewKGfayGAuu0yAEwbr+j3F6H+B7wKUU1t1d+FECpctq8QN/f0d+6qJ5XIThMd393d3U3uvs9OFKbaqZtcqjBiHZyG0sbzhIwh73TfkMaYMoBxXmztZeFGSzHsrpcx84cqFF1jPCeb0bqEDWlT8oJ/IvNak1eLnkR2CeFWGAZxWmQz+fg7lBJWQd1MuLCo5cv3/zL3wVvnh8Hb94HT9//YI/txTIRYq3x9ocX8BaDsZ1vXzx/jq9V47ejV/BsyL33b4YDLJTy70btzuiYy5COwjb04bHHPrqBjSciMkHuufHmsdXyzKWs+oLQyHJrWESMkvaZpBkiARs7JKnYByIH79FwZ7TXam9qHCrqiv+CHq37blgrDePmNhiIrC80YrTekAtmaY5aDmRemUQbLTBP3Qk+2eXOBV/s3Bqcp5qNkDmIykHvHsNgiPqkZ2kcFllcgn+Fs2v13qeHz2wmiCj1l1R9Mhr8f7X//7TafzT8O3S6kq1QWS10DR69n+WIeZAyoiJCJW5Ok7JygPU9Ev4lQOy05UVvNzXuTzVjAimDikdaE95mjSLNuabO7TH+ID_RESTRITO_DA_EQUIPE/SaRzl4gEMgzil38JdxmbtILrcQ0MjgWoI+4CyEKiEJSd31Gxv7LghCFnPgCIKGRCNUcJp6EV+KSBxRRMQlYkQlYpM0LpfpPk0TIVrxu6aepjPNxhZpafyw2WjZFEgi6pClrD5pmrZLWQFiIQK3xDnQhOKv0xLYCDarya2OtVUdq2epSa+xV+8YpgAx2ZZbEtrzgurqihxc9exORfL7vm+LbIhg5Mfj2xGmt2Q5pyrVW4QKA/yGx6k6GvipZXzSMjYcKU2YcNqlZdrFNO0o1zEuA11FqPsXAEdWXajHQCg3VanMkCMXMl6vmubuspRyNeFDjKgYnPeRFJ9yPHWi7sxBG5FFpzxQlKg9UISqd1CvajjOHLL228D1/bmYfCMHz9cqRXKf2pYvFbpv1ZeWA7aAfHVblSAZQws+ctJTu1l1Al4WQ1P8irFFZkWp7LIqCw54x+arQa6pILaQlKxuxKs1msHRRQo1RiUOGhiV7wSUmjBy+pNjHNzRKAHqxvlRntecAtPCyl9dnkummsTpAQMDhbdGf9J5J9yRoX5TLKoXLgLuSgBCzge+Sg2G3DTivIjJaYW3qKSAeAWFmX9GoPJXAHhDafaIyNHqFRrZH3G+w/eLExn/A79PWdBXSjIjPVmGSMfRet6GeJk9TJYxHOdpwUv0ssXwUvxWivKm0Wvg3LZUaY/HkU/bcpKz9NTyTCN+eEeGjAkWMCxs8MTrZNU0nA1BavqXNv6kljWkSuSSPCVwdcthpVGWrA+O4d1VeJm4gJYb40B0Fm4Dmdg5oDnUYVBlq9879FDfoDJwsPB/QeCYGuKVtesC889+WKAek6A6gtILNZjygSEKn2sxj4QFTmiWl/01dtxMGXiYLSDbTBtYf9RH38Kixl/PgHrHMa3/5A75g1YcA2StQC4g4PysMBJMSTsD1o1GZHRDE3QQa9mUxg8/NYrEpjNGtanL+dmWC/GqrxGo1trDSb3WHtkA53lEL4CcK6NL9bACo26u/DyxuXljQMV4oCpWYKox/R7BzQ6LwTXQmNe+lrEiy7uvsThTNpLFWBwFRpGUthpmYglH5qEzsNbpJcURFBLF1w4JqMuD7Kz7mbgboZIQXCy5mGyCrNgM4CVpT0OYVmR5cYzaRhvXMjMNZ9jAJsEFnSz4wx3AUyZgzEPiJ4XIUW35SKtZjH5OLw+NWHscceOAufSXGkSm4FPI/Y3wnYJq9E+qxmpGZA49dRH+57bnLgrlAVL0TWd0mhCqTv/BbCWlLDUCYVma1TOrI6NX66guGHQriOjfsF6EHw4tyf1Xhp46WVAgmK4y6iPZsF5UjSQv6PerZJyFVazZffbWZ5lvB9RtmjXoRqvyc5Fc2SMFLDV1ANVXmODHWJESbAu8otL8IvHjKHah0qiGMw0YB3w/GnZDnYAUh+n69pltBrtYynS30OtuwNmkaIlvlEP8nUNGmYHBnmJvvkYaKbaQ3kGBqJRiggRmO3dtMjPboKQUF0E1Xbyd3CNKY0gGwXTsIxxlsE5iQ3hmLAMaXgmXNiTDuGCdpW36RjUTnJCrtI1+hCEFuDVbpSSstVJjcAoXMXA1SvagJ3jaRWxx4cVdXEr+Cns4sQaiBw6sd0KtHJS5pncNTT1d111y+o6td+8PQZsZ+p5Ea5WMCd7gszT0wmqjQCILVaf6FeyQF3nTKvzTtU5M+okWp0jVSeRddombYw7sUAtOu6FJ8kE6lyLwra4AYsCNrmi+JxvY5AokLXlIQAMeRgbrKe11JJ1ezKmauMeQ0+Ta6LHugIU2D3gChGHiIp8rYxSNCmhNw8WcHWpBfgxh8h6hwGNZ0WRF87c/iG3OBRjobwaW9c0oRthW126pM8QFIwHg+50cMyBgUHZKY1hohWzhkHQ2Gz3kj29kjK4MFaOtjkF0KjOzs5IZXcCi3OeI70WdkRJB5dUSwy1OJe7lyIG16vbC4pd2zVj2mMmjZ1x9AcH3HPtV98+s8cMkcYgYcGbI/UGkyanJVfo1TVev3oDVWA6Ozvevh5IAqABKMYZZg4IEMDEO3oHfZoiqWieLAkL4apooLCPLlii1y8EU2gzmeVFMcvjuQPE7J0O3CEbCojPKgJMPxnQKRjxvMbnbqCvR2pAu4Iaiox9IBWBRZLpzW1jZdhjWhqU5cxcp5+gtG/kybxNxsKpdGbVhVg+ILVRdlcXp3LzbCIDCZu0ed6EqtG22uQLN8iYD12KrnEgmWDQLt7ExdgdedsUt+u59E7bDZwom1HZn+29dFffoXfVdryLhqludKzC8gPtdjeCpMzq+TnFuqRMishexyZm5gHr4NB8q5SA0NBHYzwsa8WUWZQXeZovkpCSJkGn/e3f/Dsoj5IIVvPRsXX64f3E1uKnWwbQSvT55xlIma9CK8pL1qQluAKlGFF9GAh3baA6JsOYG5tssDdCqCq5BP9Ndcqh5VBHEokldMLR8zbaEafqgV7i42AKDgwHGTnwC79utAb5akXp0LyzUofaUei2I9zwYurVSDTCEZJFZGqRprqIAtxVF6tMm++bNPp2jKdcN4D8Jp0w4N3NLtocl/Hsw5Ypqq0M8dxKQqknbtLml00dR9CRsTNxEWijhogiTzg1oH+CYTSaIrIpTTO0Tt/3Vz//NzXLr61nq3XC+WAhhTowvxY5cp5UuKWZZ+kl/Mc6WhdJiqee6Fj1v9iA6VhiQj0wTHrpCaGWUYw9qrOFZdbvltx30FPYhr22Ag+w+yxIvrEceuP7Iqnug65To7xyePLUyCS8GzXeKJbp9T/bdusbgS5Mo1u5pKQxme7U2QMxMAXMgJuLyIt7IG2cfVDMYZJbDp+8noWf/sun/5TTqwPXBtwt4V2I0TV+IzAZhVBlUkuADdkgDXysbucbnjy03C5Y5Us1t50P/HpuH1mf/iOgGsTGpz8DuFyZVsbK+No6Joslrv0TLd5A5xfRti8pGxBvHkA/A1gI8+jxtC68qUSKRUzZ82d8h4GRNalAI5pHDw5gfJioNdDwU/69msuVARUzEtQaWTeSt6mv8latMbdPwhnuIF2r0BqWjxc31of3fakoJKqbUplyasTghZQGwRxS8g4ODYS07jojh+i7h8o1rKd32WhiTBzsx1iKh6CE/8RBiHLAHn/YtucHZiAls+D2R20HRS7prraBcqpmIR4NM4lXQMumgOUQbtMlkdolrLMRwsWiiBechiDyFoAcByJEOPhw/vlgCDsJnE3kWxjNBVl6NhYy6UwurAlvre5EFVFVBG9YlXNVbYG2hXpXa2BQyteKo5kbUziTh2HsI/dcURgqz1wrqlElPKTCvQzdyzn8j61GHCF20EdkYC/qB3ZND2Zo4NSxX8fzuPj0X7MZ2jwUS7HQPQYhY8ZXQOhBXbRhqtB68/QVGJyjfevnv1BVEW3ReNSxj9af/lpaywT+AKFi69NfKd6C1bUwDILlmnWAL2QXFwiHletYTF0XKuJ9IcfHLLFAIZaiqsBZaxyswUqQ0i64HDAgMOgsGePFts0Icd1Zo5aKXLSHc2QOR6PXpMZ7EM2bobHTa5te4ab8CjiKKR58eL+0x4L6Z70bzkLHCmcoTZmIk54BGRbeElOWAvSLwbkgGBhkkBW8GvZuR5lXLpN55eyK4OllEqdR2TXcWb6G4dovxKlpMI5Rk4RgJbvWmzy1/vZv//Nw8PNfDq0MFSPtqCLGQbrhQbYitnXHM+D0bUqAgCWlsii+x91EHFaA0UHibvtZ8PSYaSDwA+sJCo/MwlAWyuCGKqdFWPf9uyKAclhJ9u9CAQGfoaPfzWuI8P81lM6pVAcpX1Am5slrboEFfQBZ9wJvVCsWgqIRVrNfzrSei13s/GWpD2b3cn4zMfJ+rsWUSWyQCSuTV2ieUkZp9juLJG+0zzLJFSktKlFHkADDFVE7m7XO0NFIdWsGDKKvc3SMss+OkKRVnTqqfBk6L1VdaBd3APxauRJaKAmGG2zDjaazpiCwNS1EwxBIMvYk1T9bg6+l7mwHaeLmRp7CPJN6tHmaCyxRVI9+V078onJGKhVey4Vi7kjDRWu5Xiv4lCEsdyQ1Gl/FRY54o1twcMktKugjkFciwPxOHWh5Opt4aewMe9/wAMWE6kMpUGeir+u8QhnxI4l1UNoh5RWEs4+bhA9b0nhBRIAkDtFGwMyCZLqRl2ShM13Eq1ydN6GNbA7m3KgrKUo6pTi75DNKn7eezZdbQgb2mw6fX/cy2ZBuG3csMWHR0l8lyllYU54ViVzX1hkTmBKecHtCYSogUiK3wh+5WOu9gXra9lh72GbNaWcuyQKMsxCMKj3CldV2HQebYMKULhsAdesWdjMa1esw99Bc9U/r2Eh3YAS5RqZXgW/gDV1v5Hp7E/PCk7WvhU7I5jeyV3pGZd2TQh3FgwYk2kfc0TW3GnvD+Y3lKNukZzd1DQuJte7ua3JBC5/gLCg9KLzS/cmBO3wIXPS9yhlFp3G0z6VHBUzppxwPoKdjsKJ+/otrlRuq8kg0fA7uDNg/WvEjrTQFFRqL8tED7UXOLz6LQ0reE8Pm3L3w6stQKQLbvwJbka/5DIfkD/vdYUbNq/E1V1LsExOCB643AG4Z7E9MhqK9X50QgPMD7yG49++A90FYsqMexYB8ZfciJof77n1w9+MMgJxpVV0Lpv7pjzj3Jlpnvhb1pZ8ROM+lQ0eCDmetvML2PrcviszUF//XOMPKCdvQ1VkyqcAhhPRM2mr41UgsRkeDRvtEFLi2gBW0mIB0rSCgplpXWxznpgliCqsOh3eLQ3sp0y2myfpMjcqM1rtZM17v/mLJZsivxmmV5maW/e2Lk3cWXvOUoKaoL3vETSFomlV4T98mk/dEBnW5NU+KsvKEMpnyUQ2cGqySsOPgBkf7MZHKH+5OT+1imfMVPvYEH6twIx+5Jp82GDz2qc1jf0i6FZ5F5TNQxlNkGngldKrcxyr15WdIOVfleohFRlLrRQa+UgZOQHOBuXId9rbIOF6AvW0Wl/GvWxoqCFuEonw/MULzXyYZNcFWb0GwG+gbOT+ZEas3QlXd61P8rdfn19brGO+loChbiRdQyKwh3JIG1kLiPT3uHx1bZCocqu1fqAKIBz2rwSqTlM/jinNGBJUuaLKml+DBY7pxTEeXh9aH9ypTwBJb6QrS0hciPg91ya7ex+K9QEq3Od3/TGzG2JOQHPhZHaSWWu0nCsmEy3sIzoZw/Wr/DKVb3IdqmH2x3lRGoj64LtK55cinWMSYqK/uXqUGh3x/B1huu/LWEbb2OFX2y/NkDRC0n6Hm6tbSSsAYdwHmnQIbx16F1QbmYus3zJKJPQ9nYRQ3Nj1lHumWhTA1LriSK2LaSLlvrQ9cBP7WtK1bFspmgMD1NLPNUC8Zqns6qtB3UPjFxL+AnpiuOiWLeLLjDO9NZVrHOprhpjZeerFDSXG7o/ID_RESTRITO_DA_EQUIPE/TGGW1w2UcVkE4LX2HgOzCy556y3ITB59k52ERBVS71v0yI8Rv+WrcHpOM5G/kgjnzIqUjGkJ/p65FNFzRho+op2mDul7toWm3mTw9xiayADT3FJweMJZ53GIOUGO+I6fdrAOTkZWc4e68V1esO6SxcPUFSLlbRnlv3tX+zfffHQeoZcEVRCSB9F2ndCtuExC018ji6qqutnGaKlDmb2WNNG9QyGFKJ6zqI25gvmEuU02vXax4apCqSRK3hfGtiN6O3clEuxLBOFKI42kPlPJJenjazuk9Hsa7D9jI3sDiKC79ZmRPGSbgWJqRSBbxOMnZxIiSdAsdvrFslzYexaDsGxUrUCQ91Tne/T+PPbeDByeT7aaopot6LoBxBeLMEwezTXEW6/70yjcU/iFX0O0pELFsPd0fgMGyj5uK8geFIOpfe/uG1RIW/mdS8t3KXRnJ9qs6zX6lJ9Wv9HT6lZFIv5Ip9Kuu5Pkyao6hmTavXWLiZ2sP7MJyjSedBy7zURmdygMP7lC/CULcQtKEnwQYAgrOnDPXBM+YlXZCQ5qK60uCd/b4zLUF6OAIeJt/4u7NOUWHxmc7quw7TkcF3+ctJZ7e9Jr8IbLouG83WWS4ccCygw97y2u+m9a+xiJ37tx5pt0gL86wy4vkpT6mQJlrZVjcuGeFHLm88L4icK/poOqsArNuDaZDkVhVXlsufLM9mTula/FxKOuxbz0cYPTh6McX8oGNvrdLSh1I0SJJE1iBpSXOvwLwHGTlqrSmmwoszphjO1B6FuPh/I+bGI/hMhi8hJ5up4viqA9KHA9dke0kL7oXhm6UhIBCsORmVsUX3gsEKX9ORA90b07GEnhzzm+lRYmj3Y1T1O1qfEaHtpObB6ol+vzb7mZp7//VJ+m+/Hht/W/bzSzbgQklkCd+502lWw5butsPUbpLcTEq/DBuKJ2B+4Eh2NhXnPWNJStjcPghhaah1/phedvFLnSvi1hceGbGX3YfoRYH+NpH+uoTdq1Tb8bhfnmvK1v8cvSgxOVPo1acgl2NpoLkG6jYmOY39SSNpnhxbAUygDHpw6PxWujKiW/LhaH6ABM/EF95YMfENlryFuYCj/byLZf6Wz7bWTtD9SnPVllHPUQ4Hm7OQIRsZnU58EYhXIXJxOe0SrmjD/igvCXJESLcsPK/7KAm7vM1T/gTBJiHq6bqZjiENlvjZEkf0NlaPqFId/E5tV745VyPYxJ8T8MTBz71TJ52RA4LDcZztbOWIlh92zHjGvxFhfeje4OhS5sDYGwAD2iXh9RapI2QuJiWEhEOolCbS31ItDVWnXVNcrodDEUsIG94hte3tW6zntG6ngtdJG3eUN2CdftIDFjLz8DSWVoHQox2W0N9+Rmdi+Lb2qpFr25C7oSgdaLmG4AqCTSnHVYlK1J4EYD3s6m4WDs26quevxHDUWC/+WcQF7ccFM+EUKBhGOKXA/gYyHCrEEYibN/OYcgD0Jhg0TmkTHvfNQ3ZXrOas1PuduJrFKJhujwscacJB1dOZWBlYgZsBIk1QdmMm7S4gPvYImFxNFpZM44iJX8G3ip7Pd03rcg5dXayq73UBZfRA4XUcLbP3rx98Qq/W8I7rjBpCh6p+/y++/5F/+kP8P/vXxxaYjRyf0/bnm1vjEo01qYeOIzcvbstJD/O2vFgBUj4YfYYfFbJ78Z2vCxUKQn14tKq0aBb94gYPSpFDYMTa08D4PwDgagzH/j6AJlzoMUWl9s64NPF4D8/ZQFuoSaz7lnPQKjvyrJ7FiUCy2d7CyzNP6jdjkAjDV8ytm0k4UXwGW4TcQDBVbdX1kMPvRtzw8e45T4CtF7KvSSR8mSkHfrbEw6pykczkMGuWKSHcT6e2s/qBUB5Ub6Rf8KCubUlWe9YaWBEDhhDMZIYvxiKqFAnWUza26Hysq9EfoLj42nH1TomiK1tCcf+RyOfW/epujO7a6fjqXA66js4hxpghAMw0dmWR4gmPp4eordec7Z9Lr5lMjpojsiDeKIdEXVoTKSRqiYckdei9dBYxuR11qMX8sx2iCdQ07ziK+Cz+m7xeZ5GYHvq192xSVb2AV/ARaV9yHW6vojV8UGsNayXiiaZhpcAyqmLcRT48TKG13dwGPdsb50tQMtF68QfHgzc6TS/wIu+l+Ci2gTI/iyE8gwhdLYUib4w9QD1eL3cCFx1CW442IugYsPiQzANZx/4jhqtz2J2QlGnWsrjR3a8MrmK7fFwH1yzC3BbSbeIsgNRViUVeLRc9tC103iBN8Fja1E4AjOHkOzBHy5zhkN3X2UT3CoSDqsr/7O8LRjs0u846IAX0YMSndMdf/bd3+3eXe3ejWgDB2Xz7mB/d7gvtBxo+Q8SxinAmzQAMTMlCxdn7hNyN1NEe+mMXLD1eX4+Tu9Bzy2XwEUXGtdgq9PBxCMepc5EChN01XmiZJaneeHbX+8P4vvxgd07lBDwMvNLooZva0dJHDpK0rNdIopvD62//eN/sF6L4zb83QlOe7KyEBYbGLMiOcLWRjjcNkJ59Y4a13weTfcfynENzXF9+tfZYpPmHKEKUwtDm5sUt5Sdn/8CY7wQ9b7Pi1AEtXC0w/3+YL+PpBFjqpe1dhzPtQfDQJwjEvsEvUMiyDI/d24j1NAdaYQaufKSWYnaJd7YJSatJLBxrdtEnZJ1p4BDfzTaghBBKDnRt1paNGABaAGIOAZMSIzVKbiKgiPCid6SSd2g15ZBq+vnto/YZK2hOWLMtAFyrfrll4ySazN/fYZ2owA/VAD14zbZRHy9HY7/YqLeV0RFXwWD6D33I2WbcoAZpd96eumcivAwB4cn2o05PNW5fb24sWhNHVrX1Q3lyB8aq/ijp+LS7kdPRaU5tcKn/9ao7WghYtZGfYMdTQbKMO3dct5p5DjOqc/Yco7qhX/fu08kOYYJh6X14m//+O/fbaFxJ9CTvBIHCZz3W6GeNKCy6Hdup/x9omhY8sH5vE1+jEd1J5Mx6U3CY+9eOQNXLy5+ID_RESTRITO_DA_EQUIPE/fB/cHElX8bq96VC2QBNnMURrFapAbCa9weHSvB/eF9J/axhjhAIeoIUux5e7zmyBvgVLg4/fQn2lJDu4kgQ/n/iEtxAJYn8kVE2hNexizMKSzHH6PZImA7CNKtPto0EfN9JTJxMYsSsJhk8XkSVUt/1AQWGsAaNJTAEF+0XbL9LLDZSReRGkrpGNAx//TXIsMMqE4yGdS5DyuD6fOKqYDUESfjxNPKUHRfTJn9oKSLHIAsmCaMGY9fRhVKV2ZLykxknmDf07BY8kee6JZcdHHpNTjB9Fc7V8LI7lojeOXwZZXMPpRbQJ3aL7VjSbZ59Mj+vn2+SFeZ8HQizwrB72N1CgieAd3HR1iozveoukcg13FwnBAVXGKMy5HDVeR+htMDQmY5xYTA3ZeHYCzn1fulvu7YpnpWVp/+uEjyUq5P+AVrFPNyAAxhCxYjTuJ2eh4E3IQ95ZhbNCnK7t42x/oLliJ7aLw7xr9N/7mxEO3PtKxdZnPRKTZQNqDEa5SAsxYKnJIoW2oWIDr7vNha8q0mDGA6LleUxq8OKkV44DVC8OEvWEMPGHsJXuUZFvBHDKuNeDytQqmp2oUgrVMrepIlddRsMTncSh1Yd3pagicDZq5e2EqBE6vvVC0/t9ZB9tfx/sPpfI6/wr356OFULABcmAGnFnCg176w3SLnmDN+eFBNaAsN4xIc/E9/xtPTRMA+jEZfGPtEMMyeHeMOMyw+pnrJ547xLASuyCgUBzFqr+M2Yj0MZAwxILxyhBiWWze1umKTLYp9ZDp11Z0cgnZZ+h9/rR/ZTWrVM+XM6EntnVHz775/IfKOo3Br6PzpD81KHZCeGpAaWe9yjX+sHd5l0+MFE1WoXOC6ibsKiw8xyH3Pblusn9eimsarvQna+Pr0T6RH5VBbjux9IXbfylQK1qLMSCF93BWTL0BDJ+Eh8Js8YMTVpPnSFBC38d2jOuQK2lYOrASLlLYPDOaTgc8qDHh7yDG/JG8mZmn5Ui/FhxcxJVh82bH+9itfSw82jrQfeBMTrxl+muAXgz/9+SymeH0pPouIq4I034o+fy0c7gKUrNiXBhSTryzOcn76E36DcQZShqzFwlLOur6XeYoRBtc41Eveq0unK3Bga80Qsl37VdPyBEFQj9cYIUm5lvka0uHwSBwezkG7i7Na5rBgmbgWLQML+VyadvYbgFx++hOaScK+ZiSFBWYbwo98WiWMqx+Ii2BiEbTJRX8WXvdcKRPyUFXQeUoLzmN4srQ0djGH2XbeUW+tPv0V/Z/+7NOfUiASflwBfJOMyKu3oINkMz5yJ7nwEJNm16FVwSRXiLAmw68LMKGA2uZA3lR4K1vpYo5hPpM3XYSAkFcnJ29LQCPlB8VozsZzsEXOMGv5aT7b0GkzOW8RlMUhC1tb0CvHC2hLpDJdSyI2r+QxHeAqyuEnzkMYRX6GUskYIh/ucnEg6/gqdPHCnimqEryrW7FeFq8+/Xfk51ySMJKDRDus3HD+MHCYQKgyLREQHT/Bk5boopWHdGAxJFtCHP6mEVIeI10VgkXlp7/C2xStQ5hgtMnNYcvzCTEmTwHiYwJNIPETvsBLqBYFa9aOGRQ8u6iE/83eQdQhvgDMnDuQvZLrKa4hfoE5+AAZZIYI5ZFxXMYfN2JR4hID5gbu2WCWgpRXIDTB0A1480bPZkMZ6GvSUITGfXyqA+cVyu+w7OMXE39Z8PxDfIkDlxdkuN2XYLjyAGbz5GXX4Uq3ZXSJEtlU7DAirPqbldqJIbNcG0ad5QnDRp2No6/1KNgcohy/8oMnOtDhQeRB8aQ3lr/wdjvAlIrpQ9k9m7422ROJyvy1W7nbKqPhnPJmhsS9MIoCENzz5MIRpUeBjZ9tNvqwhadIx7wD8QlTNt5z6pk7DoQilmY1fwQ1qD8Kus3vaHbXbVR39ATGhIwoiA40xdmaBPI1xiFgZQek4jSIOtK+2Prbbvs1+77NAsCPiCDv/BJMKndCJlHLsdVug3zTGov2VgS8f+0I6ExcN1bolc5z6qoxk+noOnhiuSZ7TjvZ89tb2FMkcNJB/F/DnjL7/Rq//vRTPKsCpCxtjzf2jWQ5mHcgo7FRcJ4XH+Zpfs43CnJeBEaN7gmrFe/a1M699Om65d0nlvy0HFsMWKKag/LUxZBCsP5xsGmep4w1/mjHhI8A1TVcZuye+qqHXrl54pzeGUX60X05VT1vgAkm5O+EPwU6IJaLklnldFwyxHePBXgEQF44xLGkzsuI9P71pJuxdnLTTP1w8UMz8QwvDJDfTtOvrqgHH8gjXSdFjOYKCPBL/ROaRBCwGLU8KyvGj6/QT8qA0fOua8B4m6SWEoOnhdNUJz7eeMAnr+zWpRpnpXb7r0LvtnP+yICX1RIpsU7DCm9n8rhA8i6SgBIDoQblBwbyTRDwRRqOqY1Ri6/ygHAI0lJeTXlegD3Hd1Nqx5oF0mmB4fc3XdB3mO8RlrMkETK119XLy2cvjoJXz4LXz9789uXbo6c/vvGqi8rsx34nwnL2vW3r7579h+wtmw9jS7Mj+mBAyUew6C76f8j+1SaMihyqfeQfsvy74tMfwTjDO2rm6eYiX4BvRWZikm7A6gzZwBpbUE7V+kY97Ij2pOHnH7JjNN34kSzYTWKdPH0OVU5++I4cB2ACNPBB5qK3E5ezTZF7f8heYyIsupxYOoOff0zzBY6VMmRzNCwA+o/wCth0iVY5no9ZA5OShQlWJx8RpA/jAbzjWt2wi8LREpRHhFDh3Ce492J4vCU0plPLCBIsZvJJwvJLozC1NyhUOIXYAFPmPT/QiW2k/ZjHZ/gewg5z8s6dO6/REpimMd5dlEVhitfPiXsL8aISK5yBkk8tkYtBZhS8RtN5llAzXPOePEQAvgG2xMu0D/bcZbVKu81WPJJn8K99KM/yNEXf4Wy+MNMMCOQ0jy59+/H6yZuT7569Hgz2Dqyf/6f12+cn3+Lfs5rRMSlYY+/H/fWTx8vhkxPUSGD9P3/XvSXzuA+VbNXVPerreVis8OIlcNzxeolLV999AOcNLzcEAfcTXVoYcYaZZ/0ondeay5Abul134NczbbO2ZBdBixfoHr0r3fSIN5ZydXMK7td4Gg8rPoKe6QAzelSce6BG5SFumlN+DPyVZ4snr4l5yJOU9ceP++Il4Bl+YRKUswJTBW+AvGdr0WBgY7z2GKrN7euVZ9yBPPZG8xv7nn330DpLPv2pVJW0O5fH9+pKr3/+b7LK69HY26NyzzoyIxbzPLFYa5HfBprh40bdNSG8SopSl2e4UxCj84za4ApecOAlXCEFowRESkm8RH4rB1k1TPGh/SrFzyNcyHzbjuQOcv3yAqQ6LXJcZOmnvy4SIkQjKFQnZpBj++bkCOeHwgYIyHEKz2qEWGqn/s/sFZtSyWUGyHKUUEIBM7979vb7AByRPnAkMgIIT1pmihkPkZxbysBeUsK4RdgASPPpnygwUMrgCOFdOPaLOMOw0Cyke95WYRbPACPLOjhQhxHCzwxZbISrTSgbLynAk82uhQfnYBX8rww67x8/O6b58M1D1sm793ijPR8Jo1sZVyioocuCKbbSll4s5TESpsrXYguzjpPh+mOUIydKoc3i3yDV7XPhnZangAFAMwgVjNBR2AI8YA5kgCp8jYsaFn/XNoy8Lw7H2VIhgg9AeABfgXkYl4xgdcUqjZeaqliIUmaaB6aiX0opxjx3obINftNiJp4R95YyZzl6Yt+jJYXSGp5ACkEBDIGltymh4H0dX+E1jfPlhlFyhrZBWfp3ylmRp+kdgNTyaVFNOZrT6k5xneKXc7BDgNHu8ljJGpyRIvtt3Ta022398uHNQHznVtyfeTEGmXfBAq8xMN2/llZ7y4vURv9C7c8atoogXo1zlOmGSy6BT05bad4TlPhs1YAxsxABcFErAdMO9eA8BrqvSEgIRpFaC/hWaklUUqvQMq4DtrLNiuWDsIjYOWgKQWQtRJoRAoaOS3W/UMlrJYSXOFJM+Za7ylFsWmUNIb+m6yCxZuTg+ey+LfNTex6eJnDsHc4k7bVxzjWfPE5WCwyW+nfse0h8D+xVGImDF/PHK0DgHassZv4d+lADLKhF3AeIh8Kasu/xD296sAe+Ch4BXvMF99PLir5f6UUxFSOkJ9gpQEf/6UlXd+qryHYAUsWye8RUWqPHIgG3zf3abaZ0w+eabSliHJkCPC/ID_RESTRITO_DA_EQUIPE/5wMGvyOFwFu47c2U8yZQu1TO4mIOWtjBUVqR71ZIg75ECs/Dq0lfv/rTkfNTZFSpTtPnufAYI/74ROvCyIs/Vj0LTlpBkLBxmFf17m+46+Hg+HD0eCQ9o7HX8/vzw/m4SHm6Y5Hg/VFf+gd7IvbYnY3iXuEX8o9hIW4SLLx4AaP+F6vwotdSpoZD4f70Ea+DjdVfrgOoyjJFuP78MIa7a0vbpbDawS/i1l541ERr26WIze8FgN4cD9+FEc3dEj4moXSLrxKw3UZj+WPwxrA8CGArCK3Wl7LroYjGAPScBcW6iIb47ebDwWoaV5V+Wo8hMGAaZRE1tf7Dx8MH0SH6I9jpEC0Ab16AyB1RI0e7N3fi248FqvXmAuPMaLdC5roDayva4GFweDu4TLGBDV+xbx8LdBCmBjc1NyuoWP4AGbD1Fr7tNBfS6cxiPJAOA4eUts+XBv+9ePfgJmC5wEsfAt+BvzXSsNs4d9ZV7vfvoZViduBFkhBEKcgBzbVfPehLMVkcv/OWRKfoxN1B4VnhUGAO5wNFcVnySxmKrsJ2IfABbsU2vGHAII0pu7WIJufDb3h4z6/ekzZ5qiJypKXAz4+RvaBQlpKUEqPsJJx/KZPuRbe5FWy7nAlxUaE8VkO5dvxqTBfhheVL1EGz0RURKI1eP7Ogw4Y/eSAwtM8SWPv98n6Ofx1GJZrn9uuevXiJHj67PlLPFHVQ6PtyryvThfdYjukECLbENdCl669pAwQsNMjf3f9G587HV8xtZ21i5JXnLWtcgEUvxof4A74hWNeZcCt5VGIy2wmTp3SWdIzuvqbDGTc4Eii2jcH5xLP8k8v6fj+BowoNL8TUFTH4FJMweGc031IoMQTkF7Wd3m+AJcc+H2JO2QzEmfi4oJjkKoUvchwTxTvW8KL0njg1hT07jl1wloX3oqIrTjXVoqafGXAU3kfAa4/wggG4DDzCJOzwFhI8k2J8EssR7BFeE57CeaVA3T4DQdD1z2J2IF8rmvgQYIkm+eyxu/h+QU81zUWNHG8czWcylqh/AyxViNcJ7MUPRkP/Toc/aUKWWySNNpef1lVa1kVN9FD5MXfrlG7cY4i9OZpiI8xWFyIZAYA4xN8xyaC41dI79vuDKxg3OYQI9F3TJgZfGjoISOioYqxWfz9IvIVq7hz2ofzQcfRmRp3BYh7C/IHtKiI2erfi8JWYIPKOpPf+HjQPxWM0j/LIo/nvQvFpcckt8fte/ieosGf5XWqQgj2tJAWJMVsta7PtscaawvEA4wEaNUGZJ4T/b0sP3ckqR37iE/A9d+grAgpPdlM+rm7uhsFd7+/++ruGym3NpmBwRktAYciR/h97pW6ClyhZPwlCHFtsY1qj08VKSY3LWKcx9N3IMxfJtmHFj0YVAnj8ADBm4zNlJtDMEbjZF3hHTZk93ULVMEj1RLqrVsiDtvpAq4h0+pt08SlZjEdL6Tb9RCkO9REopRyvinwsIveoZg8tvLlW48LNVFY72OJO2dgymbWlfQk2cAz4cCfqmxcCbqEpeRrreYKNMLm+95EyT27DwwoqzYlPUEiSZZIWdjoC/+Vm+ln+Uh18fcwkxjBqRj7xOCpFgvp/1RLnNDEhxEzQ5l4E0iiSurNhoRYHH12ingvBS6W9ng1PsAxY4QqDKhtQ1Q66FGuey5tiaB1yakHrZVDR9G65pugE1A5csx44OwKTezf+OjUhBVe/433lYEZx1LrxY9CZL0NVyFFZRNyW8GyGoPJznPSvhIjlp+86Eh2VNfA6CcIYvtNmJ7l1nVy07/mj8/ByundjK1rycA3pvWEp/YQWwEjk5Y8/cQ9Q36DbgW+4G04XXq4Nuqfctzvk/7wlL5bcUFfUAJ4XcmSHp7tA2oGUqXz8Wg5w578FJs8yBmskzV9AZOPaFqr8ANn34lsEUp520QxbTqLgIW8R4nJqO6F9RFpAUt8PtR5CGLMB8sAOYXh401ABkT9oXfY+EDcYfNLKVKpiQGMt9wBVedsdCeXiCONWzNM/Nvu9NZiL50H62XqCYi423b2WznJbkfOgVbWyJuZ+O0beqmDjiQiOeKatmPjZKrOsHyv+f8GUEsDBBQAAAAIAKS8UF0nwbA6oTUAAFifAAAVAAAAc3JjL3JlZmluYW1lbnRvX2Z2LnB5tX1tb9xGmuB3/wpeBjmybYrqbqn10gIHUGQ58d040UgZG7l2H0E12d202SRNsmW1fQZmFofD7tfZD4f7ttnFYpABAsxtsNjFfhx9n/0P+SX3vFSRVWy2JCd7SiyRxaqn3p73eqrqk08+eRz5syQtymhSGH4SGNnyMo4mfhmliZEuy2xZFsY0zY2rntM/Msp5aGTzVQE5YuPSL8I4SkIjKoxlMpn7ySwMnE8++eTBNE8XRuaX8zi6NKJFlualcQavD8TzqyJN5PO8XMTyOUrlE8Le263y+AWCkq/JcpGtDL8wkkwmZdB2SID/s0CmLfwyi9MS2wDpiyxe/+BkK3yicnHJ7T57+ivZ5qcLfxbiR0ijZ86hlvfLyTwsZIHjfGIbT/xksjrO8/TtGX4VCZ9dptfi/SyNV7M0sY3zcFLCqMVh1Y08fRWWqTe9wlqnVw8efP3F6bPTC8M13j8w4MecxH6emkMjiCaldTlzzV88oR/TNqb41tvD/+BtlkcBvJ88frzz+MC0qbT+M0njNC/ckfmLbne//1nftM1fPB4MTrtdfOp2D0/3d/Bp/3jn9PAxPh0f7/f56y79mOMOAzbDYrJstKvX7R30u7JdTwZP9p8c1+0a7OztI/iWdikN2++eDrBv2MvPBvtU9f5np93jY2rsZ8f78uvp7v4+PXE9486DDw+ePP38N+en3sVXvzk/OXXNMPYv09y/+cPNP6RG4Bvhm2WUwfQC8uRllBtBCKlBWhhBapw9f3z8a9sooqIMF77R73X3Davf7e91HPPBgwdBOAVq8eMV0o01Ka87Q+pIYBu+bUymM5gwSB2ZgV/65tjml0UahLFXv0/SZBrNzDEVXfjFaygVAEpFeRh4gb+Ko9m8pI8ZfMkC5zFAe5L7i9B6b6aXRZhfQcbXL8xh4PgTbxGWYe692O51u93WccWJojaIUj6WmkbQj7tKnX117L1Y9LGiLPWhJr9YYiMhbWMZGLjMg754J1isfrPNt1ESeIsCk8Xj7UDSLMyJI0FDeQxPsPH0bRLGsQdkc73yTj7YBoALr93Aob8dHruRmYdFFCyhNPR6jEPp1MOwlTnKUMoSADrKkcu1FXlIs1dnKeBX6PlZHsViMmm+nDidjHBax84kzVYWNwcSgXfiZ2qjU77zAA+uwry0AG9GZhGVoTkemWW0CN+lSYioLBq1SJNy7hEAahM9OZQqs8zTZb6WAxMpw5WfR/5lDPzKNUZWNas2PhmLMIgCH15ebC/+/E9mp54TS51L2/waXnBClrmPpOLnkPbnP51ACauaWtt8HkLlABCpKjWuwqRMIXWxXbRAbptgvR7KMaEcRgjSauFTU7laBZxKFdCttLz5YzKJfOP4xOBPXA6+asUUqmiUKqLFMlYKWRoy2eZ5WNx8HyxTmTE1fvzr31d1paIYY0UAXDKPsjK6CnECODG8zkAKhAGkRElpBY4kfKdYLqwOTz5K4NfhyjaAhYWxDeI2AmmZ1BM6rDqDc5uNIPPYCfI0S3yBd40GOH6WhUlgEb+WYFwGj9Bd/GXLxrnywb7yY+DfcZhYRUen2kkKSAwy0oOpmsB0u8BRHnLG7ao4cI7ELRz8Y3VsRDnxjg+QUpSBV/ggq0NIhRcrCNKp22tUtYgSGO0FFoyw1K978Phm6SdlFIeW0x9A0o6WtA9JC/9alvKvrQbIp78+bxbYasLUCiCmp29dcxEWC5AZMHc44zGKj2iZg0qFYiVEeXJkAG5dCbnz5VfHx8aPf/OPve6f/9SQfth3YgUJkoJr+gC3zH2Ybh4EUyADEXwMyIB0jX+jPPdx/CaMVrb4V2EOFUDJmy4zxJrMocfLlaWxlE6NRKC7uNMrmJYyRwFH2VU2aXNKTTb21Hx288fCeE8Ah91+8GFoTG++B00xNWsElHjgLVxEdgXbjf9sWMH9WSJzPdelP52OSiui/c4yA+EbWpwRa+O8FU57csoKt26WvRGLqcc1JkPWNpmVp4C573eHpv9qCaqD4QOR+aRRTP0SpgJY2s33OQ6KPRiaUFU0BRFCqGHae0Ngh1gKmSJoKrFRhIBEyEY+jOoejOt+ClSQxAzdrvkFYkf7pCtC4mfO+RfQSiHR3iNUmnh9vn/uRCNY18Xft05z3SeqkPJvmOr7zHFdBxPZ2ghfRknhgko2WZZW5ghJao+6NijI9i7824N/B/CP9KoepvbgdWxHySReBiG09S3Mtft1vgztHAfGfQLtC+v5k21unUOs3paTQ0B+1kwCn7FkfZ27pm9dB3RmYVXcicNp2WnPFpd1NurzrVOKgwr9xJKu1sBN89puS9w510Y0xZ4Cly5CMCmdxE/q9tS8dQ0FAj+KVzjQ9OAJvT1Mwny2IpuAcr1xNcW9RUF9MzK/8LCzAsLrF3MPtLKx2zKCb+dhDjwNlMrOQ5BJpLUTkLn75l7kBNoLiVfLfGzy4IMIhWJLHBq1X/TbeZWCgJ0r6SPz7EJr7RwbSnlbOrHdc7pqWZiEKA08UjOgnPnKT8AqY4npC4EJBlRCFhooYJMU2wqaowrk1AvBuM6xiqruU6/G5vlWlSaHDhJV0SkmyW3KSKSx0a49sPfGNTUti9Al4+znSKgK2nQZx+5/AJwvADvQvoDW2WYTTcwxk5WOIvgDXEAtWFuNm4vAwLq+VkSYjJuLZF2Xu7NC8xk7BLX4aAt7Wdd7/VZYSeszomml3HFWXhKBV8SOoB2Cbegkn1RGs1cprsS+cNCJI8nvndbyrbyiqmx7IxgdyqlXsdqaFlxIs2skVT5Akt1COe4XOtRvpm1g4V+GoLezrn123swhvmx/gd/0qrFa/rSp7dhZj9VDbCYjDSbeijUNPZ1yKZDycJLmQeGypbMBYpKWaLW0TtEaRG2iMV0U60ietfLIuiGVWqF3+KAJ1vtQohS+sb+4DHyjHJYOCaqOInsJhwRTwv4B8/8aQBQlcF0LKy3fuZsqsM0mBiqEAkKKvv7SPTwkT20txH7Z71IKi3Whiwgb65fdoTaABUxA6E7j1C8tkHX+ZYF/gmg6tbTiIJkZwHYb1I4GUxtkScLvTUg2hyi44QH6lgOnR8sTOVXiQcVpvCxDDzhEBtmwWbaZmMO6Wy3DMaSUD9XcFrpw1RoCCJTmJbzHy7Cwbq2e4RVAH4RKI4TsRDh93TESu10nbPUopbLdPZhMnlC/AMzLsOZCTJsPusNK8E6UIPbIvPIWGZLiMg6956ZtRso7+jQz5R1oQfMcEbiR+RyzoO8SegZjAj3Io2QG0MYu5XC0Oh6KNo7MKI6XaE/yINDnAhQoD+g9CrUmj8ynjTpoQqCBsgqt2RuqQMsljsNYNLBRw1lrDS+qGrSBeNiUJqL1gBZcUIfNftUlu5Sger9cFqhtAJkUN99Bb42b/23sGdTAME6HMI9gcMWpUbcnFErIBDQVwKykTEkXyVNy6rC3DRT4HPHvggYQySgGrTzzJ6EFJkCf9P69rqCVKLlCVTEiFgPNzq7e+mVZYOtmUJiA2fVAymwTH+BF5cqTiy3eWy/0J6D41HxxvRQIWy9JYahaIOK3PJxKtyKk42Q8PqGxp2bU6eEUrNMoTCZQfwhcM/QmcZRlMJuQF3Kg/ID_RESTRITO_DA_EQUIPE++xPwqwxTeHOiHXjipgA2zI+DDfoPUBL21kFiqEIin0EnMwUhtF2Khrgp760bWPoriRu7YB1DpIY4l8LwQiAn0F8AJ0a2heUvjRhiYKLacdyhcXZx4rwp5QhM0hyxGAG5YeaYxWAuAKF3mrtOAVYDiYqGEDHOhSVBFjir52YGHAUZdgLnmEDAUOJioFVBMkKYCQkJJXqUKgIbctNYdMny1tksYY4jQw3t98fWJ2PhBMMIakUVetmngCI/iPLdSSzJZc2CP2Lt9sImjIPhJ4m4clNAchi+UYJE9k6NjDkNZkbKOWBSwEsRIAYc3s/qBDasCMVP4dYa+jBjMeo6itdQboVzX9JtuII4tUnZJBkCN21BsATHt3MGZOxK2QCsfCles/wLbMWgmZ2VSYmlWLaZh9ZBXZFS0wXjHbc2CuJ8ixFoU3CSfQhdJegNCJs7nvFRMgdXxjysbHp96vxIuuP+GXtM527hVz7Y0fjgN0X5mKp6kIWtpURLh4GYBJF1oPoXF19itXZ4escBQBSsB0grpO76BbZ4fhylHhXq8i8nDJ1buyrmy9Bh5hqW3QeFzZAo5dBB0dTSizwJSiXIH9W87DhXSalC4vs44ocayWfG8y5jlT6AgtTJrDcmReAgcEtfUaWtD2ofCvQijX9q0msjK8Lp3667QGSfZx65cwmIWNDzXAa6Cr12sQV62puAyrJOKrBmsKHAuavwD+Yw7Nx+Er//nSuPATXNqhb0X0LjSHvR21ySJtV+Uk+LGMSqBx+dGMwxnMmYNgRGLPGdii+Sqgvmy9nlhDl5BapkB80gdMdNPMgilVX64yALkL9WSFnqCMBM8/KBXITUEEo/9ilS5LB0xDc0gOvEZ/QU8AsbuaALEPQbFw6DG3qB1uiQoKLnCTxk/5CxCpOEwpaMLkBWwCFBnIZyaz8BQSAzCHzg4MYEyZ8PfbKCjnMFbOwQe5Wg1g0HCPirnlA2u8piF1vwTLwzZW6su7ME+FL5KpA9gh5x761w5yfX6z+E9H5llpefjNWil54As22aIRo3a7uKwiv2Ep/zoqIH/6ljJVoLFJCNi/nmPvgJ/wUOLY5pMz4oojlZ7GdvzW7dkxSKWtLVDveQxAGUKLgCeUJUQUILsANa8MZ2kOJmGxBAN4Qlq8bRDe4ldS623gCVNIDXIflTWDFUsMEngXutaes2PvOj1pC7Kl65IjIwLrlOqR1bhVfUp1rlo11exy/aJ6VzZjzc2JzXKpbdwkV7RsGpHK9cEuQN2ZhK4WDyFsnmiW4AprLatIpCEbRMnEbLEWTW+jcm7wqKO5jCNuqey0oxubMNIuj5alMdhOMxdNPo4jaBQYT2M9xBc9G8YTYW/0stDa6YJFMFI1mlConQ7XBgmLsxjO0xRE27YpwpvMbTkb29S6bYC3bdGEPTId8xG8dlqhOaimASdcvA6i3OKXgh36IeBx6aWv3RqN1Z/L5XQKtkOUOp+tyrB4+pW1nodGhcWIxflt6OzCByG5KO0gi1zUWCrOp0swZoLrQGnOQM4yQGcWlmQet1QPVAf1gPaDw7k+mvhDyCAjo5wU5LCl9EhU1elgJFO0GEYLh5a7Vi2VifUuIAwaVwTsFdDE6NqiBH6G2SgXmXlLcectKsDeJTahasBR/TkPsxj1EQS6DgYM+JBCybBGoCB0QTFWurTgLQG2IkMxguEac/srV0OZWgrGdRyyMwroVQOz8/AqCt+2Yue2J74CnjJ6VqiJM9MK6KchpopwAhChWW/wUWjGg2EKCGhTikZ97KjUrEkqeSMLRQgJoAjdruJtxW9r0zIiB1dB9oqFcoPLohva6nRA2sYwFlCwLdeqmYt1fQ53TFBckbBtITHM5l9jJhxQFLvNAYpBSsVpEVqQQf/EQgNUNGTc5liMMiFVQRl/YZyQ6mAs/JWBTjuUQ8lyEVIgEpA3FI9g1EIO5kRxCm1eRCWUAIXe8Gd5GDoPFGwHksiteqhH3XHHdZuJPdRUvg4XQMlgZnGU3HYYXVOwA1D1DA3EYmiYjwg3H6h9KcA49EQTvVmYosWN7oTybRgmHvUQnTOIj9JWHAmdC9KrRTeCJiKhcvT1mk/IxSOEOtiwQ9NmKTsFu3euILjQ5BmEUAVmaHfDOHmz3M/megifb2euGrzXiN2jVzZXhSlXXnL+pm1rjo/Q26uG+rX5fY/QN+i2B/6R6hYEVgR28cTW9AGbJD6KVyn2TQyj4TCT0KAGp6zgUJABzJlpt+oqyiCtq0gRKpo+AKDidzbCrgV35U+lEdDsejFw8FhZ+IB5VY9h6kljsUqlhdAmoPjCRfoBVQkjcAurb/fsYg4c7rrB0TgUoGWdrXzXOcIlPQ6NA5V+ijNhmZ9+s/XpYuvTwAT8h3ZVgCj+gDLj+teY4hEeKe+LKFmW4fZevfoFjUS/MTaQHE32rSt1NuveagheRwGlau0I1ha6uvmUvBI3f0+BahbF7HXMzpGonK0fq6M1qrehUe/CBCRu3ZT/hu/ATi7Ojs37QfDTyAvCWQ3iKTQr4Cg6s1bDN3WrByZZHUoCOtzNXyWzJWCv9ec/yU71WjolkBYwg1kaEIr5ebeHy0zLvEglu8E2jMye0wPIPQfH3DznHEaRxn7O445EUxUw0N0iAv9qDDQfw5f3gB0fbMZuDEiEghkuYi/CBOPk0XHENCgEno+xxji7QIh5iMvazEUn6SXgI7KwH//mHw8PP3UM0euiqhwekxQAR6kRaDFmR0aNLo7SwpF53sVennd36PcA+iqJSaH9Pacv18dC4eu6jOKoXGkE98YtL0e3OyjHDerUiRMnjvHljVM5LJ0AV81W6XQV+rn9pi1qYBOi2DT+GHIKoxKERr/b34VuMiWIQcdxEa0zLAAn6OIOlOl71SBgxKpEIEabHUSY5+p3mMSbb3NuS65ikjIVQAJgVchw8yLFRYTQSG4PFXSMr4pJFGOhfwYZTksQNQdfGBHWEvgC1vIqRCvVB+u4KG5+AFmfLmE2M1ze/PG3/+fm29myHTt6SAPKtNe4ECZXUZ4mWEFxL+7babK9OapLmRrvTaFS/YG95v5AZeOohcWtRxgjIzjp4FxXY7fOmUTNMqR8c7W9lmqRBbWELBvWYrvYWHE7Mu2grAHDP4Ua/MVlBHBwRQBwqQ8saIxYjEtil8tITHNAHCEJ5LtaqMan9UEJRRuZfQFTwFhU32gNqkEm5GPwrkDIML75jpY+HOMiXDBOZmnMX9tZyj6uJyn4sZGhgFIHhmrcgj8N/lB7lycueoDAEIdmKPF0Ao9a4sO3NBSbuPW8TxZ+5prQzCjAASzcA+lsGgw2c5aan1bC1DZP67jd8ua7fAHDVUWfC5RUHNKIXpfouwe9X75YxQS7DTUIwfhcmTGBXEqw2bSC4gClR0GBe6uStDTQNTdsfCOnSe4XuOTzLgwsJX5wM3LuemWNR14dSs/IaZs7Uj4WYJOVfku/M1QrWbtU0bNtrJATPfGhWEIS7zc//tVvgSKNF9sWDLDxXzuAwZD2O3fPPtg1XhgFpv8A6Y6xKfZ/WWATABgQAO76ANjVfp7JzXcxSM0jEM0Lo1aFq9agKojGBDLPJoYjbgu0rSMwq0UgYViw5gs6MLmbqvKWukJkwxgPaBEKV+BkOmog5gkl1pRHbB/pFKw4MqBEsCHrdAVNxs76bgkeFq5pT9akTOq9alLyVxUpTjOxfNWyplYvqVWZkeTRArBKW5ajP3Vet7kQd5dUadfp5bwkNi73QA/0FR+clZAsTVxo5DZ01l1XTIhT8/3sg8FbXe5c8IPMWN0HA/ebtDjVmgImGa9lUo2CutXcFt51wWDoN2rMI3MLJmYLfw0JZIvDStHIr+yrhxLqT4MkW8eLdIsMKAIfI340gdhVuIvCHdzeogac7L5wbrF5TlLqYGhYx+u2TgIA3R0bRLhrLjPAboOXS26H3iN1IymQU1jPO9r2H+tFx2w6aDSeSgDROif2MALOMBZMQitlabJ7Gl37pL0OEJ2OjLvRT7f1ZPme0+12GYMBSOeRVqUpzH/SVVMQNTf/loBGQmwYeMHN36OZA4rnyekJKhOLmx8CsDwc4+Tm+3xCNggHNSyMZ2dnkMwMpTKKChEeQ8EOkPnpj7/92+fIg0HnAstxXYvAH+SzB6R+dgERyGfA8QgqYFPTKnYUrUJGsLTYKJuiGsZH91ZbhZnCkTAcyojRn2+czTEwt5rpCho9PiFBKJuHZgkpFwhXZPGJKcNEkLMO8PvTzgbLW2/im/UIHEoX9HKsg70/SBks2gB3woBIYWgx6gmkXKPb6daaNy05Y4GhSQtzt3sC7hw5bYMepX2si2C/QhbUfHYctJO1CQlFT4VyEUI7ZJ0wnHXhWv8hEjHOnr/AsC6iMzGXFBeI2h5A2ekar1/ofQqNv/wLqHgLjKnGLIcHn4IVaKCpQVuSs2DSVes0br4zzvxJuv2Xf4FSR0yHkOYrTbz4+oRNBIriAfKVAXHSMYFVVcgGg+gDtfsi4k2GuTVNxx6bjmgNS1K0FTajdKjR8c0kjZ43EX/4M1x97HHHGE3UAN5FGWJSFUfU0ABQPLRsvUWv4BuXt+zCv81+QV2y466LNljsJdzwbd1jyJQgXGtAgepeHUF7z5quwWYZZTePKHJRbV1do1VRGA0IklcUotuwiSup20JuTe6piuB+RYxKrIDq3ruLOA8ESniA0CHrzkije4B+u86uqXEI0PTJoQYIyP43+AsSKTc0f07Th1cY7xW38wcoVL/3xh/QJI5DphWydjJAZSAnjhlGafldEKXSrRdEVxFZc0jWxdLnz3gMQIuTj42TQgeP21lIChMxOcaztS2lwsLflpuMi3YLnaUr+hgUytpIfbxtSexX+ljH39HtBv21u8nlp6AOI+81IK++VUcTOhLxjxr56+0+WvZbkP5ezsRT7nXtQ3z2Ys7y5b6y5dCTIwdYkLdgr6xCOhGl+4WnWroCVdlykS58XQgxjmEcc9cGRXJuFOllHqpwEPsIcwiXxMEWEVj1yc33V2FUOMaXleQQDa63yOO4IHyEF+VNfxC5D1XsqXFKD3S9FavaomY3oNXRNYYTggmbzECxUGfUQR/L9ZbTOyCsaNkiw9jh7Oxt5KQSyiMJZX2XzhqQCs00MMTwMGCtACwdmceXeCKEbT4DjgR//ssymafmmu+x4rM1XmiOu4/GwF63wkAa11hioIJ7/MFIam9hSJLfz+VmZMWzI8rIoxCAXdaIUmvuAgV94YVc3HxbTNjroHbHMb5O0bNJqDXL8UkwN7ZD/ghwEC9BqCfhhNLFJmjp9YaaShwwv8kECS0bkdYVYoo4ex0h+RwPyikC8e9kbByJCDb1eTgN81rfMpGfKIaWL+w7SP8iym5+AEU8NG5+yDDuvXG2hXKexVmY40iC3iuUJ/LOnJDyQMOC0kMo87Z5DHCB0GVIflX8WC0+buD53OIujIbDLdC2wcDh/gOW8w4XSieE38w6KzQSllYTWdU6kR48jly2MCDBNVemXQV0uhj7eRdC9zxgFCFyDaw35DEcAT8ld/oJfUMMlnysbfHOPC3Km29nUVqhHi2v+ZrT3DEes4/s5g++QMQgRLpIC+FNrMS03Vyd4e+qk5Ec+xnPCQwP8tzPfeABhVwaTIlojGkO2gRBk6gP1WPrfFka68elxDW5LyW+XNRhLFZE/kDzzK+iMA5aqYC/mGO57en+ljPyzpH5DZ4C841fOYbNb6b6M3PdsT1643yTe6ixfuMLZz69TMWL2NXISYoiTKnjW23uSmHN02BZolqGSznKHgbDmrfY1dyBs/O6vfBcn+AyJkcAyR8MEYL2y92UNj02ZM64c6t1W7XxXEr6hrHfTgF9POOo7lThnZ0zBfRJr6AlSq3bBXDp3H8nrFd4R4aD8kehiC8xnjCO3gnURf314HAHLNQMsT6mkycAZ7/JbeMb5PowjcCrMGuDbCT5H2E+0CgkWwAdQ8QmLXhPFYpSBD9BRx7qIgEKDNygU6jzBJWqU8C0gUFJ0I2k9A0ftPCFX0q4RomMNzXKnHZa5k3vfg8H6XwHvU6M5rcshaPBgH64j1i80lauVNNNP2LB3ZfrUDsKwyNm7uKBNFpZcUSNCoGTOg97TndnTYkegQKMkMZ2/bTufkEfsAzQmC39mOijN+ytaTEWRti5lgDVsVfaK+kiFK5d4OZby4SpQy3Dp10jdJCQeZleK1DlBhg+kaLmMpqcwGBi6JvtHO7b008SPPOODj1xks7Q/vAyyTFl4eRDZ3cKr+d//idOOO+LlOT82cUpp9Gj3DY6dPrTD59+YpegQBYYRIuLjfRyfE3LFK6Jofe23JDg9m4JQahtT7lc2HRKVXpSw1Zup+wdj5Fu4qdM0WQjnCi6mAgcydYrCKtDu2qifsYeGXH2jpHm0QxdSzapDuJUmuoUmiMD6uv2iH6raATHgKF1ez/+9e8vLk63Ly6+RlZSx6To+phj8KgnkpdU4TBsoagmBRs3TRnW36dgBEl4muzaV2izOneL/DJ3kucRR4C1+3oUO1VYqNI8fWTVZ6Y9EgesSX9NZ7u/20b110CnyqlgQOoDSeq7a9p+u9lZHSPG2ER7MHQvVzv27HpcccpLgGkuIg0EGkmwoEqkRpwmM1JgKj6p4k3VArcN0/BQM8Y1W+j5cjZrrHmOLsuwUlZIzZEse2EUS6BGVnV5dcD4EuTRcgHZcUV8mkaQfZFeRYHwikAhVJawweggacMaDSVaMIV2oN4DVW7h4Pq8trLw+3KIjXN81D63g3puVU/Y2tQCXU+XSbUMXs+ecuJdPc904CvYDSVyB1INUF2tfdvkjCLngUGeglhzPBw1tV49JGniLwsQ5fWKfVTY0uqh6CS5GN/KBPR5a5lPivC513SKWCB19igeaGdDGJJK1v71lbbLaIMEPcWBwkk0W0trlcsD79pikSq/xGentzirGsijxSOproENuLTnBTLoCARNhVgKPqlBSf/A8Ya5RDEVf76AfClYTwvm6MK/xGKkKuAY0BsD6BfmGzGpQBF0ZDw7PgVZQvICsoJqiGfpzpKoXAbESKrwBImHIVtEvmQjAaM72/+3ohAiQItbCg8Oato/6JZa22J+iwGkeaNwY8gb0Ihox53r9GvVjCMVXocrPS5hRBNtmzAWyBNgLMxxY41CMayuH1nJVq/zkMDbb+hcR5tfGG0gYc0i+0luKM2CItzeKIra1t07d/c7gY5XJ3nYpqaitY5BTxkDZ9B56Oz05RDgYzUA1b4eUwFvdtYMvZ84Lr3GuKiqDtpum4ao11wXuUOW73sVFkp/aE2eWHNhiPTaC1GdsahqgGSZAdpDF6r1S8T8o7rp1epDm6KGfDzwyRkH6p0RKqsYlf1XO+EM1jjZ49xUMJl+yd3hr/ny2ghYp9SNaxd8sME0CoO7iJlOf9hMze1LinpQQH22IFj8gMS2mW6Zt9IPcr+MzmDM/dvpSOB4o5LkI6JWtcD5M6GS1X7+Ow2QA+8KF9uwqXmk2iDPo5vvirobfMxjWOtlYIfzCaUa8hE2iXhWjo+Ht5u/a0Sl1tHzoFIQttHhGxiX1sC4LAf9ARqGsdGgcQY5IhQ0CNdOhGeA1quN5cJn/YNdsK3ioUYbDbO6CmbRCVNty9HraxJBfbKPdoIVOU+PVs3vIPA8uXJeVPnaDYrVLf4C8hZcCw/BSnULHOnuAOkMqFUXTaG5zejXTP6fYPA31RdadAFMkivItljTApbQ361iBwB1RThHO6oeog1HZ7KjLuqheo1R1hymv1uvsJ1wLrlGTOZzXvvr1chILYRj3egO0gj1oBkqR9BWwSSFakskERV1DAZtdSw0NzF6lHGHXFUzgV2I4QjVrgu3NC3bsY8f1+3Uhreu+vYxeIWxVrOdByq7TCNQSmftyx9VRLcHOhaZblG5+tilOBGXyaGMfH6XLv/f1OfPikBbrwBRzTwHT+oSZw6s6QFNHWBXHF966h2ftCzR7apBlbVmYBtQGfGMNd2gqRd0P4Usffw1+LRluc68WL5iz7IvOFxcLbcoC8iA6bcsigjFoPY4daFz7Vjf7wLn4Bo9Ohs6lr4ijtjus/f3Aueu2qACDFcWMuhNlFxb3DsRoyIj+sSCA1uuwHOrzqruV6C/pSCTegzqMEJU0VHkcw6oPA4j4uVnXx0fGUTCASBFbvjVtgcmGcQ/pApAQOANS5g62rHuz8KFYzxXegDgg+U7tK2R31NceW8go8l31oO7kUwOkP8zHdimRIbCmEdo3H5X0gbHmmIymOz/4JiIe2yGap6t+pHBC1AcOYYQtdpWqPmdnsh+jzrF1XOPxAqDUDJ0Fz1WC6a/soRclVKMxYuzbWiUS73atnrGa95bYasrAOg48I1v8hpUHZsQVEFrwpcCtaLaPpdb9xLANahFMs9YRC44xrGiQ1SRFrgEiFvqTi6eb14jgKmvTh4x5e45DxhcxbAKE3kbsM8aP2ijbnwZBqmGNpsYbQVIuMKHO3dhj8IMgUXZPbwLBU+bpj+72G65l665ia4NfY6pqUqI3vouunqjS72NrsmhFOzpe9x9Zkx92lTQW2dMfiqGSalba6n5NTrmlQVXn50MvGsSEIaxD3dz44E8y5vvA7GhkuFu4Ul5GBXTRRxQtmOSo8pIlfBnpJVSry1kLsjuaAER4PAp7qhi7cvVKuDgrGDX4r/yq1bYiE1QdhZRvAQtVuMe9kmUYQgGs7NJeEk6Shxek4EFGu2ajSRRlRzGvQFxtUMYYm4mHpdZj4+cPVwHnFcBCDgBPEv69o2qZLEJezfQwVi42dtDHTVAH7PpGn/ma7uunYGW4W56EUoJ3pzB5xA1XBKWOZtH4gqUz794Wkdi4D0fQSI/Pf6y+akq9Vgr1XRiCJqtYjTlfmlyYah7P9p2adgLP38d5vAByRPjQP67SC/c3dupWzmWv94/XAeHVdQudhTUlK5sI19XUXq3kT9ueqywr9pXJlRzhRucbKZhjqds4PvUPBWHcwLuiq3XwPKbRAhqNsY6FiwKAedP88vix9/+rUg9UsIrypyCjEjVIK27DtggG3MaTubiyE9lq5tqaYbXYEQGkjkp/b6DWPFeLviNikhNcLbJg0U7xcNr6Cng5s0PGDio0+hllF1tok785nH8FrC55E5VBH+qSKanyQSUIr6s5mIZG0i9EbsTfkXXUSgJX+kJ4xbxtBZgRIo6so3e6xdZI87odjyWunSIZufN97TeoK6uUjDB7WJp1/vs6dlzL81pR620FvvCWqS4/0oFRhQTGaU0iJRN3E1BRWG6fKxIGcoAujX+a5NHIc3B8qauGHykK+8yDhseOceoGyMGmULlJvGS9jTSCQOKHnxk0OEniKu+Fr+HL5cwqc3+KPq2XIrhDcBiv+Q/s2I39YEG9B0AOj6j2EGkUzVpHGh7g5Bs4DLduLYWZtFEaBmZt1wAK1xJVak33Pso9D5HoVqycUa6BVjWwlSTXgDjMcbpndCWC94piWZ6qqdNyZF4H4S/T0TdbejOmi2NkfCRVUGgmxF9wIh+6cd+MhFuPELvM4QjTyFAwLi4InhqEcJgkAZEh0SFTRS/+R2o9IqJxBvPul2DNp39++8IoIv3m9jGv/9P+dYf2MZf/i887O0As1Z21QSaGwbLgZLGjiBkpIDnpJ5N0K9sRBihgtajjOpDjYk37xoXX3x+ogV2AVfIYnHMDYc6TJB8qri8Nkw+JAWKNH6JjrZJ43fzBzlWinVoaqcJ6ccUiVOFLtNrnMjrlViimYe4A9HGyBZbbsvs2tMCA0zkibTlZO7qF0RazfLwgQ7Zc808XSYBNDdwu04XBp3e8fRvktCYNlD3OlSndslTMavTMDXPMj3wkYnOAfcSFzIDDGnD9tBv0gkoRud6NeqOH1ELt/vQ2VFv/IhbCq/U17nvIl8AdcOk6Br5XOkS00IbTKpAHlSJF2fiKNIpwjZIZmXoAE3mYaCdTqm1tHH1plXDILBiFLf+xy9Ne7EsxfngeHufqx5aSgOx3+KClwOFR38JUFu0fZPbJXZrbqF7qXHMVDGZhxSxph01BTmmeNjlvZa0MXbWMtPp1JTu2Mph26vdtT3FtZWA5oLHHTsHO7bZM7Qzb14mZOPVJ96gYuvsDch5WuI1Vkit6OYlUQXZ0bpntUheT2WuH7pmObv7ttk3jvkICyinRTqLewDbyvUPQUc06lBnveSP/+v3GBWNf47ppkGn1wMBbjyuAhtfJpW2QItdhbx1jwYVmLiwAdAEoIEZCmq1nO6everYzqBrO70dzgU419Wdm9bK9qDeBPDbw0e514xgjYjz82NvCGZAhcWWs9ND6PSXCz+CWjq2co5g1ZC9A2gBTKbTP6Cm4I5LNh1fJkq8uQmt69l64LRSIUHBC+8sZ7AvHks1cFerb78v6xtgVIEMNAfseJmATbQNxs82WDkvE0WnL+0+VN9aOdZ4cEiVI/SDbody6wtfWgN2d5UOg8wiPeJlclK5a6m2fmttB33oQK8jngYHbZVVoYX76B6GTp77i1TTtFM19K+rjWoVlrhnO12Qa+ZjWsZkxz+h4+TmWwyMS/lFVcHUxYJ4YxUNib5+AJv55FfdnjeNl9epJ+4HxjOY8Z3CMJTljx1yCO8qym3tinuCBUj2F+Uy4BAfRenjHqm+u+NquUNkwdsw2VlR73wgzUjtdbUxvMzFqTP1ogivVfu0H3WdxQDdZjWPydAlEmDIogHy+Ns4KlDshyUuPOcg116Rb0M9Gwmtp1Ls5NfsqPa9ciT9e7hchCwYrbA3y5CWpHBFHCOKTVssD6orKRmoB/c7l07FIpfXh9uX0W7n7lt9p2/3nYrFb0GT+s5eZ92vPNqCjLTm110L/kWJpiAdyUW3jmnf6jk2HsYEktKlb4+gCiXmHZ4mKd8UE876MLNWHwYFT84soqSZPG5pGgHFFXKoAf5AKzmlxym9MbVx747gLNYjXBbrj6BgZ7uvMWm6TFtw8IkNquFMXOplHXbxfCdpxw5sEPDANnrI9DgM3kQu4xxi4i7SeZSH1brERQqfezDs3U7jZOEyylxu1SOu7dZBo+a1jpv40rSHJafjKmyoDZWhWhWDBBxU/ANKmNPdF33XeI2qjukyLbf9nu33eYScQxsGCTpPvBsZ/IDexmsLcrW+dZxPLNG2/kP+V86BSHsuAKanvuv325CxpyOwDpLVth5gOv4TELsSoHqmF61jr/PrLacPDLsHc/4XDBLvD/CyVdWT1VYCkWJvF4r86zsoM2iUaZULWyhUerhvEEpFUGqne49SPaBoqK9PjrroHa5wNeVD+5wJRtL7aYwEemejgi8ZCcgM7PKgoyMFBh9UmtIIVUporvmlgT1D6oA3GNdfGYcioWtvUY4Lo3cgkrZEpq+M/j4lreOQDFO4Joa1UhiWOKOfzomobY6HMMp9e8V/GMnvtDLaVRRgj9xq55AUo+4tiMiZeeD2JCJuHVaoeNht51gtGsRgwNrH8bsITA+MLHiZiAMaxMC1N/we+gGoByiWUs/n8yNRPaDjHor63Mt+pSCgSyCmdcPqhAh2DF3htjc+gA7hYCSRHh2Bbq5JilWhFyEKbr5Fu3ho/OWfXKQwADhZLjhM88gAKnKRhvCAi3+NXCQMVh+uRJi4ej0NgJWhnwl5ccmn5hjHl350DTqFLwZNnJvms3OEXBkJNSuhxeMSP8rYoKOqd8IHhgUh2zKulxeXCwOPLcMAdbZs8NA4DLITS9q4vhwlaxrEjtAgxjapA7ZZnxtajwEZItLPsuZMYN1iXws92rAK8xEWYRtD0AzFbm0p7mmUX5EktOrQGQBVAhPCv7crEnyljaJJjIBnQ+F9EJaH4xZNgAuMhqijVM89Anx4d2g2kdLA2bVRQNVoLOIONu3gqUi/53TtAXJAC3RkEudkGFVcBhniAOQgSf/SV+HtNJlkec38cdfeJYh7tF3Ftg5g3PacXWB5db3X4htQax9VDDBS1ru1BzxzD4wMrH2KYQ2opN98t9zQihr6gQPG0wDF9h71iu29nbpb+ziVPbv2hPLCBiClNmS9zWJHDg+BuUg3D3XF7aAdqEahkyG5a4Luw+L62voSrRPpTK5PTG7DSlPlOqQDdbM1vKm53HkV5iJc5mQWEKfz5UGd2uGdN3+HtrJgKT5/Wjes0NJRrGkKW5Cry1/JuBc6TaRlifjVzbdyhdjX6mdjTMxpQFZXmidrx3fuVSvF+rITGzy07I9cWPCk3cq4EO44f/X/y0HFbhnEGKGC7oMi9YwnpTCePH+Z9J2Dgy5xd4z6W7CXarAHU71rzPJlxos2vX5XzmXxMhELhkMDUvGwFeXKP3kVYcttAeLHwggyU4tJfJk0PE99NZ7xZfJ52LD4aSlY13hqX9MB+ZoOkJR2pWe4q/k3aFRoQHA8sLfQJpWdoKeKXEr4dwtAat6k2mdxYBM7+TrNxOITDAgNg7iVXnWgV+L8Z6ghO151b1xMIdipTqR4yiSSJ2sitDjxVT7zkyoevQrLpGAwnvHGooG2f/nm3xZhTivDcvZtNaCyTqYoZsIXVUkpQMEoinQiI0Q0fMESFcIIXaF1wJoIKG+OxDO3cuV40qmoZvvZ2dnXdAYXHnOaFMQlQg6IEoFVqAFpqxt8FWV10ga0cj2CSRzt3K1O6SLStcXVy7RzBmNu6u6qNShLPJXbQ1VNpj5ypbv9Hj27gch8FQbkG/4khWW3Zhy6tqJ4VITe0rd3nAPUW4Be9sa3hDqL/PuQfxfyo1IAgmOPdJCDe+ogIOB2UFE4FipmVK+h3yoYQRm3sVzTIm13b+5CJQOyuQZsuZUtbiW2Bmu75RzGk9wJlgUaDoxFh6xo6KytLQgJO6uOMW8ZJHYowYjubhydfuvg3nsyeDTBMNol6+gJLzyTD/iOoezjUO6AlXnYvc9YYov2aCwPYbrXxrLKCGM2IAebbe2Q522PtakW/krjKteUC160CNk0IQqDN8nQtuVq+8/xA3d36xsNCo+HSOWvWlTDGmaSOSIdpPUa/8YAeKyqYseC2VFcpSiIm/OWeDZJnoKqEvKhE80FZXGIisJexD6OKCEHg2ybeiAEaFSqi1lphwjjoUuMqg4QruAOvwiUH3xP5HlXzRDPesGX2VmtAxGIdfMMrIRqKQ+vRhL3uihLeNM0DsK8/eYmsUFqe1JcmUec8Z5XMiHjxKA1qk6EmrZdL+NEICFB+xpSglOmHtRlcVXbGPb2yHSw9o5N0XmuckK4qIIvTpPnMthmdd+wuhXAlqfD2I3DkmxTPZl9GvuzwlzXrDZEvtobtx7Ya/FMdntASC2MoqnsDUbl4nBhjN1HjYh2Ye9InlcR5l59GXO1+QTqHnca0M21vSmgIHq1OkL1tkzEFV4i74trqq0KaitKgZAMcjoHsFziLgAPsNxjXG+HziHM2AS8wNefAW0gBq3HL4sgz/WczUE0W+P8oBEUeyx2kSmt8cSWuLbbgasm0uxy6EpoqrOof1lri/I19cjJ8lNaoFz25LxKo8TSb3xi2QosKbq2ROqxZ3bW5l+cWtkbLKKEw/u90BO7gO7ZGoXVTKA9oLVr4QJF2c5paH9lSlR1hLnuyWbwmu0SJpRrSj0hSpxXSHIdcZkeyTpMcYLlIissPfDFBrKFJ1DjJlHEiEfdTEpXnquj09X710PLMI8Mk0f6qoPzHxV0bjRMsXVl431rHQ6luOJb117bV4gTuWR3RO3/yRXXpH1YYzhicaNinFVra5rd0PN1IhIzMgftgG7DCgqb7ixzQXZMlrj5XwiCPF0YT89W5RxHKiqyGI+jxTiJ0hCvNl2UaD/z89dB+jZ5cEtDNcaWj0zU1DAXVK/blQK0JYFa5sOH5qMcrxcr8arpR/iuaHZqGWqOhYOIF3i7YOpoTCdv3kOnXOrX4QAl93DQ3QC8ahCAEXeFYWteJi+Thw+f4E2Kw4cPDfORdicpfz9Xd+oBC06DNL75YYa6A14xZ9oSeXLkWFO8Rxcv6qwkNY8iENBkHnrVLXYKIWEHVUJiNJ2vcHE6dgFD9SvbXo8ZC1lWFnz2Xob3rAcYmE33VMMD7WegjhZCXsGft2n+GteXzTGjaRDNkIpjf3FJp+aV8+HcL+Z45XUx9/uDPUvcIOkH4hpL4DPz8JrLWVrUFN0tK5vtcXNd+V6LYSVHAONBtbgCHvYfGFc+2RaRBR7ewL1S/SM5xkajEXx7eWSPCXvR12FUNxTKS9/fZw5mHgowGdN5hiOMWwHDwBKg23SpjjOL00vAdJbknQ9KxJu4Jf49InOmXTmJ8Dqde9cIteSimib9SLrMxL2kjBcO3z5Lt3KO6fZFs766k+5wwZve8WrOsviwFh/mxzHxlzxcgvwFxJ28bsa5Se2PELliHjiM3AChG0AbFHEwHrYqEuY2luvcU04QJd0lebaZ3gQn9eiuVwzHlPKEIDWVaE6UJOreQrlSVdAGiIaZcjvUbJQNmETiKk79AGqhr0RPJMk6HdetIDfUn4r9ugqAdkVM63e7+FTr1FfdW/g9sDLeXwOVayyxNjv5vk66iFYrCx3aqXeRi8sxz8NlGdEZFcBDd7qGaNtQvfzBNiY3PwA94JOgL6RUdH9NaN8QMSBjlt98S9c/kksLppUtq+Ytm41RFJNI0xWo93uiYK/Hve0ezqNNUZP3qYdIpsY1VYuqcXmDanML+t1Dz6kDgx/8P1BLAwQUAAAACACkvFBd6Q+APTkMAABIHAAAEwAAAHNyYy9yZWxhdG9yaW9fZnYucHmdWVtvGzcWfs+v4CooRoqlkWzHSaob4CR2UaDZpnY3fXANgZrhSLTnVpIj2xX0XzbYh0UL5GWLxb5Xf2y/Q87II9u5oEkQD8lzDg/P5TuHdKPROAwCobWcxoIpkWfKMJ6GTFznsQykYWIhQ5EGgiU8z2U6a7MraeZZYVgQc5lghkVFHHe04TPBFjyWITcyS/1Go/EoUlnCcm7msZwymVjpbzF8VH5f6Cytvqdci2dPq9HcJHH1nUMhrhn+5eGjR49CEbEgWwiFDZuBuW71HzH8UVlmRhieefTlnQ+0CEgRPaJd/DjjoW42aa3rhVmgu1jPhJ4EPJemiDOfyLyWrwQPJ0Zcm2arZQXLUI+W6syToXfOokwxxWTK7E6RnBVKaO985VSARAXqs3M7JFpNtJUmTlG7JGcgiyxJZElqwpiM3CR2Pt+wqOxqFMrANHVrgO8N/YQvuIw5/Oedj7wB8/yLTKZN2qFVZ75lSDMzmYkUBjQi3Gb6lEJgK5W6I1dci6CgA04QBKbQJDLgcVDEPMyYoFiCG/HtWUFQjIlYC+ZFBRybiNTw9b/X/yLSXwoeqmxALEJJWspYlgeQzWOWWhq7GQm7VcKZ3UeAijSEi6+cggjSkfM3vvLC6K4SMTeZkpn2Bpjzk8tQqmbOFTbSox9VIdriWmozyS7tyMmJwFT3qfV/uWnNpaCqVFhe9pu3Vl206NxSyxT2QSo1F+0Ym7ScFbBKQi/bCyvWl0Ykutlaub3z0H/NDT9WsFOTtmj5JpsEetGE/l3vDTdK/joJxeRVNhXKFIpPwmzyqoppEHptmYbienTMsZsTmoQj7zFzvCwUyKaSl8FfyIf1B2L+Of05fSeUJqt7OzbegyyF+7zzMw/pp+EV73zH89mhZrtfM11MtYAj/yvgX22IDZiBbOLaZxv1mBHrP5QMONzjnOl8DzWKhAUiXb+Hf0pf/1JIIIpg63+WyOJo6UBkZ3hNsJAzYXjOfdLX+4yTknAHR3+M85QJveO5byMNJdCOdyulkoTcEnH7UtyQxLOmdyJ0jmRev1+I2Gt72RUyyWu1m95RahSdFpMypXizs2/Wv5kM4dr2EgHgDDG5EX//T9M75esPTkgZtU42YHj9expIjoUKk+3KdwBhIzAb04fF3s9tcUyJDTgNJZ1k/WEhJO13H1BI/ikiiW98dOsx0uNu4rfO+1sbW3M/eeLtWBvueP0nT2BubVRTncGg56279i4ZTkQklDuudjxeu8omOEuJCCAD5prPvywfEpjfv1IwmIP4JGyVyA3STbyURcaVw0+VGECNHm3VogHp9kDFUXQi8hmBkPvE2Sa5yi6E+XjtERrpKMIRlUPfDQaJUwHRhDSCHXwZZ8FZz+HTNAtvRt5wvjs+gW+UzpjOYq4ArUkWAv1mIkFQm2yRxYYjC4ddkHobVlh/mI9P335zdNLrPX3G/nH89iX783/smKuEnVpJh0rxG/b23evDH9jebu85LS++ACaIjk+VjBlnF0U6t+G019t76g+7+T0VDoFDoZCc5UrCVDngv9Ccvf3+sA0ESXIqXiVg2dMtbKmAIbhEm8CxAgiSCdbyzLhIYoevGGFdiq4kNev3nZhjlDDAlAhlmCmf/f3w9BB7/HR0YhHsPeU/9kB/JBi2tAa9p25E+g4R1Fk6G7/KEtrd5ctG9/6wW66zpUwRdn7aWpGaQg/Ym5dHbJn4+NH396MVu/wJc4du7rA2d/Lm1E7Sz9vZtGSnnxPSEYbo79DyV1iseOzHZrlcVbSi+v5BtIL0P/9jhe/Z8YM+OYIZqJ9BF4L6C93JMoESC2GNSPGVsWj9QcuAarmF8CKBkZGIMgLTBkW0SLUVgBrAkQOLcqlIJUUMj1FT4qkIszaqygVwPxRt9H86gxvLViEGrNoyUhUJqgWCvfz27TtmY3Eu8/UfRliVqVJYDcpahL4yJQZtMx5lWRUWdKmYIMSc6gib5KNl5wEDTQtjMuqPRg0zF4loMHOTi1HDzTcQp5J3ckXBBJKIynFjfBgboVKEKfVCGfXUKrOVUSPYsmHXMd/ZKeULJ83C6qhxWiS2ajbGw/neuBoxOlCtKCPV98bDLB7fLZIAsX4lOpbjIWdzYNSo8Rh966ZQApvymKN58Xwg/8QD+jbG3k4JUM2HyymIhl0+HnYhdvsIXegx7OIgVuXvUyotrmeA0jCwEjNealxUGpf1cIQa/CMdnChDaagQ2/pl3GQX3f11t6Qg9H9bgizN10tj0/vBtpt3xLgetBRTUnxazDdq/R7RRYIS9hMqCKTMFLdTTsw35ejTco7j4joDY8LrkqLbWSesRvZped/GBUKblz3Z8bu6UNcCOIElnb0OHb97QNDrDDfE9e9Iid+ClLI7p6uNqPqAkJapotrF7ms3/NENJ6CdnIqAVxreNtI2Stp0P6Q4LN37YCj6ftfbIUIXdZbxo7FVZ/x8Q9AYl40wakEeo5W0ebPdz242GnaLh/NnY6xKifLOZ+Hgs3lEOfBFuUR097YKobSMNWpQkaDi3YxfEyzL6k6VuAYUJahcBpRvbVb2pW6HfFyrZlVHW6tf22q6NvcBTteVfoyv1qpueLvVMe4fsKbQpg3+mOhNb2z7Q1Yx1pv2j7G6Tt7y3aJ75WokzP1r/3a3i0te9UxQ3pVBeObZOHIPAOBAuTEyLcQ2Z4LGTI+Wq61ZmyJUSezFw7O1gbpuWxrubn4r5szynI/cg4o/ffYU9kBxLltRq1MkY9K/JD3zUJYWUlzh8K4Hnd6gOKIJ9UNhWVtbe1V+cXZA9O7XAsrK34rZ/Zop/9K5qv1kMqMKqTXKp5x1gAMkBQnEtApGjRAX5b61QTdPZwN3/ra3s2UWouaxGTXuaIzPTTw27ihcO29Jdve8dW6E6zGcLPrsDpHOChWITcTXLzloQSlH4/UfM3Qpus56e/OxQsq7jwvSmkJ2QN64nz7dMvzuwCSw5IjubWjYDDVStpmTBs2Uq7uhXFTWxkIWx4S8Nv55yuMb8OiJoWsiBRK8RwltJU1uJQHvUD7zZpDFRZKidHtX6KKyK7hbm3CCZKDG3VmOHjbontOsvVe0pyhXQo167QjXKTNB6CTcjGKeTNGqXfcjb3lt21nPGhUq3z/km7JT1GyGW4PUnzvd7cXqLyv09JMK1R1fg3RbUETk3i2iB0pKbosJaoKIqgrR2CDj1vSDAEfLQR122aZK3qEqVFzmgQ1klik5k/A5lUH/rkjYTtwFzQBm9TyPFF92OtNZ//Fub/fFXm/Q6UQYRAfR84hjEMv0sv/4eU8cRJEbiv7jg/1nz5/vI32Dy5nK0Bb3F1w1SUxrgDjKVDmOMI6gX3+vl193d/ID_RESTRITO_DA_EQUIPE/L2dyUCkV2R9Jm9xn93WjT6x1Mv97faBME4X74YpVwmS4Tft25kqGZ93d3e9i0Es8Lkw1yHoYynfX3XuTXq/nuklTraPmr6O8pkazme7WZXf/pASZL/o7J8v4e0exv0ewSH1/Wz0wqtlbuUlCnJWGVArt70OyLrOdiuQ96eiWQIattUi52UP9lofsHkGmvv6qfA4twaVmV7cRye9q6pHMlyNRo5eJwVdb1ZWksUo/1Vg6tqsn9np0EwC/pUSrmN/0pLvOXg9Lcvd5Xg7mTScZe3QJf3QpkeZ8qgy0pG0kpLnu1AGAPkNjN7tG4irQtx3dosaRHH0DAVefaaWSRcFkaDYaOea5Fv/oY1LR8Di1N2DbzZeWxr8lhjnOawbVJf/e+S0RrQC9DHVxIZ2k/FpFZ0T0wiysxHZpz4VcC/tLp2qmHGQy9QmraNL2wWYquvKCXdn8mzFEs6PPlzbdhs7zFtnyIPyKY/g7wTr82aDaCWAaXjXazNRov7f2ZWauNNqKsJS28EhOAdDaLRbNhqRqtwWe31MIcGsAwIh1s9Ztzo32K+XTWtLJarcGqNajOk6POAyf/BvF09ba/REJ3gv9ZzNPZqJGbzssTYGZCbX4w5wr7jBqFiTovqtmUJ7izUzNEj38Nei/CoUFlI3EUok8KhEOBtkxR5HjcARzGYrQLEbbvGZc3MrpxfcnL2LDr2ADlN/gBUq0dhtNwSLaEcgAgLNlagDU7HFKZzQ2mLxyDGw67jqVrz+9Mg5L88V+GdE+qz/p9kri9ATHWX0zJxltvpkTw6P9QSwMEFAAAAAgApLxQXc4DSnvXAwAA8QcAABMAAABjb25maWcvcHJvamV0by5qc29udVVdbts4EH7vKQg/p15Jdhx7++Rku02ALlLEyb4OxtJI5oYiBZKyoxYFeoi9wR5lb7In2aFkKqnTBkGgzDcczs83H7+8EWLSWPMX5X7yq5hsPn14f5ck84X479vf4o7y1jojnFFoBYnaFKSwoloUKHKj92Tdv/8YURpv9kZ5lDlOzkLIgEijQ8h0mk2zweqkJzZ94e/wX+c81SALNmVpcnE2mDXWwWnyO9pabPqb19ZiNzniufRdwNd22+aPis7EFSpZGqslnomHzTo6KvTSt0UINltOV6vFLFlEyOgqYm/TLJums3mapkcU1XAQaobT5Gj1sqbPRve5rWuyXOsvH42Dta5IkZuMbspDQVUo6jwG/Czr1u+O5nSZsPlr35GG45jiRU88Wg/K5KjCPVmSzd8m/JvG8KQLoKdctU7u6dTx4oVjgZ2S1c4DJ7fnThgdr48FYVVZqgaolrr1XATD52Nyw4Cek2M3U7Sqd8uWyxhHgwxM8DzyAMyP5qMvcIlQWdM2IXgWD9XksUCP0CTweGBouZqNc9uSOl4FjTnw+eAwGxM/Qjnl8Eg9F667Vhco4Zpw38GNLlrnreTLP1giDe812aqDKwNwLTfwB4e6v4mNism/jLe+vAS4v7u5hewCFnD/EW4f7n+DTXoJDxuYLxMIf/6EVzG04U6iAsz7nLOLRZKc+tT4FPFZwj9jv9G5tm7CPNxz00uUNWpok55Q0xgtmlM2L6bL+fdmHhu3HX1rCVpHBaDj8pQCXvWnUKG3LY1s31Jh2JZMs6PJGamkrqC0mPthjZPx5oJTl/Y1nD3jtXQ1+nz3cw8WD0095E6cknFpfnLPaTuZSKHtPbha/gi1VA7oYnbxaujYYFAU2KIjLprgAIT5Lg5nFCWHsDOthYGdpiwd+d4SZjUmpVRgHu8UL2dcAKnBUWBjmOAPPRu0qBSHDazVVXBcjKzYymb/TAe0hFBn/R7HWFwfWdI5AZWlzCV/dn3BUdAqrGveNO57WMarIHqh0/PIGrsz0Gv8cCrOyWP7wjyqWTDvpZNbRT0wOx+bekBbwI7Qfz+z87PXXE5PjYHJUQJeiOhqlKtTEY1k9czz0LLJrmuM35FnZVbC5aSRxfWdMM1gQV0Ihm3N37mJrfJOaONFTeh4WwphrMiZGrKUVEzGIRyMfSyVOTwPorGctu3VIp6FT7frQZKB5Y2MNcqw7HgDo8f6KkoGryX0rNp2XFKJrQoPcInK0TjW8IpSiBpuoadGGSaMsZ0gx68RsmALo1X3jisQUhfU8OvAFXF1dcMvFX/t+Wkseonva3nz9X9QSwMEFAAAAAgApLxQXcqeqHYtAQAADAIAABYAAABjb25maWcvbW9kdWxvX2NlYy5qc29uRZExb8IwEIV3fkWUuTl8tuPY3YAuSEBRiegYucEkqQJBJAxV1f9exzYgefD77undnf07iaI4N2V97tqu+olfo3h9a4cmKZNdE7+M1Xlz1GWjW1sjDuzyhb0zokA4vXWaZnc9K0qrEVTq9cqcq6H2yJPP5uAAAaWkI5uit1r4BsuiL4urOVoiQTm0L7o74imwYDtdHjYRbA/EBPg03V5qbRNdP0I4Y9TxLzNom2pxQgA5Cpk6nheb90U+NhLgiQ6RCJIpQrLQfRWwAswUMhFw93SjyiSaBP0gH25HAjwVWXBbVN/X4hQkTVOG/hEP37d+GGMIME4yzh2u9Olkx/FDc04o83y+3O7Hz9v4P9uba99055HsZuuIEpSAaE90pd7xpgczlnHKpras4snfP1BLAwQUAAAACACkvFBdg+i6xMEAAABOAQAAGAAAAGNvbmZpZy9pbnZlcnNvcl9jZWMuanNvblXPPQvCMBAG4N1fUTJruOQSk7hWBwfBQV0l1KKFWovGL8T/buJpxfF97riPRy/L2MoXbJQxZYH1U56fDjE7yQUqrRSZLxJKMwTgQLQhssI5Lgxaad++IjdCfxpziHEguRZoQZUDMMQiMXIRRZPIJHE+CMQhEXZkNNLeJkSz/Ldu729JusumX0LHFToF9MOsbcO6PlxjQctv7xt31Xb3NyGf5OuxD2XE5lzXnS3ubTK2DFVdhXs2bUJ59EWoLiXrPV9QSwMEFAAAAAgApLxQXVE+I2BcAgAA4wYAABoAAABkYXRhL3Jhdy9mb250ZXNfcHZkYXEuanNvbsXUzW7bMAwH8HufIsi5ifVBkVLP22GnPcAwBBRFrdny0dleh3bYu09ulw5I0+Y09OAcbJpUfpb+ny5ms1/tms3mdb3R+dVs7qyh1XA3jLpdbXXkwiMvvw773fzysfBHv5nqrsfxZrjqur2W9WIqWmz4my4Hv+Qt3+93/HNYyn7b3dwW/t454/xi2G+4f6y96df32j3M+vj+3YfuMKk7Oz7fjTq0BSBgcH/vtbov05pEPYZUJFIEwZoBK4aaBH1JZMQdegzX7AJOb1hSqFQBYpXssgr4SmhNBiN2elCFEoiIjUHBFilqggWumcmzhqeOKvtdeVjXkrDd+335Aq3ubtf9frfV3bia/tyq0cBShtv/6vvP9uz8A7C1lDweE3MlEzyizeCDALmMHqFkVYohUXhOjIpYE1PgEikVqgYAMlsTsw8uOcOkCCEKMub28TKBYnSxNU9EuZwiTvY14nXfc1nzTvRNhM+NfwJOvgH4I+BSBVW08aRIACY3ihIRS7P2xafnwLUEFmVHOJFCzCkWo4EKamSLlYLxVmrBaNvmZueFqw3OWmFWcekUcITXgNvB1H5lw/ZNfM9MP/BGSNakI91g2FhoBzlLyhmp9ZMMSSN4CeDMc91ctAbjkAPZUAO7bG3braYUQmLjcp3OQYqYc60KCQVybA0Dtd/2KU7qhlcTYqMy9mvhzdsExJnxTwHsXSTr7PH+NdS2MFhUIy0rJcVM4q2HdqqDI7eIJ4yz01QJUsvYpmx8YXbNEVgTc8sbYRBq4dGyBkMLXyvOkWlV7MDziRSOy+gm44vPfwBQSwMEFAAAAAgApLxQXaUZidNSFwAA70YAABkAAABkb2NzL3NlY29lc19jYXBpdHVsby5qc29utVzLbhxHlt33VwQMGyChJOtBUqaH4IIm2RR7JJkWZQqC6U4Eq6KKIWdmlDMqObLkAjyzmvXMojG7Vs9i4AYMNEbdm166/sRfMufGIzOyKrNUUmMWtln5iMeNe+4995H++jeMvcY/jH0khx/9E/uot937KLIXpnKaCLp2JlQqprnkTKuE50wwmed8KOd/yga4KF5Ocz4VeS70NBf+bfUvmcjp7UueFiLxl2U2Kaaarj/kmKAYioglKhu7P3liL2KOW5XP3+RSsaHSTGaY4I4nSm/7kbCkW2XWfMyTQUEL42yitJz/z/y/lVvqQKXs8uIIw/FVi7bPTUQ2EPk2+4LxVzItpoLNf8KdbMqHisbm+VTmWA7LVI6bGSYR2VTigl9ruTjxXYHtqYyW9/HH19cDpa+vp7diyuNXh9fXWmbX13c8n9xK92Mokim/5x70d8wPc8f9jXMY8+j6+jsMP2Rn8evu7eyQ/pPNrq9T/nKjPlPU3dz++ONyTXdySDs0Ipu/IZkpNpr/rOVAGQEMlMrxBB9yjV0OilTQ1rn2AyQyldNyW49JyrfzN0zdaDobK3aZDQXkiGEgIZwjxD2mecxxHECK41KZJgr38Sob2PMb8lJ8uRiRjnxtfuH3k+6Ou2V+7IY/9j4yf3/jXh3JcYFDDd8+6/biXAyKXKu4nL4a4re4PUl4pmJu1qrrA6pi6lV2GQh0DrmijUJP7kQGIfxNQEfx6ixaxlZ/GVs5h7BeQeKknCqd/4TxB/hpdHKiMhIkjoOb+1bIa0Ps4osjgAviZakYyiE/qI0JkagM2l/Mf6YTx1Yyo+cSG8k1HSawIfE3w5nf8QbgnYeIAlYmajr/s/kxUTkDIgSmtLBz6oGHdMENnMc51gUMTQWWtM3OHpzTXSBJviLIJRE7eWwuAW0pHuWKZCTxbyN8QSI6sS8N5ajQPHh3m9HWtcAeRwLw5oqZEybwcqwne9EGVSzjEPO2oOkeJizh94CAXKSxBBAB1hPCKZvGsgVyj3AEVgi0NmGEmxrJwhhNEgVzNH9LukWHEZxTM/ouRUoi62CtHZICHbPU9pyVPggO10yZmeP8rsBsCdnCELMrUNcLgNYLUdfbD370P3snBH/b7cfBnmKIoBrgrL9Tu2klQ4anDYhfOthBUuMSQHXElIO0gnFnCYyP1BB2yEqMT1OlR/OfCI0YGiZZ8huZAEXknfQEjzndhR6vjchLGtAYRkggqtwaXXGgsNNjM7mc0tNQeGNebkQ+LfImGD4uYHk0nKfIldbzt1KbBRMGh7/++F/zN2NALlVACfaSMsWcMbRA2mZAil3WROTQMVqeHuTCIH9h41ijgfwbUlTIISPnCXRrMu1aJMKunlw2XWUyKTQ5WnmHK4RRDueA6xYCy5uqg/Eqfu1xdTYD2kY5H+CKwjr5VOUZT8VrWEzMNNv4ASDcgiuUW73ZD5uztsfw1OasBaEntG2HUQeQAdYC2zNcUIAVHhFGh8OE3xQA6IBjy7qmShAJSZ00N6MnErIBKlFjyJNnBdk5VqQMI6SGmawJzv46XrAf1zbhfWIryo69Bracp6jbdvAjYZx4C952l/FWmsSaORwuaOja6DrKycJhZRdXJ0dfRrCGxF+w+AhHkQX2kZFnwNiF9dd0vnud3h5LZVZMGznmeaYnwmhD7oBqZBixIjN/YIqEDwrodsQ0yCN0NhPOWDCjJpjGkS29zdzO3dy6gKri5kBiQA0/8FNCz0dYJM/A1KQ23o5PExwBXGyKGxNsgPgZJGYXQF4Qvo6M3wHDFic5rNmU9pqSkaRpSNLEZ0F+oWnylRWNxgMFZJ1xSGkoyHvnEl58Aj/ajMvjw163e30dPSZ84tBu8/Q1nb6ezTrhNZhJQBCTzNogd1QMJSBKXo/siIb5lpWVrVhojffUUXey+FYi7nDq1uWBz0+N2SOqrhWWCHOXkMkxVAAQnP+VKB1pIE/GRaYrpTHmEI8MIB/IA74lEaFyrEZktx/Cs4bV2p2dOoqDH/ff7VUfgrqOkuKlinHaL7C6wKn2PouJAkg9FUTJYlL9gRLtTvUSkRcP7LfVDgwM7yIdr6vZ7Eac9xsCyGOVlepOFiUlQ7iE6iueZUJrmuTpLcC6jO+ncLdk0/3xEQPH4qbK0Z8D41cs123AcPg6T28kYSVyAwiYXEXmmXwtHDKFjli0QXxKhIne9MJh6fwtmAJY6xHBTht77zfoPMfQMBLSwMyFsQIrA5c1E0pPM5xBTU18ic2PoWy0Nbc8Nv8jeW6RmBtedhQOaOdqd7b7LSh9GiBRTWazw6cxRZjGiZ6FKB3NZrPXX8Xde1/Fvbs2pAYWyy2OG54xDYTqOVczUL/69d9+xAv4z78iCsyMPySq+tQK17la0HkOnXuJgZy4cSAwiCBm1lSBLuF47b1SIGHYuQKgn67hJYmLZhYqcbXTAFjd3TjYdMlWW3F1qiEDWP4pLdSPmDBhTFJlt4aiphmt8FqOIY+SGzE0FsxrjfHK5KSNplTe5QMw5waHbhapNLM0smyaqAwcG7D3BcyrXeVL8lyYEacDe+qCR7qREVcDph7TQyIf0IrN8QMf8L4+sxNZKrYFotWNet1ON+rTv3a7DMYats4sdlWgm3EbFjqTRu+bnI+NC8nnGi1MVaZG0EqeHLAbaf9yHualHJOVgM6CvsONOmXMuRaSbIuzG23uMwQfSIiaEbfNb1U8hk9FUId/O5j2tlwIegOnNHvdbwPncWBkalrVjMRTewbG/mg1zedvJ6SbcOHuWF46O0erw4H4TI87QkuiiPEnQhrgk9ktSYP1vBC6uBHgsmSq4FihH0Q9vKU0T5ACyJVpn/ttAegaMedZvx/bFYe5nuU4tNUfLsLTa3CQ9bmVk/lbDNSa8+k3hJmXKr2BlqXG8ejihWc78Bw2YCcjaz3aBwDW64JxZRyA8E69VEu60f0kYv1PMMreJzjIBC4dyKjssGziwEe0WKPhJnVnx6YYMSleEUOSNusnLBOVuQsPnbPUzuLBA0LqQJwufaO21LQSZvvS4Q+d0m2zmhyhvIRQuzSR3amE4tdUagBtcGsWpYZWKDffk6uNcIykn9BD4sk5MfmRylPh9dcQ6FdrYJgc6OFGb0tvhlexyVbaW1ct7FvBk01tpmFEWDF7WpH6uQXpoGzRwGgh7k/EK3o55RNDpTWJhqJy2oiG3StcDsickWXEnAguJRaFVYlSE1Xh3m8HZn+/TlvfjcVu7MePnZsb8PbQsy4gUWooEQAeGhETYiNoxgmm8zckwlYgLsefVT7XpDbgp5My/QI/VdYSqqwhG6mpgm6BEgyW87DvBud5hog/q5hNf++Xv9D2XL0BV3r7XVyq0u/EkOB8bo2TDTSlGaC1TQzFSGb1nQiLU+v6KPMjwwWR1zPkmWLGXPhxgLkwtXpqkl8OdhbiI5mQXdALcis9ItzNQGqwXGywKgCUqD6g+YBEuH0EWapSYLgMytvyPDKOhNvI7RZjmEER8tiJquWx2w4i7w33pEkfbK5TlZGH9WRv4HfvuRJNecf8ojvmoY3r6zFPUx7rLf8X36QkMeWIr69FkhzedvA2x0s8mdziwRarEKhjVX0xIoUpNBYSZ2OVvNkyHHl25bQrkBCONhCwTYYTmIYy9WOXx2gswoELoG1thil7It7aUmQPgjYyBoizAiJZu3rz2fumjhfLM4E3360KOjr+/Pziqj2ZFdRnjA4GlRabnzEpCO58hAiUvMWi7DREuk9rVNUyquXaw5LhOOM3uQTesJYzWSRldSp0662ENlqqegYga7QSLs1TaMo3WnYNsdTqCWblxk9jYlsVCizLC5BBWAqXxVY2RFUBBQTHguKRXSVzWKs2GKPjK0xq4vw65YZPIAMNWt/v9ncjdprf6F9//E9/EXOJXLwKrpglVpGCOTbYgTJd1LF8bQ0Hblx1S9lHUtknfBjBaDGb3Vsm8e+s+wxljhGNDS4rbGHlpxnUDyBrHHCdiga5SwPT+d8z0mXy9yMBZ2FRag+IsA+toYp1kdqoezmSrtdt6wHUh/DzXpjK6tUe21+DLbxvQehE8nE2f0sxtgo6B2rFzIWKing5SArdHmnvNETaj8pUDIYOMx9BBuNDwG1gNF3MblW5qYGCKx9Ii/7fcggjawR2UzLGuGhOlVaHUpFY0wcoUkrmsL/HnnU20l/+l/3zpsvPHN6P9nfZM6bp+lu6rp05m//JpEt1kMEhYQYBta0ErchsGKz39jDVDmW4yZNQ2j+t5W7WSGUtUvDOhs9htXU9PLICcFmjlnzyEZTCZgaG9ezWsAxlOi4hZYipv3fnwjl/UKmFUuXJ/8G81HtmnZ42qgLkPaZ8pU3xhmrVioLGMqlDgVclHiRFV1PkdcAQaNnx6XG00LgzErZm3qjsjZknp/dDFwNaY0nrzCm5zwYFfJNm53AtVxG7oP+YzC3Mj7WhCCwoHVm1NxinSAWP2tI0FB9Y+uUvx/XF0fXedrfbBcyAsm32xJ4BLQfGaWzHtJnnBIGpic48XQ8hZ7k2REKBnq9StCDm/PA8frh1Hne/BgV9OdnYuLp3/iTWmx2+udX7Zqv8+SR+rW9nnq5eHF6dr8ZOaTxYKbiyfWdFvHoH5oMwO/dyJscnqMZp08jDOr84MBk8O3rmilaaGsMWnFaQiW/H1n4b9+x110DdXmzXEbtSJx1VmAu+7+8HB97OQRcFRlygDMjLoPDRxUUrGhuKqJVPag5TKbgaqNyUBk0Obv5zVvAPQSYWFsg8Yv3t/f1umd/B7102zguQlY51u8p26Zg0g2Ynxz6L3Bi8nqY2xtNaDXxJGcKhRiCqelKUQnQHYQjiSGzJ9SwIm+212zPFTuKFiaDE5RdlwNOvlsnmf2D3GdVssrE2jUpmzXYMKiVS7on1+uHG0kARRzJPidQP+I2PhkwGRRWUCZNjn7LCTJDX0zaAXsX88HGsr+LUg+/cXJmcV1cu7DO4eBFerNzeyTF8IZ66vqZkVfxigxLGiDjjF+9wgXwMR8BNMtN3BRlv5eLLFtd4aYXWMfuiXUsdVkYP2NX8D+eujDv/+U64VPBIScT6yrRRDiiaqPrEoBNlvawq1q70lft1CAc/dtaIJXdiB5FYJMIYshUe1ATGUQUeEax8qGqSg94dX161wnZvCbYmHs195H18bMOZY2qLSUNEeyR9kCcN5VxanYgZr19WdH13gi/x6HDa5viR7Bi7uHrGp1NfA7INkCXVTuRkAl0hDFbRdVh2QVCy02XfPjMI9LMdsP6n0X1z1dfercYsEMqAVLEr+BFSGkpEj7h8WSU0SU1dsWaqJq7JRlgMQzWC+iJHrFGmo2QbZEPkHR2bqk0qM3KwU76xCMvNxQtR/fUIkeZs9k0TqoeDbgSJGGgvv9Ix8wU3MpW2B58l9akdzxo1oiPbxuKtmbUXrqvizokcYiXDTs2r1Kzk68PWMBj2bBBk7IVtd6rMp9MK5/aHKzx4r/d+ID/rfhqXqFnpik0YWGEhKtW2BMuSsNjRcSvI7y+B/Dxs6fPNggvNTlStAAuUH+SRK4TbBjdmelV8r7zJJ5ddTcwwWU11S+0X01jnsasOjZ0b3HQU1oEz5NV8FS22vtmkg+3eqOpKrTWaPOkLABtEd5sduSam8jHqkNBGQr4Bjhxr5SCsp6H4kqylLZOIhQ8E4H+KPKP2HdOP1sqQT8v23Yta+27Z31teO+xG/b3ra1Yi7rYNbrXzLg2uycf5q+VummH3hUnu5fbbg0oIkQUeEaS0zCeZfsGDUnau/26x1xeKpygxXvZcDykTtcLD9vprQGw/xjZwxDGURCyS4s9it6YYE9V67c963fKePev2tgm3Lddt2rGPR1UDEmmJmloOYrYmtVcM1RbO7jbkbD+H/cmom9cszFRXjAuzSdKh569L8Pzd/O/AJHsib4TM1TI2T/V0/mYsrT/ID_RESTRITO_DA_EQUIPE/OY7qlt+ZXVgxduQ7BilRa2NLUS4Ze6ZzNfpj135ABYSh0HfSBl6mCSBfqLeOMNpfqc1J21ITjfLKk+cq1GlHpQfdafziEP9sncavX9zrzQJQEmZf1B/r0WMlOkcy40mrRzwRJrOo5QJIV7f0VktPfTjj5K47YdXck2Lf40tp1jTonCopxgEb8+zWfMxww0aUrCRBpkwZspmvy39rrrHWV7hGTbbXi60OWUgKU1h6FxzJB3g1CdS53jrQgr3lhOqFaVKtWljrHDEIaKk41ZRcfVhQCedSpkF/aFBuDfuTTZLVG8uT487RcZ3Qi0FinAqtYv+zHbDQSQP+nudkmasOZSrGK1P6rKIBnO5zHvToMh7MayQ44BNIizaNR0fho/7Bo8UHt9mDy4vOxeWDeqSqjFc1kelASJuw8mtzn/TY8pNwftemcpX1xq04fB7nhw8WSySdMOWaU87V09fniFRPFwhv5yLuVvdHtftHC/eXBj7shQ7322ed9Pet3U/n1K4Hm5Kb9MBNLoKC098MC7EWrBniz7nt9qKYiz6RgpQpH+WEVXOq9KlRSH9Az5JbX3QjfrvoaWX9q8AmV1tBdmeNfFSvH09qkIkvngTetd+LsXzqaordwlvx/DyPoKQRaZ8we87KPZu+Adnayb/bkBJ+wl85mWBNlAmDbaOem7qxtV9yUd+5/cqpxbGuxvSCyQiSaRm3DLOkuSXlbvpo1FHViyclJVNl7G0TFfXPGyqSOvTzWHlts4cD3H2o3Xahc2lVExWlwzB8eeKYi+nLTvHueWabjnglHT8fFVwmIqH5GsxUW5D65BBg6wC/HlsP4wEu5VvAaHVJ4xLHpVELpmpicUy6SuW9K+X72ObK0htJ30t4uYUMtPzAxQja9A/aBq6D5Tks+HSdQjghVWnkpAxITE/Q/zfo2nB18SSCPkSkDqICVO3jrlZcLSd3r8Jst7dj2rbReEMU5HqXY8jVWHpkXIA53wVGCvfjW4FcIqCRjR5ROl6wOzn/yaRJx5n9fJuUJc997qzqqhqKKX1jlvlegFM8dOjGV7/++3+UOrfNHn1+GrFHR6d468mjy1N7/kDTt88OWIabnay6PKSTkfabjaQMV6vAke6UlUa/YcpNOZD6r6Dp81QXQdAWRvT5GOaczH82JQ/+otAmi6lV6rN+FKO3NfpilQgybwgCHnnYEC4pcOVEZuL1D+KH0ofSdigm/S6fvg4eEXB81TO/7x9Sjtd9CaeppyeWeML+2Pg+llt2xu832x2mtX31gNS3hoIfcGorgQY0Y/t3Bdn2sqegzGp4wblQtdYvUmvPZ8c1nQg/16GPTainA5MNTR0ogyPBHmkF3HRoUiRjoc6J7pQQKDPMK3oWP31ffrwT22oTdSoGIexujCflsFC26mPSKOH9vep+Q3Tcux+Xn4xg4PLZ8JFPY9O9A+jaKLk+/n58R5E30U6wi/DO+37y86hsgiJzIk0rUQ74/Fl3aPSOocy+gdR+YWuGrwrkLgxsrR/vNqS+jdGojFrAGIalYyyDeoLu/I9MDCur/96R+KreKQhaqNwlh+3H8gv9FtRWViXHmypXpolHVG2aukg6CaGjowxGmr78CatTlc0RrNftMurFgEykbc3irrRFbRPmSipth5/lnZScioK2/W7U60cwS1rlVsTd6P4OE/VX76S2BZputLNnPmMC9Ca5FJZUuda/8gMHw2dceZ+y660JcnFostPs7MhbrC8DZo9FUae064PcMo9uNj8qM5Nfn/Ii1mdH90YLt5ZG9WNcPjg7DvtCvquGWRpkYSHv+raiahokT3bjs0ZVT25b1cw5CxcmJmJMopf1T2YOTKrAaZBzvZHZTVAWI3/q/8cd1PZdvKSqY7kS39fi86TGhxFwTH/cCkJUa9yqtWeuwY76u6bxMnatd8beBBHJnr1rFjmofYmxfu9mjfJ3QE2J6ZXfDlirVP0vSIbuf9USdERa4/Sbb/4PUEsDBBQAAAAIAKS8UF34IbkalRwAAPxOAAAbAAAAc2NyaXB0cy9nZXJhcl9kb2N1bWVudG9zLnB5tVzbbtxGmr73UxABDHbHHKq7dbAjgQsosnyY9UGRFGcDjUCwm9Wtsnlos9iybMPAzGIv9np3gbkd71wsMkCAQYJ9gfQD7DvkSfb7/6rioQ+yPDurRBZZrON/+P4Dq/jFF188kJfCyaKS/tx/fvBPTiwSXBfRMBHKc97I8sLJZqko5ChKnJKLnSiLnVJclc64iFLcKzGNiqgUsf/FF1/cGhd56kyj8iKRQ0em07wonSPc3jLXL1We2eu0Ua7eKns5xQgRxlHONNb9xfnoynZ2Px9hRllZP/HVRVSIuBqt9JyD1HMeZ6MLWsXxw68P8iQvGg0EFuXzGkyb7+6H+08eP3wWHu0f7z883j96tFSbFt+ofrr/9ZND3ejp4bNTj8oODp88CV8cHp8+Pth/Uj9rdJVfpYnt5DmuDxOxsBaq4WfKVnqd3bp1/Pz5aUA07IThWCYiDLt+IVSeXIpO1wfx0YM665/fAg19Ir0vMyWKstPzVFl0qPmGO5TDROaTIhrLyO12La3HeQEuRKMoD6NhVhLR6e8tFV2KMDakDqjIbxXdev7taaB7zmfldFYq99bByYsAxRsuaCWSSG2M1CVKHzwMiOl+kkex6pjpjPJsLCcb0yJ/KcrcpwourSqKQ+JLBzN88Pjht8eHJ63G3H2Bzsu8kDlGwNyTfJKDLpNZEamVHZ0cHpw+fv7sZMU0sBy1ocQoFyocRVNZzpLVk3m0fxIeH4K3J4fBMM+TzopZqVmah+JKjGYg56pe/IkoO6gIDVMiPHq+73rvP5hSEATFExGHF/msUK7XIxI8f3YauKeS1OyZeOMc52mUubdOnn97fHAYuA/yrBS7DsgxzIto/l/zP+dOHDni9UxOhRNBkYpSFlBqlMa5goA5Ry/u73/jOUqqUqSRM+j37jqdQW+w0/XdW7duxWLsXEbJTHSuvFhOZKmCQXf3loMfOXagg5hHngknL5yOVBCzMspGVHkMUpRdxoaUJVBlUda56nZ3C1HOisxxn/HsxFhmMs5d22Wrk47MStNT1W7svr/a9d/ryXwYf3BB0WkSob7rux7+63JXpjbJ+1XXLIRgCQI/veiAzR7xwBvmSRw8iBIlvFdCTM3lCAItCn1jVjsN0MaP4jise+nuTf3qLtSa41M3IQFlmGGAgG65g1GeTqNRqYWFhiWa6YHoimbjY+FFqahxp2OYiQU9y8uI/h6LsSjmf8lGEkqZQkviPJn/PAEUq11ocNeS0Iyk581zX57lWBaqDBOZiVBmMan0QdrprVwQV1LoUWaToH9dp1RJhEOBWxEclZ0dmg0vVYCQzurudaNoTASnNt3GCFEiJxkDzjIe+wdA0sNjXrCmIg+youKTwwenWiaCKXOwmGUdond3r/BZAOgfXI9BcT+DDQtIz2yBku94Nf1B1xaNyHz4xWQYWFsCaMV/LdGbGqnLxJsKJTulLBPh5VORgZpGtPAwsEaso7sAbbAcNgWOzJwE2kkyCwF5C4vrh+aZL7Huzuus477ZnX4dFyQFNdttJcCJtgkdQp80h5Uwj64f7O87CpsMjfKzorYbNFI9DSI+zeEMQg/pSCD1p0Qx/H0E3ATNnH7jeoDrg2haSiDreT0lVQY1sc6oz/M9VS6y15Zcz0zd35IkWMbZQcw0z5dFZm0llrzTYibW4osWFtPqJmBT93YT1dE1W5hoBbNa3giqHMIoT+E+SEgeTyngf/dWPY9m5UVeBO7DYjbNnZOjh4fHvd7WjvPtg6Ov3bW9qtnwpQA2ug+iInVO8iQqnP2iiN5q86StUjQsZAIj9nKWXZDhcGClttymxqFro3MwfzFk1gigR0z3YMYCF9ZlQuLEHOpbUzYMGr4Xibl7h5p094a+gi3Woo/2bpd6aRerdyglM0NddtvPWLTw2O3xjzFNVmOiKagdd4bWPLFHqU0TM/4CUg7HwCvyN8p7I+PyQgVkbL1JIa3RyvJSBC7WAx9hJALtCNQGOhFZh5p3/2GzV+vH6GKWvVLBGT05k7vyzmDznNVPku4VUTYRUIGqrTfY7J5Xjbmex11QbcHhALz9ju61gQzWou/Wkj2BeYAD9KrTbdWCb1gGMghoSNPNb/oLNeBBBuQUjZKZgufg8vLQTsM+PSllNou4HF31FsrZGXJbfZKDlpTBAtnvuE7HvcPj3XG7bsUFnpdhA3PAa3XW+iGuLMzPsmixuCaEEWI9qxW6qYWCVLz+R7cuK+zQayGuBX0P0qeYpGYJxkMo29iwELcYbNgrSZXzsSy1oOn5QGNVUPphOUx8/B4VXEw4xo/gVUCetejj8deshMqtPRNyBeAxgjzsNe7qVsZa4KGuqJVXLepks8fabsUTYzDKfAprkIhxiT/DvCzzFBeFnFxQAZxKGYtH1dWLpr0QGZEtDoirzopOTW+1Dhh4MRP1qL5noYXWaTo0PM4k7Bj7QjwAF1oToilgkUD3V5FLC5v1sXcNBJz1dzaaXD3/snmnKQP3DyiMighBOrpdd8+2f/Olfs4EfEML1U/OK6pCcmZppoWdnr+T0w4bSpRaKOru6nufb8l95As9AT0biApDTO/coPR0BU8fcVWYN90GwlWQLxHmRcgSXRwV8CgMhdBFzfuRSBJPB+ID_RESTRITO_DA_EQUIPE+RuMziUtBFwaWw+K6p8ckB3uK88MXeOnJlYDP0cRNHSBYFR2Mk1kCYKh+vXUosrd9rTzxKPptUemjnj+C+DNC2mzV52hi/PuHj+6JPM9ipKwBShrkh5Np6MaYBpwR2E5aq1iRKto1xwGo2kLX0YL8GJ/KB4YtlAGDQ3EjIZLvS4K5KiNMU3i3QxrzneX7MIivnQqKSA7RSHyOsTpLlYnVTeC8pt+1TQIbIvu0uAGq0bD/wtM1QyrZGuBlGlUTACty/R8GhVraOllvOSOXvPOdtfrVMTUd5rC93p0bZAcN6sovOzA0QjdPdHwxt4YRy3rtsvLt1NBTlp8FUGrzELsOsU6Aph6y0ub0qpYrCv7rZZn/OlIeHBNuFzH2INljt88hr+mxZoEwYqm10YaFHuTSHWs7AM+IPTdtcH6CpSwhC2IsNDlWaZ2bxasc4jVVLabBfCYr3bWGimUliumn+qMEedzyOHrdpcsBE3Y2MDf9M81xu7eTGA+I9jDdGn8dZPV+SMHIQ1Nck23ZQ78vYDgVb0ad7Q0wQllNjscFNlE3DTWucyRuuwcnLzY4Kd3XJ+SvF0b01QvC7RrqjqNnEcrMeKe6hwx5SRNEhjEKfPLPCkjOcrd2t2+Lkr0VoeJvnPMbjVnPTGn+Q+YFC6JGKhbiNhmS3OY3en8R877OnW84O4rx2SxHQo/AGCynH+8FFJ5lHBzymL+F1ShrCv6RQws30HQItgh4SiRAlPhjoBPqFxM0fVpTlPB/xBN0CeGOtM18yR11AwLiDIMRysYzT8mI5qQAVJ49cQNV6gyKqUiGxzGQo0KWcrLyCZAKQg4Oxu77wv/Miokkf+D08HdLJPlB0Q2hIUSpKfniYy7HNzO4A5APeidD4I1BOiUXKuenLmpiDLYpWZBLFtFvirjUEXpNBFdHVaGHquu4kQSzatj4sk69DLsd/rOr7//D+eQFjb/kVfm1CsjWkxzJTWrhBNLhQlyNFhLx5n7Aqsl1iSogrXGIDxMaka5onwIl2UGnndugwDu0/kPmDxd8CLo6v6Re64j7rNtv+/1/R1v4N/D75b9Pa/H0tH3U6FSMHLL3+r1SaLAyFhCQ3K1R4WDPhaPhYM8qCVoyiyGU5FAGrAyEsYciAN59p37R06E3iBICctVHOfjoO87+7izk5//gJsMAl5GnpMRLaLGUxKoSxnP0AHJThRHEMBMAe/cOq+zMhxfZsdAs+OqJAeKRPn1jF4hkJZSCprXzSu+AQOezn/MZApWud9QDu+bTS77eKXLHn9z7DYoe63gfnlmpe8VPCSSsFfaeUolRpilrSHS6MqU8RDn60WyMfq23wOrl37PTb4FPQXfbP76r//2DXMmzUFj5sM4pxTKKJnNf4wjjS6XhhggGuUiQIRZpD7JitSoeCpKIJX1RjkrAa12H0A5AGAOvTiaESdiCQTBGhceSHqHpHJmkLv/cqZK4YgU+EbwhyuNlnh2WBWigRwzItHzNJJ563FJoKOfMcK66zR5U4sOiKu03pb61YWTRUq/ouA3VP8NbZkWoJmcAixbYnRQVSIFZwV++vWh03n1HevufnV5/PTEXLckSFPrTJ43kC5roBw68zYbt/utW+oU9yYx5nUgMd122JQ2hKclPTuAjW3Iy479tXJD1AgUxDEBCSE++VARLsSwB88RtFNMXnOHZCkilhG5VU5WUlQs23NyEjuCkLFhOXAlhckYCfMCnno9Hty1orbMoa0GhzJOt8t3jEwwUfRKtYJZgWVCtSwnrmER84ehNdNM4WsSzONfflrHHUtxam0tTpMT3NWqB4W33eDXAHd/G7cGzC36HdCv5dYzSxNNiCnRLGWL4VjGRQyEBvPJjwC6j9hUjRqUAR9++Snog+EnJ4cbJyenezVaqNkQZq4kwMAQ5OX+8lOLb2sh4nULIiIVonUITKYYBkBzoYL3W7vuvlbw7V33Kavyzq77W9bbD2ukYrshFZA5FTHaLwhbSwiewvNZZ2A/R1/1rM+0ovJNd53qrnVQPkepjQ14vcYGDIzxJ7m4a3+tbJwYfVzSQjh+CAsjKGjTPDoUQRV5ZZBTkGy9Yu40WMB7G+TlZzPjMzRxJd3/7kr5CWpveuY/S+H7IssJdGDXdq3akU2t7EhDB5tUdbRTR86KcaTIHRsLeDAOMUEQY0RFfdhko0e44l0Q1HloLGrYMKBdXyb5yCYzl7l2l7l2oHsxc9QDArw1UNS9OaI22g3u3cTGNTna4qNVFSykoSx011IKLthfKDCK0ShZZPB5WzcWuEXCWm3mIPDLV60aCGhW7bOPb0IqC5ZMEuMTGMpJA5VweWfoTttODao3xUeRiWIClnIiw4ZaodGmtTp4T+ugbgtu1f61XlLl0JN+k9W8MSQSAXDz9LsLuj3R3oAt+CxAxJolQtcb4eJhqPVFxKZV+Oq7i41+r9drKu9hyNCystLNQNPfZOcHokH/+tuVywyKTao9ScJo9IJrOP8j3Oxt58J3fmuCX4qjIkPjaD1iftXiVo0MUo+asGVNmSnLXDoWE2hsQSx0D214Rtemu7TJsc9hUEqOUzieJUnID8NCwLWKVbd7XSVxRT5Ei60N5jUqfi5viCcD/q24cpqXFCbmFox0yJIzqZh4ssi9yp+kirwEGL6ZogLEuM6kmH8cczYFvORMgvV6eIfXrJTkQ8GzpeCj5AEpsD16vk+jge2JHvXo+JMq/dao9FspkvjTiNzvsVgcFXmMWVxyIKrYoVVwswXlh7Iqz2JkrCUgjy1gQBpeRAn//dYEtC1BcL/HI82tt/73RXhBSuVeoBIeRdpec+dVnUhrWqNmnRdEm7GBulabsWkDxuN3yi1JCHCtRxrXot9qVknP6pbNkY+O7ci57QIy9uVb/+hYj85tb/N4qFtFMBYHlxotwE7VvDnok9EKEj0ZLZEINdWqmmqxZpM59xah6OjXf/59cO+rTQerhyyMEhZueA7fF57zfYTfMYQEi4OMI9xiR46ci49qRHIKKdepntoIoGlU2SljxHBz/4CSMYJeF5CzCLe+P0DPx5u9KpNXQsSbduptyEXr7VK/34I6wjUo1vzjROa1Z9gS4kPzuIFoK6CM0nYYuIIcM5t1GMOVWzhz1u97y7mx+4Jse5WzG0WUyATz9nTqc1TwGxvKhM4ogKVt3try20UpTVdJFOS8RArH4p2E7iJOjcAhcpFhyEHc3j0PFO4ThfuDT2KJEllF+EvAHL/CCClTRrlGWb5dzwKdFTuhqkOZMByQVKjZS30NGKTZJisC5hNbJ1JqlrJZYa8AbqOVGsoHplhog1/7B6tYVisamJdjItkk5G6hZb2uV/iU0EcEGyq4AlEh84Y92T/4lIUHJdbYkS3vLhRqy6rTPlxms7/GiYCXMs+MTZn/HM+S3IYwkY6Y5n9yEiHZIYIJMEYCasPcN2SE2bFmF/58K46KyNUXJPkmaZQQ5+8S0wf3Kqc+mlqnPqKXXuXbmzF2cxVjcyeRKShBXCB1s+5si69PdBXD1bztxt/AdyfXjXxgSbvIqw3NRxEiysBBM3dthNv23JcDsxZTQY41TN3RTlvt0TdTO8SWCzmd/1xS6GRccgU/kZlNSsvJ7FQkF5TBsK9aqBknCQn5Ni1z5NhpbJ7nd9j8/qith0ZklX6d5IsruGjV26N1ir33epVK264ar6FX8F5nw/aToaDspjMROSdWdLKSI5Q8IwGmFH4pU1p1QwK0FByIbP6xYKh9xA6OrUrcv9hIf/mpxfYKeRf18xGdBWBjnQ6u9+7O+j3y5lpd1m8lMPm+39u8S++jCFkTMZn/yKlo9o0v9cuTOsSSBSyhNDyvfGfQA4SAFQQu+8uwF2mSpRHqZzr4WyYVE7GoN7A7cPR7JBZbv/7+34/7O6y/X/kNFllWDuX0MqyV43ombi87fNqUc1rTcEc5Xz8+erHEvIeW4+AePICO8Y9IadGBhulHRMcbsZAhFmwL++iDWFnrqEZs609pl6gZ1clsOiv1qY/ujTx7rcDeFtS3Sl4eVIs1u3Sk5gFUAv017ZNG2MgZRgr+uaBjQBG9udHQabmctZKhHr/TbITrqDGORhcUpBFT2QZ/9TeE6I9OjkId9IXWIV+L1jstTwhTwXopVDfC/noGm5NoObzezb8vo2o/8acCdAjDySOnw1fL4X0L06exzyd1yiidgrFx9DZEuBIlXbhaxRiKAYS/HW/cTt1WAC4iNVuMzxusXg7VVwXymCRjiDHyF20ZM6XWgVXXZBC8/k0Dfwot+/xLOdO+D3Gs35hhPgHPaKPTh9/NUAgsMVnURjrFvDmt0irmBWc8a5giSmlfmDemTgZB1FFlE7CAYr6TVW9bKeiLSgND45nK2b/8MwWwDmzPCs8cMWLZPrxGp/E2zC6FMM7DI3NUjcrdvdZhOJZT6qF1IIQK7FZvcVXO4JnpE5PrN0V8M4viIr/Rpoj7nHzT6ujp811ew2hjdYmcGBrO/wTD8BW/exBWgVsbH54retPLQ6/c7aB3MxAqJ6KM2hsl6PwoBPgdp2sFQNggvk7C8mUVInnV3gYCHANZe+BQFrO7aOZrzypyooFUvN6wMaZke/3ui+I1RGkTDoYmdAihYij4HzT3rDBzzSo34BvO+NAjyBYy8bTjsYg9miFmm8J9fYhOKtg7Y1eFcRaZoOD3jHxBfaIG49sttF2P71jnlF/mXKFLOw7gV/ub3t1zvcW7wV0eqt7nQqANFyvSyUjDCBssQV0RJk0LAWpfSP1GjxKywOhctYH7kxDNHi2fsFHBWQMzm9ZysP3LX/ec6B2QEsrcv9f75a+Mo2aC4GcEeU1BKcq6srgRELIbohw6wSrpqGYLPelcT21+dIxOY5m4vX0cTieQTNi+55D3HOn8872vBt49DvSFc3B4gPu+t7PFJQvjnVJKqCDRpWEeRPBbMudbShUMtvfo4g/BjndviyEfqjaS2rJMILR6MxEoAB29QrsKqEbzHxIYj6axpBeJxfxnRC/RwgRsaIgRerc9Z3AbU96+TeNVppy8rZXRVjuo0tZZZjV9KMvK8RU4myyMe6TD7vsHtGzr/A1u7zmphFtQji74ZgSX7opZB4eTZvWohhYYyFJOE6gmhXk2PsiHpeTQgFKv1eLr3SkL09CeNy0enuEeb1mp3cye1+859Ger1x7abOAqiyhTja1BV9MEOlDOfwaZNTUmEaX7hhJeCmUcI4jukF+6LkzjcXPHxGBrz9nsQViIgEk7xON11btb2m4x5F0UfI6eZje46+2gEyaiPpdsaQHfgH0LJhg/LVJOdLclA5PNJpySvoKUJmZbG6AZkoeI+o/ODhMi4dSPe1h7PlSxj5omFNfMYFi0dlXSZsBUC0Zj78fCDMhhJjJKTWg+50erhtvhwJTvOf/zhwA8GuDiX9gsB5TIp7tLqQgaExRsbpP3VOSQfaFTsTBPeTn/gdxTSM3Jo4cHTbeNEpGQKj0hzcUR+bJUYF4BmUkuJYANTOtEzVFUzP+TwIoNqTatDJzNoHZN8qxua1PAjqgMHO0pSlnudaLn3GugpXe26d/ztgnN/WU831dWdvReqlETU0wGyBrf7zY6oLHzj10M/Z2j6O5n3HmN7QapyS/SBshjhIQUb3NIVfkzfHpIBe/7u+4Da12ZBIUYIdAw3pTrDXbd/XQo9WYTK6Emj7W56z7l5OsErKBMTG5DSNfb2oUHUqXVWy8CzFYDPt85S+Eu16fM+t72J0L66qHez9g+h0a2z7Gn+ZcOnqkzV8buefNUNW//5jl077ggzfK+bBrGbvE+Mz3ccR33Dq5LfXzzjvu7jO/zN4g9FqNB+0PPzYcQuMVh5Qnt6t7INYKB5g7932UP9AcL6KHrOa7/MpdZ58pXdNij44Zu96ynHfErzs6dufogLXrnpaydBMNWpI/I0jz4WPnCMKiG2Nx2db4qxB5bnXqv6Xdn8IGV670WrDNdev5hKaw+MU4mbyUiiYVu8OZAzm+WhZjwu/raUUS5yaaJZsZANfZlfuXR7w6FF2290rpFXxwo5Dvy7NHH06h4FYNVevvdD4gWonRIf/MhHGraWuFxzpnTFPxikk7WK9+pQzsEhPOPvBlVZ77kJGMUakp5K9VfbUmADzSNlmMK63ZyTGGc/BvHFHvL8cTioWpYYjHJi7ceOVd8+rV18NzjPYqxCKMkaX3lAFRPVWA+tMEnKeuKesv+WVHvxW/UK85cO6Z7HgT2+nx1XNM+BV+hg9QhZr1ni6eDCLR5JorRNWhvcR+7Wnuc99JIZXH2BQ/yBUnkqjOTBiPkP/R3TZ/LO+NrHLIHLqrd8fRDX1bRnzwpSBsT0kWQATCa08U0mzSODN74+xELp0C2m99D+H/5jMNNT4rTjz4M1t/mU1aGx0Hgmo+tuJzMlUHQN6eHduph5Kjx8YWuJoQckR1lUCZadr2Fo4TdZnMfK6Q1kG4cFXx8x+Wt4a7Hwme+AtBdFBX+1sqCvHy6wc1JfX2rNVS/vtHSJzHag3zywE1DQpurNofD1zy99vMiLWtRtKzFArRZqeBP6Vy5GxaAPgPM0giDGJ1ntQrOlo+ReIsplNoILIKhyx85GvGejofmcgls6ZHdxyAqt/gmmRe2OQd5YoycTYpwbIvgwegG7dyltAZiELg+o3zp6MmuM9iu91J4zgz1k9lVrt1z3nnPOwjtzJpJF57CfePTai/W+I71e1M++YJ5JAIha0ztnyu7W1Eqmw/DLI/uP0AgQN9T4jTw0bOHmOYOHP14KnnDB30oCB4r4RwZbgX/Ea64xtZruFCvhhjxoL5bwYvG05tygJusej/IuT8s8Kq0yYdWeAhC7KuKS5RLKSgDTk3K+V/gLeicGKd2DOka6TB+j11TkWinSWbzixQAR0qnHgvt/rCHQJmX64hlEBURRjKjTYz8RaoHLyyBTGmdvLMfcLqORIfq9Uzzjvz7+Y9m105OCZOMM2ZRAeF8mXutlzj8Togjew9LRSBdbYkWCZylQqfkDH1RSsEiS5fQ0WozzssrSbtGMt1ufRSdD8cxCuwSR0ttLbrelNx6+tYMQJkO/XkQL3UR1Abefi6KmgJYYLDCkIAoDGGwwpBAJgzdXQ02t/4XUEsDBBQAAAAIAKS8UF1qgqZq5ywAAM53AAAfAAAAc2NyaXB0cy9nZXJhcl9kb3NzaWVfdGVjbmljby5wea19S28bWZbm3r8ikEBOkJXhEEk97JSaAzAlOa1pP9SWbKNGEAJXjEspUsEIOiIoy3YbqGr0omcxm64GEphVZ82ikQUYKFRObWqZ3Pf8h/wl851zH3EjSNqu7jEybTLivu95fOdxL7/44ov9fDpLZSUDb1bkNzIT2VjeFa9FIb0kw/cqL954k7zwXhdJlWSXXnUl8X1eeOMrMatk4ZVyXCV5VoZffPHFnUmRT72ZqK7S5MJLprO8qLxjfL2jP1+Jkl4FSR58V+ZZUMigfFMGb5PZJEmlKZXNp7M3nii9bGYezUQW4wH+m8Wqlzgf34blFUYa254q55VEK2Elbyvz9uVBNHp09O2T6Hj0bPTts9HxQ6d0fjtNw6w0ZV9ld+48e/r0dEiD70QRjS6KumEhyzy9kZ1uOEPHWVWe9c/vYAYhzTlMslIWVacXlFXRoeobfjkukllV+t2umcmlLEQRodP5lJaX5/Q6L+I7d54/OTqNHj89eP7ocPjOPznd93f9l37gH9tPo2iMT9Of/4TPj2R2WV3RV3x5mcT285OoxKfx4sd0norSD+54/lFUjqNCTvB8hBIvotx8fYGvR9F01nhrv9Jbkc6uBOrT242/xYMLWQk0QK/pO5o/jZ483T/Fg5//uE81Gm0/cpo+inLn2zMe6L//qD5e6Tf8fRR/Ny8rfPsSXy7FdIo26Rs6fK/W6ejJCyzSCzHWHR2XuVkuMbYfY/vxhfpIRfd7+NDfoKf7ff7ITwf1x03z8Y5n//jHWVW3NRW3ZobmC03p8WxWRWn+Wr/jr1fJ5RV/ID_RESTRITO_DA_EQUIPE+touq5jkN8KVX0puf1BuRXsiY5tLnxS/zJAVjRpNCMCPiBT4u/m3xv3MUjsfR66T46PtpUk5FNb5aX2KcZ5nm8tWFMAzxiY5ImhSQFxFRT5ZPP/JW0cGqt2MxE+OkehNdiFJi1jJ6HUkxvlL7glFkohTRFcRTlIoLmUb5ZFLKip8QrV1RY2kK0ipEldzIaJrH81SWUZJFYN1EUqHp4ic8zMt2WXC8SFM0iieYJxU1n/TOfnN0TEQJySCi6cByKuYjISzGMpKTSTJO8PFNc4KKwmdY6hlmuc/Ux7yEKRVXeQShI4pmlUrM1z2+ScrkIpXtNYRQj6MrKaq1e/jX02KVpFUUy0vmfaLNt8l0Xl3Vj97feTT65vBRi9D3c6kWopJetfixmCbj3Hv+yz/8ptXZunK/ddlgpD6s4YWT+XdJLGLpiRgt5ZlIPXzhLVn8IRsnYh2THMsiFl4mvEmilsg7UBuyhmVUeYh4z7zlCh/hH9ND6aGAvF38WZamj5XctDyi0f5qzjqkVVPT8/AkoVmLspxPsRTQd6zaizIvuK8VvOfWb66Whwbkq7nu//jFS1FVZbOZjzDpo2SaYCtH+2Y0OS9Yczif4s/Hmj89OfVKEEUiMUcaVHIjUiaVOIkXP1SgleUGVzDxifrk8SuZEuz4WHM1dy9+i4+0nPJWAkXlgEKYXiGyUoEDuZ733QWW6eJHDGHcXmrubVkyuDyBCrO8snTsiopncgJct/g9d6EeUnuu2DiloWLEzUItEdIqhBeLDzcy1Vu+UqY80DKFlmYsUmywuMDu3tB2x6DqS6JCQVQIcknyoilHjrIxiMbKraZEGfE3CbFyJ5YTb5KC/GTWwU7NZTDD4iW3Q9/v7rLeLvLX5fDsnD8nEw9TysqKAK4uD4FQ6aL0h1DudXCDYXn8PgSpTstOd5faCQEkZRZ3bI+6t6/80P/qmppX37GbpfSuAfeoSZD/in7TpGz3m6h+CbMCHla65LquJ/471dv7s3fJ+3Pf9lZKVUPMZlTjnT8FT8cCfE341N9VtQKfW/d31XDUt6h6M8Mj+lt3HkZRJqaAve9V84Ws5kXGq6pW3xoIHT2b8eRySLA+THMRlx0NgSHcJsnlBuyL72SVh1TAJyQt4ojgeUePvoSOHstyRQM0gY1CvN6Y5KCYMprdxOLV2nbeDrU1Ef73ZPYA/7Zb4erRoN+7Fw16g60oJ4rMRFKGqOirRmhP1IBoW/TQ6i2D9JK1NUNWyGB7p/OWB9NRpc98GoN/3u2GV/I2Ti5liSEOh+atquOfB83i3AXtmrsOumGfxwxrA3QZmZ3V66CGnc+roZotPs3mVbkR52WZYHPlOCMBtofn4fQaXNiRtyDDKL8enhbYbK7ewdsNTTOoWK9MRKx6WdAa6WUnc1CqdeeBxjDZyg5VDWRWzgsZiXKcJMMHAkQZJFkMShkO9DBncXiAoUNOTKUla6qLxaryaFzetEdS5TyeEsoFVlQZoojPrd6qHvSuYQmJ4/dgm0Ljppb7aTuJlmkz34b0iXiw4zAh+DTLKy4EyzEuXyew+3zup7sLCobtO5c1y9LIh5gGUx+N922Yg+U6VL/b3cPGYCB4j9nQ5yqhiVKlM0xK0PrEEdbxPJBFkRflEEwiQQWa+nhAWfRq2OGG/iu1dIomIEWmsw647MyfkeSM/XMQUiUKsjkgaYna/ouq8zcfq4IJRvJ2DLVIKtFWtX1zpXLIDYVxkc8yAYuXPkTxfJZCVeEFnkCoVxELC3yzteNkMhmqJkL6rKuqNiosSSWgfiUWFaTd3djp2Zpj4DSIyeEEZF91qG4IAEDWdoJBnvXO6054q42cI0HeoSdDWv+A5T5Ut1rxbmCpwT4Lx3k6n2Zl927fNfE8iAe1HDD4i7Iakh2vVgEYCgImqAsAJzTei1t67zY2zUtSi0ASoH3gDTkDbszmRBl6oqBgrF4C+WV2qhyC03STSclLBpzUbrmQM6hckJAaysradqPiug3gNkZovEBUlIhsVQfYGxLtUOdD/yovFj+AcjzuDJBjDJgApvSOT07vHx+c7ik9DtVSGVQKHYauNx6BX0fZpcRO+Q51ETOq9Sd2bOzHrjsI4kn1fDhscM0yPxqhyaruWr4Zqnrt9xjVkIqd+Y/5S+mfh5ey6qjSwbv33WYNAvGAhkMfGFoC5Ykq99sjJKmhmm4Ovu4ySmJebNVLWJTYmKrjR37Q7571z7tL1UASQM4QXh1MJSAU0lXgRH+jZWtPQ2MVxjn4yNPybf8+FI/9cr7coVJnxB66b5TvBx1/X5DlAFjrGVGMvkHQgIA5wWE1peUZuHsR6IXXLYOLl4vbhT468BZ/IU21BzECo+DSGjEe7wEjyhisV7R2QlkHWuQyikrGWuKqQS6L2r1XQ1UrJOFCrNAcmRYbw5Z00SSpaaZBde1pqy92duZDUNOF+rS0W0Fricb5dJpnDMcaVZznqFRC8+bFcinn+XLTJNHmWVI1apiHJbUKhAe+Wypj3rfaE5eXhbwULDvc4s5zBTW7Gvnwt+Z46+fL7ZcQQq1FEOk44scorRw+y+/Vc1WAbNJGCf1sxWSybA4B21Yo+jHLbmZtTUeQBa7I1sWmCSAYtt8puF60489HpbR5Wff9qt2tKeL2++rzuoR+gn01Hb5S+q5uSoqMHuIf96m4NaVJ+61rNONtv5F6IGnV6S2N9a0scv1evqrfd1frg5Axj8Zos1wQRFPMeuYXOcHoITAOLHHYmfgroe4FhBIMedEUG2ygLTcKcp1ekIkNA/Q1YGZUzqQEX36kH9CTzPFw8dMlmfJ/RWeqB0By5SJaNZk4KWd5xob3npeRjp2XJIwzwZgJAnJWJBCUMwC55S79SIyjWf4a6+yT7tDK16rhxmBoHii3YhCuF6j2WeQXoOkbrIAemFkPjI6wW5q3BwQbdcX8xGW2+KkkJ4ttO69ddq226z3N58Yv2Fprg/kMRNTfFUE5+s6UI4U32B4yl7d0a6POu/GZI6jPFZjhRdUNvR8O302Xy6xU2gY6v19hFRG+bVpDytwWgGKRKMg9tdYOarRkZ9hoTK8sNTZWIcA8wgLAvmJLeI19pUGIMaXOOv6R9vqVxhEJTUNwD+DhsfLdkXu7mMNoi2Ltz8vN+zkzGDRYJS7ldFn86j8dWjLVBbOV7eFpdSUL7ygDHOBgmhqE+UbdnNegrLEoZ45rhCMF/u518Ktf3bx3vUC8XZivRVfnjTVkY+8rbSMuL5ZI00i5PoboTUdJ/V39AZ1Bvqvu9CNeUb9M2G+ojbXAV+Y+OcLIYzpjJzIF4ZLZDQmnvLieULxpGVRRy+iCMbY2sckM1J2dB/pD93yZZOqhN4lGPZsXYiws2YhIsfkaktFOI/TMgChYQeTBanIN0KXyM00FKSPrYlINcV1r2lHhoeOPUm4UJsD1HilFjNFYjlc5k/bQ3Pq6htbX1VYurfl0Koo3K1ox7hlQ8nyawxiX4zkWdW1LWNyWY6ciqhXlBi37XixVnJm0rOuVwP8bPtmGVUKyVUSqIIlORbbceJyPhxSBDjP52gamO/4BuY4WfyABy84jlCs9dsd4ksIA7NVMtc03yav8Jk8rQZLY8rH/DSA/O9Y9SG0Z68JEOtA4SXqlNcVYzBYfKmyG9wwLgXVVzmj0w8RF0oF6ZLO0KNv9BZ6V/f7J8beHz3q9rR0q//zB8Teh99SzwXaMAeaF9FqDfyCKqXfCPY6KQrwJPArLaxlL4zt+cTD6O4/cb25fovRuIEEXP9zIpCRfvNZPAYX0HW1Z1uqSli7nUMPi96QiVPPjxQ/pGLNHv8WlyJK3vMgzjkUAIaEYZASG+mdAR6f/p8CnuTfoDXY4KETOUcwrJq1MGlpc5NYRTzTAy57oOe/RGFX0BFOFik1Ksqu8UhWnVrdCTR9MG7SJl4WYXXWwmoH/cPGDB33hKX2h4E5OkRM74v2TF551IGJJsASUa+H9t5OnT6gH69xrTAmDmWHVMAmMK0btipqfgU8IZUw9dgTSEipHYKMhrC0tMJH74gPTO78ng5HHmbo9jUpPsxCvDpmWityZReQNKM6Qe2OtjNij/SmmIpNjlMxceqn9yXr9YI6Ww3f+KXfn7/YC/+/mIoYFu9tTmp+EHAbTqZIqlcEVuBdUzm6s4DUldpTDJ3kmg8sCkF85VAH5ZR3ooD/X2PmhaZigMpVW0QjT8x4N5IwKnn817NuaFxKqQjLywdaGGAhJZfu6UrKBH/PeT/x31MZ7713d3nvvl9/8i/eOJ/DeXzUFHj2Pu+kJIkFAOqru+kwNaPe86VDhMAmpl0BrtTpYQm2E1Fd32QfDAE2mKdVAkZA+l8vFTFHIa2weAzoUDKNqHN5S7KTjhxuvd6vxccH/PBb8bz77+xWPL/Kqyqf+isGYP6qXEBZp51UGA2D3NRCEv7ntL7tS7DrZMVlOXDMN+jOrS0WoPRVVWM4EjGoxAZIaHledfndvbSFNEVzqr+mCg73UBJmd/bC3tu4kTwFaqBRvXVRdpGSJZ6zx1ta6EmWkfAND24CXlOyCIwaBHIs93w+/y4EXbAkCb4XWpQ1L5wEJzN01S75mhtdSziKqHtFYh0yPf0OcU5Pg3T5ZVvVYbfMaClWW5YlLNM9D82rU5ADWIfGEiOOolr2qtP8QNWnyfX/lLrZGSQGeOx+Z1Az6NboAaLg2O4/hNJiUOA81i3lW7hbhRZ7G3OheEZLWCcvkraKWQddOjhp1AyxffPHFKbG3d5Hm4+vSI9TqaUduPi/TN3sWB1+p2UGkprSrvGfcnsoxbK0n7bzt9aZz2wzq3BKBUJFdXdx/QhouyXjusWOdNiO1t8FFnqddW+skmfqqPSVSqRW/PRRIxtvdsD+4fO8Dg81S8BLkBmB64HeX2ucgR1c1R3GEW7V2TBR+vwWE2PcOXKDUubTaSFvfbMslpaZlUiX+EUXckgmUoKnTbKMF4PzgzN8nlUoZbWpx1PPz4MxM88x/Qj7Y2eIvFylnQ/ht4ESlsVRksQge5QdsqEJPKE34CbtMgUOopnuDfj/wHpyMoj6q1X084mFSyg7lARYX8/E1SN7bx8PJ4qciS4CvDjFwUvjPM0Id3Ot+mkzZT2z9w4G/X4pG0w/mZd4qooMY3AS/roMZsSqrsRkNZkVcw23+KRQ/m2oKLm54Tw8PjgINmc2qMeA7kCQvhMKkgCiHmYRCEIywD5+PGq3uk5ha/CVW+NugDFpOQDSGJmTFAvdMJOF6TiaiaJDJ88kqDt802iRzi/pNssUH0I67JlStVDuIPnr9Dfw36PXv8QpxvQTGQCJSBj3KsmcUVmfNoUbfVHJ63Q4HQX8Qbp8HJDsCQmBq5Lprxp/OoBQUTTh/ppJAYTlBrsWHPOY1A2ib8yfy7zB8Y8jZwq7MC/s5EIzMuAPpfQfQlvL0RYbNTUpJxP9tAe0h39KkX1DOCv4l6iIybLDAI9hS6Bh1blRYlQ328zM/NS/Oocp//qP3hFfsUZ5drilv3+gKh4GnXaWE33MCpY1NG6Wm5z4g5FRtCFkXWAUYFxQDLPDy635A/jTFFceiLNnemoMu+9uolmSNRo+I3G8EpVNRYDYh/IxVJpL1g/v3NnfIo2KKcJOHqbzRMkTJpuniQ0a8h2HAKGe2ogH+/MfG7t8P+8FWuBlsh5vnGr4eSI/oa2uDNszr9XZ7PR1kBID/kd/dW3oX0GzQ0dQzEezc7LVSOfW2jywib0hJj0J5lcyYvjTXTHWil3WzEkno+pSeLC8TYg6i7n22eDjPmNYWBioG3MG6dt2FPYNqTCC3SLIXChjoz7WD4wbf1geJu8pxGAWsfFW8m6AMtdXpOl31e+FWMMD//fB+sBn2zfI+WfwFAksJHC02WLolRU5G8cwZPntXL4ELaBUKeTnHvqqsyXzuiXlZpwJCp80zAZtNpfKBaiSltdOyGj8ijD0TfGZ+EzTHokWnrpQitl3l3HRZ2HRHNEDGtmORWXtLsfGcddgxdYSZQrjg0wU9WvwvMNC0frK8W7XS5kwb1GFfjH4WsbeQhQv7uvAywNY2UwYaTyhH4HN3cTu8F+wEO0Y4PtV2siOYUYsyOSntEYPY3AOnllLZwd4VyBPKkbw0mTb42YglR4Cl9TUiss02RwX513W+n/QuZc7ObOGwR1Iapw77JdB231eigVWawzFoYF4lFnY0Zenx05EJlAQ61jzLRZTwAAgmRXnU3/p6+x70nHWoe+bJy43pz3+CAuS4ohqFJgUIkJvLsCHnvq1nQc6UjLNNeQClTrqUlJKIrc5yxTBghu/AKAwXrxY/ID_RESTRITO_DA_EQUIPE+CecD+eOv03BraQ4CEgydHWKqDh0d6QUjAPFEK0Fn0jD0VYJ3v5ljm0BsprgO8JQ2JeTBXFfPFB+vLQQ0IZ4UM5O0Mmk0wpYCHG6NY/EN2yVm1LNTZbX9yPFLb/Bb8n9AZIVnLfiyXysXcoxkDITk+q3FD3yrMi/EFijbVGrDbiQWNbI2kSX+3lLkLftHazX+8+JF8QN7JjDJoCjqwo6hO47XZDZa/HpNotr2veN9ILYh5SnDluljI3GKPWANBUnt7dgbMCyY4BTb86l6w7YEnOYcbL+JEIZi+eiq9rzZ7S+91Pg3J0JFOtDFOOts9tN8cGvUqmS1+wsJJJTqFRowBZ2SwI9rAdK8Oe4i0QaHN+Y9qZtHkj171aKfLU8CO3xJHcfeLP5RWluuAHHAydoGCMEYvNHT+Vrgd9DfDgRFqJ/ML48f0+mEf4qMfgpEbrlKGkmqLn/U2f/nN7571drCSz/pb9Ll/n1YtsW5gHoU+lMVVqZ7KIGHKczhVTcBlF8HOp6SgirTcxHm8NEYyKlts4D3Wsd2cMLqODEt2TuvIVVDnUCrAabyiTkyY/ZPGg/1nydrqhXYfEx0eHdSQ0woy51FmA6oNGXoKSiYv2LwQSm4BfZGA3CYE9sAP+EwW4bYXqDpWTaHcjZLTXBQYbzqjozHTjVKjykLaDG635D2K5Ym5r/9x93oH4GPACG8r3DEQ5BSdDzv4+8Ev//TPm4Pu4vvtja/3VJNMcBvlsP4yu1p83wu2tu71tkKvHgM4H+LXwWaQtZSkbyPC9dLUYfAGeKgjhJpbWsF60gDsYs4oUnKjjc/PMYjJEFVNNs3K12U0Kcuov+cwJumu7cHXIPgqgcp+LQWHLzk6k2dNPn0gLsjUNGTG2x64T59n11n+OtszE7+BNM5D3j3YCYoYylrUkq0jp4v/wzEHo1qME0RoLVdKfSxjXrMC8ShkWFuKOipO1hrONIxmM0m0VrrLokZ3SnMn28tR2zankMRxTc8BCMXSLKFzTRG7XilI6isBShsea4BqxLoijgrDEIySSDVqBO8KOZYyjqQPVzgirM+F/ZrMkU1xABSN5+pEgzaclcdDy8hJnhgIFHNgxpLM0ZPHh6e0wAPvegrdPmUpVL9H8WMsy+LDRUuLHbCuNdkREDSA6CzSmuunoQRHztRpUArD0GEpHa+BItGnowBbijzLwdVTBfOhVrGhAVncAK30CHoreZsrinQwoYTCUWE8rCnh26Roq4FtmH4w/DW2XQKdMrsZOgFMleu7u3Ve8+8BvzWLwh4OJk/SIxlNs8SojVmfzGnX6QSOIFzZFLKPefP5AxmtOX/64VZ9qu191zCgEB85LgMOg3DKRqfgNO6MLPfWU5VAteKFypWiF0VIqXMNuwBLsNoquAerYACZav53zLutcKvX53k6ChtwmJnAUEBOn/SxN4AKi4o0FFMEwjiyGTaro1xS5XGxY6DSgq6OpxITqShaQ+A+1orR45go2PAz/YuPrUL1HyS3dCTpArgc2h0g7GR8hRFhkUjmzZU47d8b7GzuMRwqSMZIBZDCtlvOSn7uvT5f5A22f/7jnkGyXv9+7+c/slOaAGEND2jSEHA0deIzOsdksqIhoygKTcXxcp62GBVGaMbanjWVtAyyJIINotI7RKdPphdmPpBeOg2XTT5yKLZgnT4C6feCQc+erdtj+Z9cJMqdWnq9ALJc4p+tHvGNOqqWl5qzGtYBydbaPDM4aywv2FpK5a0iBe05aY7GHLnEeDj+3/sSeBwol9ar9AZfYgjbX7IOgGVqjhI6xzNz40xEV6mo3TOLf/WwekzXGJpRWoxD6WRkaUyn5mhe0I5ZLzd0KB0vhQTLyH2gj50OB9sYFB0sHe4E97daC6eCzCABBi6mLVWxbytutrHvPWDfVch3wMh3EG4pKOtQo4b9TgodBUsUlzX2SrPvnD3eNw5yVdZbE79ueoev5sBE+roB2cw+WJtVsQRq7bnLOs+LI+6s/zQJmTyHljfgsJwBD7jBiFz5ypvcr3c9q0MGD9/Ms1gk3sPk5O7jzX7v9GjPG2w5buc+6N24nvEqvH+/R/X5kEnoMdhwIMguQfb87knSJBHrrNg/3MfWp5IpkRhKeWxVCtY5pDhnFFHOD6V4++df+ehDjjONBeLc4oxdz/+K/H2qBsH08RUXQ60uHRpsigkn1Z022zZpgwlaUJjxYwVprObp43laJXfHNDHv2ORK3CU8B2OX0yUs/lEawg5Zx4AoqZJ8kAWblTbNc6yux3ChM3l5LQxtGe6aLtwNHH3zjXf67Ojp3cG9cOfu6aO7T5+fHtw96X+jUbA6hsubOlfqt/zI3sTSOSvc2pv6/LG7OyNO0VGQ+cTbAnW8cHZYCzbSdkXObnfjKyO/Z13VrlJzbKf5TC8jC+hYeuZUceCbk8t0bFc9xZ5l86k5eGzUKQP3khyydgJQbPHK2Io2hSicSTNrsJ7R2DXotBvXHvbh6rPNvsNKztnqxffejmcHPTXHpHNv6PIen+PmEWu5DFlGWb0MKDEvZv6YMe28hMBbLywbZjRhbqW7JCsDLT5Ylx7se5JPwH9+8KZu0NocrZjh/a83A//65UxFMnOFhskBLJRwWXwPKeS9pIL4boo6uo+qnJzuM7maGr2vgx1Vpx/ubN2vO3DOw8c5o4jBFpX3rlH63qBHJRvNP9GH+SmtiZPoDI5BtcE99EIVd3YoGqCqEgOZZTvY37A9CQ8z3UAXvgtU9bNguxv4/cYO9XvAnhRe+Nrgz6dkJCTKHamyQuOlqHJNjIbFCJXw/LB3esBKhVtfV2lmRpqcDSDv1xOUPn5GSe8k1I5J8WKoqDtb4cTWp3/o+FSAaXVNSi87xnRSaOO8VON8dtCBOaiD813HBjk1iWeu9kRXCkWULmpZ0iIUhzOV6O1KMq1tDoyVRtINnAuB+KDKNQXjYmakwtfxITsxPW1s+U44CHZgc22HW3WzatN0SMtbnQEbEuUONbnueXQv0OJ7ug5oeD/YAV3ueC/A3VE53OkRf5PwotP9jtqj2Sn3pOF5qzUyFY7yJjAyEgVqzB1FnKOZOK47bLRn7h2CzUq3UgSevkqIPuTqw7Oo5L/46iASBnxrUDP0a9IEdd4rq0Ltj27tmWuiqO1ZtSnnnyInykb6GEkFHOrpG6eom/qocxprrIA50Z0yikMgr8rSrOmf1fDNuu95B4It8xdQjpRDU2oHAt3BQZPbOBk91qhRBd0Z0xrUCG2nlXqdLyqNN2NdnOgzOMIo6//PLHH05MWn+SFRNvXncPlHWeaozqFoUzrRbdvx2Jg20/HyfSmBJ537P+zFLNJI1NF+fcrHyXitL/4ovRMoPNTd7/3ym9/tb2qdS1wNvAyTlS6+0rtoB0xgTtT3hbCWciJdfw3XuPv6H2Kb1ub8R3nmhRgPCdNBGAkglIt51dwjh+RCb4VP1Qijkm+/W/xhapO991z4V592quGyzf5l5KNQc1Ks4BX3FpLGGjB+dU912HMmjQy21rpwWhplGF4Ph5+47mrpmLR7N8gZ38uk2ei62+Av54Yyw2d98JdDIMcOyxt3Q7nyOFqD01dyOScJn30dbgNdbAeAgm0OfFhDg9g5EeD4sLRAdOM9JXu1tGMAUJEF4pTsJJWaQAFRAtGut0z7BsgMLzlpnEKCAqRxm0ytZDRai5WWLCnSr5w0T0YnIwXMNB6yRj0gGzERUQpsLFXQDV+YZWslLVmzXAJZK4DbOErghobZ4K4Ab0mW3zgez+cfjb5Ld/46IqhTYprvGp5EkSZsWEzZXUVYbVoSiCC7vqHIm3bHca5dhlN2uYrG3UWnyvCi9J6i0PG1WlY5mrp03GS130ytU1YKcjEdQSi+MGcS2mMwTfLlXAzXk1hbb8p9YEwawjGX80S7tjUlGWG9tlm+ksvZO5IOVuyTCC55PW3QSIt8w0TtQAu79+OG04VTjwqhzjWQC4jG5djPKlvBrOIB8JgxaumzfTFyXoz2G6s92l+/aDyhF4vvjzCMA/fYZgz5dyNUbp4+FWLOhrLXwcR72BkKbHGZWZvQPaOhFkSQqmpYOBT9UWdYmJuT6Yrovk6jZCcv31eTEo+MuSLvoZmHjQLtef/+L9WwFwy2vSvOHqjz5doxjGbyouvQ22SH3ma4E3rPzR7EfERFpZuZ86zlrjcKvBeY/fVLWDb1YJQHruKAGSdcvHQzT9fm6TjAxOReLdGCIRvBEmLxP/lkTStYeXRg3TWU4NvQ3+TDSlTqGiknx81CR1u7gXrIVwucn+urrMwdFfUBDXXwtT4rWh99badAbQf3zBIDX2IyFWnl2uPkjb75xuTarHYsqYgFeenJzO+TmDYJZtppSXqRDiOByi7o3/o0kXLGsBgbQwokl3WmiPcIvDP/jjKepovfZxQmqa3bhLV/LJqO1y1vRMf0NQFjUKkJZaY6mw1f+CC/8U7X/g8xj5NqqA8QAibQV98NibHsr13UJq21bLsGaNuPLEj+hH9kORu0mVuqEkJVqImdC4/VeVzlKFRJozzUM32OnDKSipiyCupaK7vjsF3eyD7VDcnbmRzTZTexeJMml1dV1N+ermnwWEXDdFtN/aibEzciSSme9/ntCcCP0u6ZgYy2xZlICnd4fMQmusiBxNdNvL17OqNLNJwx/V7vV1tbvf7G1tagH2x1A/9LXvMVq1Yqv5wfkMvI9NdMER0AXd0jH47x31Ds/AL8qAPFJvvVDkrDKwrts6Dm+2+S2IAgEtg6m03nyOXzG2kyghIbHtSJQQRLVMKuUYpY2cWHeA6AlVZN6+PZmqOT9qoBlmXGMUx6CIZRlqyKJ2ryZKVsKBJScy4jaIdY0sUI6kBtpN5ylsv9vTolXeeF6Voe19pz5VE7EUPyVTSQF9qxu+wdXNLm6mlTP0Ejt0R56C3+B6t7QiUZO+xrR72TGu3em0N2zeKnWcELOXU8qMKkABRtF75C0HUsQgejWGcLyiskpz5fYm1SaeoAocGF0IZpkl0JzSuOOUCRteLGHAlm6VhQFim3TJNek7c3IscOBjOdLT5w1K6UM5Mx8kBU2GL/q5uOFZcol9AGpepemGiSVBXfHZW+iYDnk5SDMJ6gVmmPy1xFGLFGtLNp6D0GkCShOs+uNLgURpb/uZH1DLWhbrGUDShKG9XwhLZy8ly1R8Di3prkCAyWj5DbqekH6gIYzhzyz/f4Lp7hO59u35zp7M3H3xziG/mD/ccj+/HZ4xP7OUMRik+MZWau2s7ofetZoa6a9p/xBdn+e5dRy3mqTuLELbhWm8wNU/DjmqjpNtAzPbsGzOAJ4pN2IxC64Efg9nswGndgNg7aZuNhUeRDDRTzX/7pnxXBhx5Nm2IxNFUy9AhqKjhYSxhmUsorUe4t4r5nP/9p2EczJyeHGycnp0Gd9mqI0ZwikJ71A7xZPo7/JpFprG7YsPfa1UtqTgJRMyWZX0R/zNZSXeQbawiDta5q0+9zAyCm+XqiiUXKDU3v6CFLepN5mkb6xh6KuB1Go30tN91X0fXLK/S62aUQxNUqgO50n9nEHZux43T9JuQ+VHEwMNr7aLuj2jL4vJZZaK9tVmUiG98dTONPtfkwQilqLpoOTIuUpd5o9deT2jDS7eStdn49acwZ/89sc+2I0/GzTzZ3/KxuTqGVXwFccItfttsyDLO+KV4zt4E20NhisLFZn0NR4ckLUG8BfaK2ytIdETetLfqj/LrQe3hyvHF88pBEilrsIS/rRqcPu4lWs6sFLYUY5y09BOYjAX7VyoRo206H5Tif6XBV7WXBrMnd3kpeoGw7SnRIx+SEa2XfUp5Bi3XIVrI8Q8ChKCVNgFhGXX5pGEX56VgVXeVF26fwjLAzex/lJee9Nob78X7G5B6miyCX+tgzYk3lJ6xtgE3SSxCM00BRy3vznj1tjWHrBH1SQQd0PqdUp00C77C4KH/5ze/MQ+kdy0K+rZ/smeTE1o3jlNPUyo6V4yvtdLPxBj5zohGl8i/lTUzQmuBU3Jp7ArEjY6BpurQWM0xiurTt5XSg5qxPmbTzBBwwLBpZ5WuOnbhRTcpr0cmUhIi0A749bcHJaRuck0aIc3ylLuiYA+el7NmX6pAgSn6NMs2OvwHwyBb/tvJO8kN7XbnCUJxH0+t5mOiu5U/O8iIjOlC5R2iFv9SKnePv6n5vhnPmOu8NOnFEH1p5WLWhSx3qCZlEOPJZ6UealQPtzvMAYNnBBAvi5OG3+7VFEud14rJWxlMvSedQk/r0COXUxnJKvx1j7wljUBsnEz5p8bFE1drJgy62wm2bnlUfqNLJEyrFll24TmaWwemc71CrvIap8xkxNHO5vJFMn/Cm1xjqIw5+asv17K8ITPAFU+efHUGz3Tb8+EoBHBc5sJzUzsmErJwfOc009EZ0Mby+W57R1f/9Lf31jyy8h72AsjuFJcDWHfRWCyyVULLfGzXibO49+/rMcDNZNrOL3fTjjMyNNN4B5yi2XW7Yp/oqHNeJs+rinH2icGUdJfaIPtMLXcZHhKKPfqjsdn7kXH6r4u7j1jEm5bG0djn5u6RaHa1m2Zuqc3I5a84Ml9xj9sYd7SiDlZ5Y55gkCwpkrUJLgZu3H9RHO2/oILWyEiew5/VxL3UyOOdP7G4NjGimgm/V4dSG8zerj6qyl9Kq1Y/dRfQk17XjphO7Eansbweec5iSIqt4F/W368sRE36w9fXO9ia7AGk1VpazxdQe0eE0iH9jNKt0F+0RpBNxuWevtfVUvb06MY+utXXOsOp8Vp3Dp65U8uKa7FRyRUHGgASa4kGateFfRBDTJH3TKZwYIt8HW4StK5Ep6ko3L7s3otDE2tfWXJBlqUqae4u7e/q+C3p3NjgP5zPwUIe0pf/VO58XiX6uBHKJk0kvpf45pPGcYw/0Iw/++zOuvXl+ztX4S7++xU738M5gcEgdQonU7NNR45iDNCey6Mcq+PyT/xLGtqeu7Nzl/N/cPKxv2dytTzmRR4PiGmrY+AgFFPrvWUDSutk7doPGt1Vi3AQ+7HZKKGywhXHisA18wE4LMEfKsQojuylsxOyujkQvu+EzGo49B9tloQzzUl3/6r4xd5RsoCdLEBi8c5Vvq67zplHbkE3jtIK9W3PlkYUBlKj5v9+nfAqlTG1oRGqaX6J057BzoMQQ3U/vXb+Gmt84MFebqQPPLNzMjSZxI8mXRZoWIp++5tJGA9h1rv2I5OIplTNI3rCLqiTO4h0rV9gULT3h3Fwm151u/Jg8I/OSLmxb/SMOJnNUX6VmDSmjOIJPXJI2JeriK+6c60G4kDl5Hzj+yWBFHmpgTz4Gxo3oyfrgUuA1RFZLQXpyeXyiHTS3t8diM1EMxJCvuGDN3B9E1wi4x6T5tsDP+bWPwApqncF68nB0d7C9Y7a4+TMg/4lfJqll6ooNb/4CiIMf1hGIHiXnmLd+TmQ9VY1adxKSia9+ksUQFCsW/XNF5QarQDrvnZmLaiqprxiglSprbKyyklH7IrlIEzpENknEhiFe3kFN9CWdA6kKq+dOjkcbT56ORkH7LAqbhnQgXpmGwYo0jaCZMkgZZnV2uAm5k4KeUmrQWxUXMwEP8oSzZFDgw5w4XvxrY17kJle6veR0ilrRltWQb0Q9UJx5qjgT+KCITiRd5Ek/Kunv8T6U4kbW12rSXlB9fUWpyJIJtfbON+zLvzVZRip/kjQY3dpiAKARvf4uGCpw7rCVYoKK5Fn0d/mm8savqtQZU/oEA+dsKM+dU0m97Do/Kba6HF67TdK19hib1T9Lo+OfEdL3/+NdXVX/hmhkfkPU390KfG2VR+o2QH+Xbxk0dxieBz6xVMQ3sduX+trDWguZO+TfSPqlK7oiM/C1BcY30Nvn6I8Ovs3ph8AupYoo2EbMpvm7/EMf2KdQn0iCMM5ZBnTJK0cBoEhlGsyAVPhnys70wQ1zYIPocl4ymdanNdy+TFamcwVBaW/SWnMeoL4dyQ/qX8txbgMpza0hU3NtCJ+d5ltAWhKTFsMZT538VmclNgC1uoNAOUT0laKVPQ9RH4UwZog+5+b0UP+gw6exu56pq+b3lpE3yaU21PbPVcBi6Q7tmP4j7v3ETxppDv2cnzUqCKL91XX3VD0iL9jT8/LK+UUmDYPp3Z07gGzmJ7mGQz+K6F7kKPJ31f3Id/4fUEsDBBQAAAAIAKS8UF1GaQfhfRAAAEREAAAjAAAAcmVmZXJlbmNlcy9yZWZlcmVuY2lhc19wcm9qZXRvLmpzb27VXM1y28h23t+n6PJqpooACIqkJHORoknZ0lz9MKRnPFOpFKsJNKm2ADSmG6Akp1J1V3mAJA+QuVncmlR5k1vZZBm+yTxJzukGSYgAREojyb4L2yB+u8/5+pzvfN3tf/gDIf8Efwh5xf1Xr8mrYd19VTMnEp4EDM8Nfuh3/57E6STgHokvRSLmIkgo/mByKmRII48RnyaU/PanfycXR/2T5TtomlwKqfAteJo4RL9sefmWUQnXojQIsjOKeQkXkX6i3iGu3eyQpt1cPiCFaVKf+kKRkPkc/4WPCy8NWZTQxV8W/ymgLYRHKqGB+d0h0HIWLf5CSa9H3vwEb6wv35jKAF94mSSxeu042AtbxCxi3BZy5qh0EnKlsElOs9U+WD7mC46PuXW70To8aDruQbNdb6xsFwiPBmNPxLd4F77UkfTaiec+/XnccOv740a90RwLyWc8olzZn3i8spnnMaWYdgfc1bbcuuW2l1c9nlC00KZJbfIIN9lE2b5tv4JX/3OtCIVGBRRMC0kiRKAIjXyCBsO2rNygW1jEgHncWd7/EBgUELDhcQatgkaJGtFdTTQsYinmLOKL/4o8Tis8PuPJZTqxPRE65zQ5pZOh8K44U5mzTF/vuj3X0LuO7kGTU7A5iYUk3w9Pnf7FSYcM+m9JhI3kkReki8++eLirNyy39PZuntji5r2Cm0cioJIMhOL4OOkGM0BqchmibYm5OKQ+p+ZqHAPi9LHSyDofHp067wdWq1239pp7axjlkDBkPq0R6Ac2uhv5klFVI117AxKNev2gBBKu7XbInu1uggJbbPCgdCMZ8Ra/BF4aAEAEWfw5muHhJ4AExIYOkWzOlUaPgOOAJou/Si6qIoPwlB0F0p6Juf7hTG/rB0Il3NGdtGN/uiNMQgZ28JUzGnTHaIhxZoAxdjf/nt3xUWVPmzyxL23tk0osNQtYegexVNJgsx09in4xXyrBx/lFt0veBWICT54J8Bc0OpoRGJ5CwuCWtw+IHQWYdCEu3PAwCx2popAvlMDhAiGDSgpggDv89BOXRBAPzLX4FYCBQcbnKobmTHjAfap/pzKqDC1hYEeCUg2ZmaS+A7AEIHiOhif7OVI2BIcvHly2GtsmO3hxS5RpFZAxY8mY3SSSjuUKgIi5EcQwj0lihkmHxPOAT0jddttrV+agYi57Ikokn6QJnt0MIo12eRDZBMaJxJZAnIB0QXTTEiYlU4lkHfwCUArECPhdsimTJq8Qd6/drrnkgxP+339jKFFimlyDeytQodtrxbfQgciGceonl0wHFy4cFjlz01FHfwEN4cy07RPmm0dtrluJ2dwuMaF9mYTBF4dU0Sk2eYy/be29SlC1C6A6Ez4LEL0+vQ347DIhdE55QHHEJrc6PK7tBw0MYTjDwFdkKkUIw1kCQvRdMzMe1jeX8Rnw0KcaGZqwewKk0btkUY0MzIkRo4FaXT6Di/D76rZGvsuuJwxwkuAdG5B1Dw/rJZBt2I3SvKc7reMTWDeCCLVMgz6fQnjrQKxH3GZ0M6jMcIbzAql1627bqdf3Dqz6YePHbw7r3x7W662WdVzkv1tvfTGK9GTOsMnzocjWvq0E9H4B0F3oPRQg0BaIE0SkiS/AcsDzICTosyzUx6mEq9O7xD+EJBawsvz6lkL+A+P0d2RcDcO4GhXIwwQZck8A4+KANBlLlmPmiqUKjMSm3OOYZNUuAKw3nJjH9oG7V4a54tWXgtnacjZ5At9s4VMHBTychLqqQbaiETanSEV0MIVPUHwvCzS5u/M5Cgi8zVeCJaDoM8jvCZRQH8yQ+GPAOPR0BB01J94w70p3/oNdytXL0uyevYfIKRRwb6hiZLr4rIBvYsGOxAsygOSflsDB8CWQt/9vBNDqEB7GAcvVfL2jHuE+/OYALM3hGL7Ep5rJ7RjkPtrAxTDB2tD+ll1v23W3XhHl7r33pfD3QCfZ5KkAo5FanYoPC0g1edwQndfwIXMEjvau6IzpL4bLOJvVa2Db2S1RtwpGTlnYOhZBOJPMdO6t6e8xZDw807OXJjnjV2KqrniNnJXi1C2tKX/7078VQDrM0bwcs+uQOZO6coTrCZ8bxGWVxF2iugG/j0IpG4KF+KhRGFMwsEIwNly33jKXIR4cNIsQrLrjpYD3KNvb5PeiwNYOq4LdevytYIcFlg4mauwxTzdKYlxhczZWNNSU85lKiizWtTZhdFISuQwdY8AWhMLgt/gzRj84HnXPCORPxeRcq5xwy3eji/PnqibiuTG0fdduDykjwFBTPnN0RhP64Y9KRLpiwvM8wtEi5OrK05QTD/Xz/aWEW6Z7f6AJcLofoPUYLVvkjEYpsLm8MNLuNupWu9FuluGoLyZCyzBAQotRqPkABMGwXoacXg8iVa9LDO8EA2SY6ukhn6NZK1b2AEXLbWpFS3foEYpWZrLxvDXWfR9jNx+naOVth3Lng3yhg0az0tclwnZOIh8YJgJf0tHpneS+BfaPwE3Mh5aQkYlM62/3BlYLPr23v9c6KIHBGZXQ5hp5Y0KkgCFaElHqrRI8NJF3N7Hsa9p7m6AYoEgF6UdLUWCrwRA1byZ9qkiEvQmATMGPhyiaLaNoYk8e4/+1GcdrM46NAcaaNT0KDKUWBFQ8mdc0uWlVAqYokb8fnlxYjbpddxr7dtt6f2p9P9Kv1k6B6s/gswQN3TdvdhMvS8LAKQ851E967GNsTfXo/x8442NuN3G2c+fimqQgy06hWqGGCmD0AGIIAKnAB4ROSeWtzWw6MRMkZnLJOfT96ZQ22y1K3WZ7Aod7E79+0GrCfZ534Dor41hr44wzw4yNYR4BLjDcGN8MOBo39sfwyuKrdocUvM0mO3rxfmnTLYreXZ2reJSKVMMxRKqNNaIWKVCWmaZQ/qir2xz9hj9ALv0sk6Evc3pOxYxaH/Id9Le2JP/fMUPI4GemdKxkkSKFKUtAbiYxYcApwO/9XXXJAA6V0jhApRinT2iHHMmJsrKGkazLy9+PqMoa0Bq3UT8s1QG23ftiVdnufgBa/Czo0PymOueV6O+XfIx61TgWdOybDuh56eelyEXVfYMds/WUC+SxNMQjDw71rP6SBQ0uur/9y7++Oz6pQNSTquxVhvqKtfadfbuFFhcV9mEZ+EiAmSk3cbgFPrnxsutsTXMVl4qLAViIkzOSYn0JBXJufo5hUZlmSVJbWstJVHXIlHmXunIX2YVfM21kGd18FrPIv0dM2hFkMxoETN46a0hZd4znQPhMxhLPwSk2ztny60BYzlsQzx7s/y0QK9G8c3HPZxIYbiaxZsq6Ibc6FC6DJQCLf4IhcEdsN3I8Ppjp8AkPkA3eO6mz5prFkq2MopfNxyx+TVC5pPA1GYFV5oK8e39i9U+GJ+fvawRtC5f1XHO+iOsYT7Eb5qX6jogp8GcW8x6RO90WnLNh2O+QOwv3vhS61gbH1PhSntclYjXjLwrvDGjNMpQ+Z4Z0Kyb5+kzPMy3JFyCDh6g3QoiDNIgyUf8c/iH945MamVDF8CKcxZUt2SNhvPgMubSK8D9lzsxb6+sIYmVp8l6fbglbRcH76CYGYCJizTKJIyNkXsTQlySNIFAy9Zq8RWQOhZgmIsalW+RNygP91AlkmpmsWjz3vT2ySR+nOBKt5QNnNB/YrZosofM9nBW+SdCqiZYS9PonBNSbk8EPWTCCkQg1NLuTGgFYUwrJE/695LEArpZUo+r6+to2kq4WGryQcSfTdh3oC/cCphy2NJ2lNWDLPGCJvOksmRnNmmQWs3jRYl8MX5X+sckTAuP+UrRRlMOzb+WlkqGenTQ6yQegOeJa1UhfAM5qZir8ykx6l83APD0K+2zKI76aLB4dv0NxXLMLXHqLp3GZ3uc5CzpEmOuQGnFB7y8I1yV385jMZgKrMuUGEs2hopBes2MrN91lmTlcZV0bC1k4t6sssI+lNu3zVWLud/h9C8hKpPKLD0dDcizA4LekOyhb/33eHXWJvm83pBxzpeORB+537fJFwG/1shZMi1kBKTE0LUtEXKsFAJFZIEuBAmg9FEgEqqReKhUcSDA/j4EhzN3KWbtYXANFgpHr2RFVdC2Y4iQNB2c5NOYOzvYLSQPnUtvBeaC8hQYaawONjSHHYMg7qfMBK/pWxrbJpm+2OLeojfeOhkcjMvrp3PXZjJjOIeWoacKBwAHKka3eEoFYh4Jn9r6Rv1drfUOCSyzZDggQT4iAXLeXQWQapDcACXUbocF+Bwwykz8BBh7swi0gKerhZyxhQuKzEIED0sdtBhdzHB7sWjOrs6PhsGuVrQV/DmzkmkMfAo2nDA55aISr9uwcF1a7RvTivbF+9+NmT/NI2NVPWwBQlLxhNEJxkuh6BFfMpkkq9aIJj0LVkaSBeE2GzNh1uTpf13pUT4JjIbwS96brdSi0BDBHP6c8BqYweHc0rNebbeKQ798ONmdVKsqrQUAj9tHoPgUqku2aEGYVG0Dh5xQ1JCAojFAFvygQZaKY0ZM6K+6RXw6sJ18Xn6d6XRMjaWi6WVm+Q5HLADRiFjA9yTLlAXN8x523Lg9OPuz/ePrO3z9lPx5fsPm1+OPpaH56qn46vDpx0GW7YgnxiD0XY1+Me5Ct0CFjnT/HfWwBioNt1ATs5CZ5OMQqXGKT54LF/cVZo6h2G+at8f5eb5fxWSg04t9SGWbEvIuLnkoQN/iBdCMa3EK1osgHNuERQGyn+ZR7NrGtyq+7BIYhzDL5kpnFhHlxCBcZieU2hVCAbaCKxXUjNEhE1VRvPLcSLHk5DZSd7XiCol5vQlt1bJx1zOxOsDD+WHpnkYWm+jqK+DJHLPe47OLb+ydJGkW1e7mhyuy3Qpsolmzd6PYEG9yOVoOlsL1xprdi+HpKBP+mEyh9Kn2f2+CWNcu503xnEoiJE1IemS1vduh/cTevtrntYP4tmaqoLkOJnQDXoRFRMNIk0+NHkm+GZ6OjbwmuR8NrdIITB8nq6ln36Nu/AxDN9JfB6DOKe0wJnQuui3R8HPqsRceAJ9m63xJw9C4pr5H3ZlqwL+lNwKRemF4yR1u6SKgklqDijMqeuhNMmDKBgqJ4Y8BTwxtMSlsKiTdxsPiM1tezJbhBACC3Wj9ibkIhWu+n/MiSqkw2C31AGcSriHup0pr0St/Zd9xGc9/BDm3wH9SiW+5hEx+39i28zcLbXlqFvscrNnlRyNy/eKlRVKZHguvlk6cC16OXIG6E6hMl57qz4N7Vzi4OCMENxIOz3m6xqlFchXTGPBpxFZpNuAmL0gwz6CKVflxmspKNAZc8Xvw1YapyR0A8j0PPVrr5mlgvl4pas5T7zHGta0ZxMRCkKMVnkcWjOE2g9rqkmZioLaNVG+BogemJFWhDOcur2c8vHvV2cRMmu7y3t0S/okit+S8KRTofkm8GF91vycl9E2JPhx49tdEwfzVLJzl6y600TM/3Q+NwOUCKGNI8Cf5ImdFv8nHxC5pbSJRdcNHb4j/0/wmg8fXEiIrRbJaYWnpdvBULaq1nO/5WoLPF9/djaa+oLW+Ki+JZ0VO1DHJDQP4JS4jBsAPlRsw8rQivZi3ydRrcynP7PR+Jl5ZFPctMZ1giTQArSLbzOrLe1ALl16a2LP5mULPp5TxO/vCP/w9QSwMEFAAAAAgApLxQXZHJ6JADEAAAiDUAACQAAAByZWZlcmVuY2VzL1JlZmVyZW5jaWFzX2RvX1Byb2pldG8ubWS9W8ty20iW3fMrMqI2VRECQFCkHtZigpZkS9V6NWWXq1aMJJCk0gKQMB6U5FWv+gO6+wPGPYsOT4Q3UzGbXjb/pL6kz70JkKBMy1K1pYVtEPlA3nvuO6+/EwM1Vtnsf5NAy1woMTZJoXIRGpFm5q0qjPjtT38TU9/ttFqHyVQlxexDpo0IMbuaG2h6+2msA5mviWL2MUjsY6gwLTS0LR7TSCbqrYwx2Ygyl7RDovJC1V9yxcnsf4wITJIXuii1KGMpMjXVOb0e6VGkzSSbfaAP0awgKmefQolNYp6aq0jN/kFzU5OJZPbPWGWGvhzoQmLg/1Xuihf2zCf98z4OJ6Yq490l3oKsTPCvNFO5yqZ8xMDERlzovJj9munAgJaEpvA8kJLxJNpJRjqU9vsqmn0sMjqmLEoZueI0FzrEbE1nDw22F6nKYjAkwOHBhNmHqdL5jhh0Nn/7018H620wN5OxkAHmBljJn0llJgX4GWo5SWa/gk0BcxdDFzqd/QrKsLOMR4SI1Lnban33nRi0fQbx7Ke9/h9FWoKPgUgvTGGmJsK0gM6Cz+E4ASFWSJ5+ur932GrR38Kza93fs4Urcjd0cZJzZTF4Jto7wne7O6Lrdl3xOjfPxB6LSaxCXYmLCUoSlIqfYK+GUMjI/t4R+LxKZv+QYndXPP8FG7XxAYb2mbgoijR/5nl0CtekKlHaNdnEy8tRrPNcQ7y8bm9jq9XaOz18Jvy22+ltb3U9f6u70e74rtid/ZpqKSITyOgZE+Nl8spLp6F8N+z47c1hp93pDk2mJzohLr/XKVZh3xK8eCYwuuH4bcffqPnfafBfBoHKc1EYEwGqJBR0RGLynOYCR2y17GyvHnYftPyLTK/5fYu/ECHa1qwJBrFgEKCW0HdtbcPn/J3o4qIcudAP70QWR3I0MMGlVnnFKHvQismsMjrhzUNzm8U161hvXw+OPKzZEWd7L+p1VtPN3UxeZyafm0hm4szkmvgg+tEEOBUXMRFWDQ4k9MeOpikkiZ9zXn0y2D/yXp05vY22s95db8PoDVQo1wT4T8zuJ2GmyLT1Xfdbf8oFTe2tJcx8198R665foUafsoDlvDss2+xDFJSRIXs9+3syocf3wAyqsrMwnRjMVCTZgpkVimKC3E2izJ2YKf/wxjftLQPr4tmDpeH4XjjGqrgwYe6dn/WHxLZhxa4hE4Zd7gawy3x5CYXNZHSbu7uSCLX8a7VOTvt98TIyI0w8NqAXnE8mAkIIq4kfN+599vmSltT87kMDrnVcKQm7LHCe9EZZU5wpzAjL9zoT8FsAf/aRfSMMmM5TnGukySnw7zJLVilRHLmJkZJ5P4Ez8QAtmBl4DLF6h1NCEZ5QjXqMwkQVQ3UN7zbM5iLM6gVrE6isgnpHpNNIj0TbxXq/1bK/4Jrh/EYlcACLf89OLp/rtipUqBxmtBGkHVZJ8M5w3fCSRaZ2bPBAzpxAyBahjfDXNzbWfPHGi//1f6QQuRkXV2Dw55DwQZz0BsdKXIhvWFwoVhFtPJV4U3tEjzcnErwJS1qhQrvU1XxA8oTuCuLdiyKOnhDPDeb2sQlVRCoSyptITy4KIadSR5IEtLhh27Y4NsU8kF7IOWK8zMSQ3kwFBc+aWKVbTIanAh/er4mBNZGH8LfBhUrWxJl9ca5klM+HjzGI35c3a+LHarxQAKKgGe4jntMV/vZ2e0mmOm6naV7526y9ACyB/tbWNtRjKP+OjXdF5fejVYbURhoIKfy2v+G12+tbTnu78/P32+0fttvtXs85WEQdX5zyOHKwyXLQzxEh5sRgSLUwZREabIoYAgLMb1XMz2WG0fFyiBfD1EUK1veFhFEEwHsA7BtsuMLxdazj6ywjQ+Y1tjG4pngdYXojgslVSdG6QohN4a/KvwJQu+OlCNy2/PUmJou3jwPDFsNwGHNwRa6EpdUmD2wbwSRJnFERxxFLDJOQ5ptmqI2TK3i4AoHbG6tNf4iUBjTnQMa+eK6CS0brjctRy7f6NKO2bKPX3XVCrQ4yn8sc+eTsU06JEFwnuUxY+0y/r0Ej1TIUuvwTGSPCeh2nkWrEpbv7u828iVI82iTkJHKF6b6tgG9deFGyzi7O2nNx3LbfvqWBK+c8DvbbjL11dNa/PAPD7RO4E1zKiWLOx7UVrOI8nG5yI/IbJMsx9O/ARPEkUxbTFxbmA1gserPr1pJwrC/NOL/Ua+LYIv+ffpgQ928FqEhWa7gbhYSmg92Z59gYL/TUYldFU5W7/wzItybPXRgU85bxTCUELydYO77f7tlh2IytbiOJuzXyKBD6bYaQAjSW5nwYqIC5nZFgq6ka5sjaFzDfFRk9dJcVUVGlcb06NFqhP9ZhKThHwwWE2d9JB/F83j9eFDoo4YvFj+enJ48QEKVTK0HuMsX3j4TAt7GeeOwvDC99m5tkp36vExIxk81H7sawroa8kQUCh5+wlIxfTxzLpETI0EzONvqdtrPR2egCwj0zMpz5IbBxH7qcdad7F3Y4ca0ou7tQq92+sAEO+FWhucuy23Bxc494v6TO73JSZw/00KSuIng47Q2ZE0Mm6GtJnV+VPhrVoTPrBkArm5yXmQ4d7JAgclMh+CrOrblZcHIXWTIYub653ttqtY4lcixYuufWzBlII2nTt/sEe7beElZdike6FC523fU6G6fUDwaNEzwIxtmAaiYq47omnSSCo8OPeybcPZtw8wEejM2C9uGC9qHl1JCp+SpQtnzyanB46nTabtvrbLobzqsj5/W5RZCoRWxtpbzV6j8H/+85e0WevZD7Ix1rqsiqSpvLrKrVkhOpNXtnaXDhSyisKBF9SssoUhcEN+D650yHIcpkduMqV45s1cqWMb3tMByPZXejJ6Xf3RjhcX0Utrd6XcwLgi3fm1PpLKgcVhQOKwofihjYN6R9Ac6wsznEhouN7sbJVkn67Et0UpqSZTym4I0icE7GKGNByoKY+PKmEdDhDzxyWHka4mwj1bFVxz24H0T0a3X0+KOyYQV+VgncPNtzH+kQ7Oa6t5J/m6iR+tVy82o5R7OSQgWBNKIaENW6kLLtZ6PcqYgS1cHq3w+LHzv4vt9pb6/fET/O5zxO8FGVZi70kLLdYWrkMLSkcEX6nmHH/devDDj8L8QaalEAg/0rY3qi2wmu3dee7ey0/9uf//Ly4PCRyy5fIvGJiy++Lb4MVom5iMjwNQrCy8g1FNH9HRusrKB150o0vwdQMRXMMknpAELnRsFSUQ5QVqaYaeYsTeY7YqyCi+oezw58XLr90kmoUpWEq3O0e+I8kVGkshtvgaqzRL0HNS+GGb3DKzVsMOOpQa4qKw0bF6oMcUFVE6mKUjYkYLNXG8YLRObvwZ2lOpWtZNHCqoRV6IiilmbFbRECuU/4XY5he18wB8ezjwWl8hKLsgRATI14+erQ2TscHJ68WuM7WLpUprJ5M6rdsaxW1yooeQZdCc/vZR9monE4vIaa32Gi53MeRxZseUfB69R2555W+c4lK7V5qXa5p7gWWntD8FDHlGBDjWFtKbfbO8E/Yu/gcE2MZK5oEG/pXqhaEqezT3Rn/bimuUnnUyuqrb/sX6dQBtISezW0b4scpyn2LsoE1oX4+4K0YWDMuDApXbeK56WOeNUheDPJqlva1+65C+ZDnAuuqCEAsvu53/I7KwLobjNxLNR1QVwoOCXhaz5C/vnh2U+VfoGv1GuwZKchAWMJS45/L3Rq4LuLlfBfXV25thLECUsQK+1VJSEPdOsgUrmnamodLh05doFjmtQ6WUWnM6qIdHSTmU8lCB1bxakAaSaOAy5a26zxDfyYuUJit2egoWv2ruLS3krkdyP/H2x8J9R7yPwTPa94nx+8pPoO+0VqbaDXdOX7aaqiHWHsOEwq9Upw007trQOV1a0oX4XbPuYSFrl6dhpVYMeW+XPnyhLlUPk/d0CSky949WTAVqWd0zf7A3Fgyiy6Ef2zw1aL2334tfvZ6AqOHyzafaa+u9Q5wcxiD1ZFv9mqNqJK60r4UC4CwAtTaSAokcyKNAM1OoWLnfqrip+puYKrghIFbiJzuagSUKlOB1A2mWqPLlaQ6ETeBVPiPSj9JH4MmRFDy4ghMYLM8d38tdWc3f3B/rk4/+XED9VE2O+Tm1ljJ0PiDDdTfctEZnKzDMCDlz8MIVuXmfdIxIKu1dU9UDLfBqXGwWuFGUflNWDLbxIi+XdDVbHsHjjZYs6xKpTJ6CRQ9UjsUVvW6ZSESF3ZCfuDQd/pLMNz31UPQ6Wxq3wIKN9IdZqgxPOj3FNr5i1gfAE85J2/Wmau8mlII8KygiMx6lEoizLj+5FAIt4qyohvTix1dVcPB/SSa/bc6FizaLy4j5Ot1v67UqdwAmcv9wftdndDeOL1i7Pn7qN9ckUUetbo6bzVWmbsHS3Qe1dSOgnPxY2K70qJMEXk1R47c6fU7NZotJQSFWVsD7gqH0DGg4TfmEmkuKo31pHyQs+f9i62Dt9s/nz0Mtw8Uj8fnKrplfnD0fn06Cj/Zfvy0CN5vh/8JEBEqRmGZrgL40s8HPJBhnv0faotbFCq4RbXxd1S0Wt0qbFqveJOvhCJOI+8kFlcDffpypUaAUU/kdENAq1cvFEjncjMfcAOK+ppnzdgzoPIZc+mCK6qIqDsrX4za6NLPFO3QsUGQoM8hu6WkAWaFYXvdOoUlO9oGeVu1T+IHILbKec0DisabQeUQ8rncLujQyQ+cc7QsRWcusnS9mDSkXJV3O7YnHdq3mP21xo09+d6+1kz7ITb2kIurdHfcoRobhW3Gw2a1Ym8pTN4IyT6Xix1Yls23Th8QsbaqgkynwKuTiYih6RliuUnE98Pjs/3fxB0I01jckS1qGI+etzf/+G/IN0TJgYsmEjqDxZyajTnTrQcB+FyRqSLqgGl1dq9kHpNvLKV7L1MXkcq49YkKmc/6VlWXAgutPG4ah/Pl9RRVQ3gkpI4KwxrNMEa1zrzv06j2SeqtnAJjxqVqGW9vo+yk5bb7lc0JYaQGih7ooMy53LLPM/b9PxOd9Ojw3uLMkvP3+7SMmfToWGHaXsUsdmq7KfmNoUjQ+ID_RESTRITO_DA_EQUIPE/O951b61aoYKd+eXesQpkovPY9kYXKinl4r8W5OXb2iSu6ENadOKvMoBpHLg5n5aDk7rhwpmUOlSe71wpSReasHW5niTIj9OyQOSILN3m1kwB51fwl5EKiGAnYoK8erT6+YTKbCsrHA1QIsx+R3x/dtr/oWrXtHXL+8H0lW1W4MbFsI79q9ssi+3WDYKKLyKwE91TlIQeuzr8ybJqG/F29oHoNRnlVXSLO/tv/i8JjOy3w5L/H4xjxg63VDmpkc6iPvaEoK23P7uapyqFuTdMtxeuLF0sXZnfql/8QpHn2WAHUWqqAi5IzCtTzWgQU3Wj6ffhUPQcGTi2ZOWYsgAMFIo0yxjcl4Yw73ZpwzwRIP8GUEsDBBQAAAAIAKS8UF2CzkJhfQ0AALwmAAAiAAAAcmVmZXJlbmNlcy9yZWZlcmVuY2lhc19wcm9qZXRvLmJpYr1azXIbuRG++ylwXFdx/iiSkqxKZSmSkrirH4bUWusTC5wBKVgzgzEwQ4ne2qo95QGSPEB2c9jaVPmSrVxyDN9knyTdwJCcGVE2bSc52JofYAB83ej+vga/jLjyvxu6Xu0ZITRLb4UkvyPfXfW6feKQwctu+w/f46uUpyHDN/oRSbJJyH2S3IpUzEWYUrxhcipkRGOfkYCmlPz2w18Ifkh/IJMhdr9N00S9cBxsYIuExYzbQs4clU1gJoqLWDmNZutA94lFqsfs0kAoErGA41/4uvCziMUpXf68/JuAwQiPVUpDc39EYGosXv5MSadDjl+Rhu0eEeozpQRhEam79ZbluZbX0oMEguMYnmvXm4cHDcc7aLTcuvf9s++fPfsyR6deRsdg4BCc/2p9FYCoj+ORVIhQERoHq7abuaew1m3IzHh6m01sX0TOJU3P6WQo/DvOlJPMA/pmbD5chqcCB8sXWyPaHqnGLJFizmK+/Hvsc/oEHMUl75WXPGQBrZG+rdfSjgPJqKqRtl1e+0iEVJKBUBxXR9rhTEhYToQTIeblkAacmrdJAobS10r7yuWwd+5cD6xmy7X2GntufavjCF/ZcSjtmZjrG2e6cA+ESrmj+9hJMC2hg7MxwCg9AUb85Y+hn4WAlCDLn+IZXr4FbMCDjohkc640jAKuQ5ouf5VcvMd/FoxqhOque1BCsFFG8PKq3SanoZjQkFwIGA2giWcEDCwk2EguylCewtaQ0LQCaYfi1A1oW70nCu1YUKrhmUkaOLBqHzo5evXsTazsQfekhFAbfOOBR7n7ZIrChlICPQrchkoKOECLIHvLJRHEB4sufwFM0NECrhJYy4SHPKD6PpPxLu7VLIOTzEM+Ib6IU8knGaChymjMWDpmD6mkY7n2H3SZEewqn0mIDfCl4IiY77i217K9bfDo91aygNaxDT4cpLdMexQXDouduenpSDZlEr/szLQZUhaYrjaXegLwyt4yJ/s2jcIStn3dHrwMdh3RrVMmJVOpZEe4XghbCDNAp8c025N4e61WzSM3TvTvf6AjKjFN7yl22cEN62WkW5XYBQt7WyNDs5P7EH/9WxbXyMA8GDEaqvXrC3gJ93eLGvkqf58ymEiKLcoWuhABC9GhA7oI+ew2JXROeUjRNdKF7rvBDlYegd+AhykylSICv5HMT3Wrmdkim8bbo4BJGxC1PddrOa67d2C5h/Vvvzh0nx+6brNpnZXsoKenXRZMEIPTroJCwKfg8UcQodAOsCdnPMY48EGgvcNDt5JAnp5K0SD7ZYOcUNh6YIBuBdC2UjADRBR8lIgsDQR0gDQCLqSfskhfZxLeTsupOILdGrKtAaKEnFt3Ep7YB97eNrBwm0fcBxAgv4LbJpIVcoximQI7sin3OYYKtWuMrGJWmEQRpoMyTF0GoTCFnHZjHPHrkHFAbWRDEtIPjpl/p4G8sR/lpX6k0x+GNN12TjFe6SgC0FHEi4U6SZVgpOCDiyKv2cUVX9sQaTFq2LDapu22bNdzS/AeU8XIdPlOQfZD8oIxFgKY5G9X6KJbCsxO/4oB/yPCoyRkhRTf6XUID+CeA/o6XDP8SEB10N7NEFX2857JF81yWDbLmQijmWQG9hNjiTPYYPikY6+MdcHvxFTd8Rq5eGwcE7NNTH4BxjBXgIp/R2dMWyVaBZc8hcMEZwuiFgo2wVYvfy2UsmHfiNfaNgkFIyo0Ud3z3KZ5Db540CgZZlgIwaWoO4feiDu8T/ncoJwnSpMxdoLcq/r+o8kUgPbcj8uQmOO1I6mxz3yNukSfYnM2VjTS2fJ/nh2TubGJXZ7NlrS4xaFN9AUAkeRD5+VPuCngetS+APrKFJNzLQSgyVejq8tPSIZeReZ0xURoGgvpr8rib2gK2eklWB7DRJNc0DiDvFQkqq123bVa9VZjK5rbmKrX0ExV93nEVDsiXjlap/PbD3/utInJlABkDlOHJDDzQtRdB+ndPLDsYhVZc0GB1cG+PTabVoD1q142KGi8gQlb0Ebv0VPJAwuWEEMiZwFIRzIy+3MDWQe4PSC2t7/XPNiZ2zcNt8c+j7k90lLYkZp8giEHQ1Q6TAZUQRuYZwgxFW52i4jNEjgVAdQ+Pi5DcT3sX1l113ad+r7dsq7PrW9Geql6UkBmjMNsWydsFUnlwmY2nRilZ/S0cxgE0ylttJqUeo3WBC73JoF70GxAO98/8Jz1oNZm0HE+4NgM+Aikcx5xSM/al+Ipn2Xam/4JTyDGce1yQh6VXm5CHeanDFgIJTE+RW+ETAqYfpjjexUB1IVgBJymtkrYXzGTJuA2p5trblphQjrs8TgTmXa1CJM2kiLdHhkcUDggJHeLQiKHfxC7gzwQ4iIL1O8p7f2+XF7fsz2v7h6WqdJ1mU0aOJHpJyHKOhSPAFVPTpSVr5/k817d7xbGGh/K1pvpFU3wsTLrlo+Rj48TQceBmd8YR///pI+iuHpqJh/OJWwjUSEKZBFe+XCpy0SrADu4av/2xz+dnvU/JYu0nnTsMpzDbX5HQtyPhbLHfwnWGQ1DJhfOBkSrNK4DHpmOJT6DR2xcmMZjTLssQm0K2h/4GPCTgsJnSMKyPILooKBZKgbZKQO5iJYQ+Ytfcqa92hwBA70e7MxRK7DvP5WuquFiEwUCJiE75BoqF5kmMejAsAod8EX+FnZFSXcaZYodc0ma8hAz28cp0+Ie9ZrwzF5t5rXaWv6SItmn8HUZg0Xmgpxe961uf9i/vK4RH8wEr3UlpsgGjgzI7IH5mW4RM5XSFVfdjQ80PxRWNlMuWuLg48IKg/C32sT/90hSHHybo+tSxCp+A4KgxiFSgJtDcEDwupfwh3TP+jUyAeGGL+EplgbzLlGyfAcR5qmM+F6Xrqipb2zQs13UcqnWqpDnelrrlPHsPSTgvejWpjpo2pCrBMhhmsWw3Zl6QU7QfYcCRIxIsBJNjjMe6l592IAz+WQF+v7+3jYSSxMxP2LcybWWAzPjfsiUw1ZzsLQms0wHSxTnYMl8dGuSD23xytAF7puyhxSDRKpJnS67ohmO+4OXuavDjgZ2yUpBBcwxpRB24O8tTwTE/fRpWxSgr7ufBr2BusiDh7oeY0jwDcQ4cQ9MtCtgF9RMyezOFMS2CtUK1uZSUdjE+bVVKD9YpvSjrHszjIUlIWXBIJYqDbLx7imP+bpkMzo7RZml0wKetuBjrHa/m7PwiAjzHgIJnp78iPCvorjPZF5qeCquFJGtiKzL9qgN+famN6xoCXxEzgQgsiDtwdZjokTcQxgC//LtmCq6EQYoBjlMxKEJd7AUBmwrdG71x5wSBie6sIe7PGcJEn1mxQOw+gorlbmHgZgwkgG6o5DwMyDHID557PMEUsDce1LrFxGo74ZApzfsjcjo1aUXsBkxk8ewU9NBB30Hwk5e2xahyH1xV4gK/Va+NA2zB8BMLWIcsYyT0VLrU5KI4OkB2wEr8XlY7e2G1QUoTSFxNeCGIeniAePVHL2A3eucctEbDtvW1jOjnSCK1gNUHKgwMv0YXD7Thyrqqfcm4wls4cFpb+i6jRZxyDcng4omBRtCLkt1+sIzhiwFPad3NIUklWaheEGGzExodRqmyRLVdRfkF2uGPN0UQulWogPMCni6ELOQaQ075SFzAsebN28P+jf7356fBvvn7NuzKza/F1+fj+bn5+rV4V3fQZNtPcAUps4MsL3JkGlC8GIEpPybjEJSIIoZ1nm0jkvFMxNdD1m+m+qiKiNZZFa2GxEqp+V6RTYNXpJ2TMMFZBZFbtgE5LDcdvCpnfJaH/kGQKC1W55Q0KvmdRvLydvpjpUibeI0VHZ+/gtMB+XOeD3wOB/YHORZeIBu6eNlC8d6zGzWGbUc+hiimXN5Zo4TimwSi5pideoXCXAGwiQW4YCdit3qKGWuWK+Ipa1n5qvDcXN2jktTLP2oo/L8C06pjzMB4u5ElMfm8NyOghJEvfUWefSbgpk+fA20bMT/6QRS4JMAFNdbUSmdW8pr5NoUOLqSPoBI0+dr1SoHMLUUYj2NiQILSaZxl+SL4cWo95xgERzf0Qmqr3T99qLde/578KyZnjzMc0bxtxCEzgXXXA+7w7bSOibkaX5itP3oOAAowSFAzWVKi5g129t3vHpj38EKYiU4At9C8qtKbsaUcSGKTM0gWsMGZk+vuPZDEi7fobrRohKP3cAO67KdaYSaRv944TVLd65xljVN0zts4OKsfQsXYVXLoPWKmhkhr6XkUrsQzGl9Qg+MVv8mZnDRqe5+rk8nzgWe2W3f4Unk20p/Waef1YmGNct4wBzPumcU662wmRWfxcCTkyyFTA3E1nBsPYKmehD0QtCxMDkr1AM6q7f5bSV7+TTmKjK/C0lZnOXIYiFZZa9XkWDLCR/w6eWvKVNPHu0VQTz8XBAHIY31SaYOk+SLwVX7Oem/V2t/KqoJDmWJqaUP+KxEUGsjGJ1KFM0PqpkuIMGksL6UIWo6ssI/KfO8RF4vfwQc4QmyUaxBL/+qf7WUB9gPYbjnfjaGFVkiPgW1pkV9y+g9S2QpIIYJqihD9GEl0KeqNBFl7Crq4xWyjcHwCJhJwnwtJ9YSrpjIoSkv/Griadz+A1BLAQIUAxQAAAAIAKS8UF3hhTSZ0wwAAJIiAAAfAAAAAAAAAAAAAACAAQAAAABiaWJsaW9ncmFmaWEvZm9ybWF0YWNhb19hYm50LnB5UEsBAhQDFAAAAAgApLxQXaXmcWUXOQAAuqkAABEAAAAAAAAAAAAAAIABEA0AAHNyYy9wcm9qZXRvX2Z2LnB5UEsBAhQDFAAAAAgApLxQXSfBsDqhNQAAWJ8AABUAAAAAAAAAAAAAAIABVkYAAHNyYy9yZWZpbmFtZW50b19mdi5weVBLAQIUAxQAAAAIAKS8UF3pD4A9OQwAAEgcAAATAAAAAAAAAAAAAACAASp8AABzcmMvcmVsYXRvcmlvX2Z2LnB5UEsBAhQDFAAAAAgApLxQXc4DSnvXAwAA8QcAABMAAAAAAAAAAAAAAIABlIgAAGNvbmZpZy9wcm9qZXRvLmpzb25QSwECFAMUAAAACACkvFBdyp6odi0BAAAMAgAAFgAAAAAAAAAAAAAAgAGcjAAAY29uZmlnL21vZHVsb19jZWMuanNvblBLAQIUAxQAAAAIAKS8UF2D6LrEwQAAAE4BAAAYAAAAAAAAAAAAAACAAf2NAABjb25maWcvaW52ZXJzb3JfY2VjLmpzb25QSwECFAMUAAAACACkvFBdUT4jYFwCAADjBgAAGgAAAAAAAAAAAAAAgAH0jgAAZGF0YS9yYXcvZm9udGVzX3B2ZGFxLmpzb25QSwECFAMUAAAACACkvFBdpRmJ01IXAADvRgAAGQAAAAAAAAAAAAAAgAGIkQAAZG9jcy9zZWNvZXNfY2FwaXR1bG8uanNvblBLAQIUAxQAAAAIAKS8UF34IbkalRwAAPxOAAAbAAAAAAAAAAAAAACAARGpAABzY3JpcHRzL2dlcmFyX2RvY3VtZW50b3MucHlQSwECFAMUAAAACACkvFBdaoKmaucsAADOdwAAHwAAAAAAAAAAAAAAgAHfxQAAc2NyaXB0cy9nZXJhcl9kb3NzaWVfdGVjbmljby5weVBLAQIUAxQAAAAIAKS8UF1GaQfhfRAAAEREAAAjAAAAAAAAAAAAAACAAQPzAAByZWZlcmVuY2VzL3JlZmVyZW5jaWFzX3Byb2pldG8uanNvblBLAQIUAxQAAAAIAKS8UF2RyeiQAxAAAIg1AAAkAAAAAAAAAAAAAACAAcEDAQByZWZlcmVuY2VzL1JlZmVyZW5jaWFzX2RvX1Byb2pldG8ubWRQSwECFAMUAAAACACkvFBdgs5CYX0NAAC8JgAAIgAAAAAAAAAAAAAAgAEGFAEAcmVmZXJlbmNlcy9yZWZlcmVuY2lhc19wcm9qZXRvLmJpYlBLBQYAAAAADgAOAPMDAADDIQEAAAA=")
assert hashlib.sha256(support).hexdigest()=="f1b4ca6d1f5a9604f028923038a25d2416e0b8e0e479fdd559588a0f533b0767"
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src/projeto_fv.py").exists()),None)
if ROOT is None:ROOT=Path.cwd()/"Projeto_FV_PVDAQ_2107_2024"
update=not (ROOT/"config/projeto.json").exists()
if not update:update=json.loads((ROOT/"config/projeto.json").read_text()).get("version")!="1.2.2"
if update:
    with zipfile.ZipFile(io.BytesIO(support)) as archive:
        for item in archive.infolist():
            dest=(ROOT/item.filename).resolve()
            assert dest.is_relative_to(ROOT.resolve())
            if dest.exists():
                backup=ROOT/"backup_antes_v1_2"/item.filename
                backup.parent.mkdir(parents=True,exist_ok=True)
                if not backup.exists():backup.write_bytes(dest.read_bytes())
            dest.parent.mkdir(parents=True,exist_ok=True)
            dest.write_bytes(archive.read(item.filename))
sys.path.insert(0,str(ROOT/"src"))
import projeto_fv as fv,refinamento_fv as rf,relatorio_fv as rel
import importlib
for module in [fv,rf,rel]:importlib.reload(module)
import numpy as np,pandas as pd,pvlib
from IPython.display import display,Markdown,HTML
display(HTML("<style>.text_cell_render,.jp-RenderedHTMLCommon{font:20px/1.65 Arial,sans-serif}.dataframe{font-size:17px}td,th{padding:8px}summary{cursor:pointer}</style>"))
CFG=fv.load_config(ROOT)
EXECUTAR_INVERSAO_EXPLORATORIA=True
GERAR_DOCX=True
REUTILIZAR_FIGURAS_VERIFICADAS=True
TEMA_VISUALIZACAO="escuro"
SALVAR_RESULTADOS_NO_DRIVE=False
PASTA_RESULTADOS_ID="ID_RESTRITO_DA_EQUIPE"
print("Versão",CFG["version"],"| NASA POWER fora da cadeia principal")
display(pd.DataFrame([{"biblioteca":k,"versão":importlib.metadata.version(k)} for k in requirements]))


### 0.1 Auditoria e cálculo da cadeia principal

<details>
<summary>Descrição e método</summary>

O filtro diurno usa NOAA ≥10° para preservar o denominador da auditoria; SPA calcula a geometria no meio dos intervalos. Carimbos são tratados como início de intervalos locais America/Los_Angeles, interpretando PST8PDT dos metadados. Posição representativa em +7,5 minutos para as médias de 15 minutos e +30 minutos para a inversão horária; convenção temporal documentada, sem otimização por defasagem da geração. As médias de 15 minutos dos canais de 5 minutos exigem três amostras. O fator empírico secundário usa somente abril.

</details>

In [ ]:
CTX=fv.complete_pipeline(ROOT,make_plots=False,include_nasa=False,reverse=EXECUTAR_INVERSAO_EXPLORATORIA)
D,A=CTX["data"],CTX["model_a"]
display(pd.DataFrame([CTX["audit"]]))
display(fv.data_status())
print("Cobertura conjunta diurna:",100*CTX["audit"]["available_daylight_15m"]/CTX["audit"]["expected_daylight_15m"],"%")


### 0.2 Exportações e diagnósticos

<details>
<summary>Descrição e método</summary>

Dados e filtros idênticos geram os dois temas. Cada figura possui fonte, legenda, natureza da evidência e subseção no catálogo. Os resultados são exibidos nas subseções que os utilizam.

</details>

In [ ]:
rf.generate_all(CTX,reuse_checked=REUTILIZAR_FIGURAS_VERIFICADAS)
rel.coverage(CTX)
print("Figuras:",len(CTX["figures"]),"| arquivos finais PDF/PNG:",4*len(CTX["figures"]))
print("Saídas:",ROOT/"outputs")


# 1 Fundamentos e caracterização do recurso solar

<details>
<summary>Descrição e método</summary>

Caracterizar e representar o recurso solar. **Responsável:** Samuel.

</details>

## 1.1 Geometria solar e irradiância extraterrestre <a id="sec_1_1"></a>

<details>
<summary>Descrição e método</summary>

Calcular a posição solar com SPA e a irradiância extraterrestre com Spencer. O azimute é contado a partir do norte no sentido horário.

**Entradas:** Latitude, longitude, altitude e horário dos intervalos.

$$\cos\theta_z=\sin\varphi\sin\delta+\cos\varphi\cos\delta\cos\omega,\qquad G_{0h}=G_{0n}\max(\cos\theta_z,0).$$

**Evidência:** Cálculo físico com coordenadas documentadas. **Limite:** Não há observação independente de ângulo solar; a geometria pode ser calculada.

**Saídas:** Geometria solar e quadro de convenções. **Referências:** R03, R04, R05.

</details>

In [ ]:
display(D[["zenith","azimuth","aoi_deg","dni_extra_Wm2","extraterrestrial_horizontal_Wm2"]].iloc[40:45].round(3))
rf.show(CTX,['G01_recurso_geometria', 'F01_plano_angulos'],TEMA_VISUALIZACAO)

## 1.2 Grandezas radiométricas e componentes da radiação <a id="sec_1_2"></a>

<details>
<summary>Descrição e método</summary>

Irradiância é potência por área; irradiação é sua integral no tempo. GHI é horizontal, DNI é normal aos raios solares e DHI é difusa horizontal. POA se refere ao plano do arranjo.

**Entradas:** POA local medida; componentes reconstruídas quando a inversão estiver ativa.

$$GHI=DNI\max(\cos\theta_z,0)+DHI,\qquad H=\sum_i G_i\Delta t_i.$$

**Evidência:** Medição POA e estimativa exploratória das componentes. **Limite:** Sem GHI/DNI/DHI locais medidos; reconstrução não equivale a observação.

**Saídas:** Quadro das grandezas e componentes estimadas. **Referências:** R01, R14, R18, R29.

</details>

In [ ]:
print("POA é medida. GHI/DNI/DHI disponíveis são estimativas da inversão.")
rf.show(CTX,['F02_componentes_POA', 'G23_componentes_estimadas'],TEMA_VISUALIZACAO)

## 1.3 Modulação atmosférica e variabilidade espaço temporal <a id="sec_1_3"></a>

<details>
<summary>Descrição e método</summary>

Nuvens, aerossóis e vapor d’água modificam o recurso solar. A série permite descrever a variabilidade intradiária e entre dias; a seleção dos dias ilustrativos usa apenas POA e cobertura.

**Entradas:** Série de POA, intervalo de integração e critérios de cobertura.

$$V_{\Delta G}=\frac{\operatorname{média}(|G_i-G_{i-1}|)}{\operatorname{média}(G_i)}.$$

**Evidência:** Descrição observacional de variabilidade. **Limite:** Não se atribuem causas atmosféricas isoladas nem climatologia anual a um trimestre.

**Saídas:** Critério dos dias ilustrativos e irradiação pareada. **Referências:** R01, R25.

</details>

In [ ]:
display(CTX["analytics_tables"]["criterio_dias_ilustrativos"].iloc[[0,-1]])
rf.show(CTX,['G02_variabilidade_recurso'],TEMA_VISUALIZACAO)

## 1.4 Medição e estimativa do recurso solar <a id="sec_1_4"></a>

<details>
<summary>Descrição e método</summary>

Inspecionar integridade, unidades, lacunas, simultaneidade e limites físicos. Medições de superfície, satélites, reanálises e atlas têm representatividades diferentes; os produtos em grade são contextualizados, sem uso na cadeia principal.

**Entradas:** Arquivos PVDAQ, metadados, canais medidos e resoluções de 5/15 minutos.

$$C=100\,N_{\mathrm{pares}}/N_{\mathrm{esperados}}.$$

**Evidência:** Auditoria de disponibilidade e documentação. **Limite:** Disponibilidade elevada não certifica sensores. Altura do anemômetro e alguns metadados permanecem incompletos.

**Saídas:** Situação dos dados, proveniência e cobertura. **Referências:** R01, R02, R21, R22, R23, R25, R26.

</details>

In [ ]:
display(pd.DataFrame([CTX["audit"]])); display(fv.data_status())
rf.show(CTX,['FL01_fluxo_projeto', 'G19_consistencia_medicoes'],TEMA_VISUALIZACAO)

# 2 Condições ambientais e de entorno da conversão fotovoltaica

<details>
<summary>Descrição e método</summary>

Relacionar ambiente e instalação à irradiância e à temperatura. **Responsável:** Vannessa e Thais.

</details>

## 2.1 Condições térmicas <a id="sec_2_1"></a>

<details>
<summary>Descrição e método</summary>

Temperatura ambiente, vento e trocas de calor condicionam a operação do módulo. As distribuições observadas e o cenário de menor ventilação ilustram a passagem do ambiente à modelagem térmica da seção 3.2.

**Entradas:** Temperatura do ar e vento locais; POA medida.

$$T_{\mathrm{op}}=T_a+\frac{G_{\mathrm{ef}}}{U_0+U_1v}.$$

**Evidência:** Medições ambientais e temperatura estimada. **Limite:** U₀ e U₁ genéricos; T operacional é aproximação de célula, sem validação térmica independente.

**Saídas:** Estatística ambiental e sensibilidade de ventilação. **Referências:** R07.

</details>

In [ ]:
display(CTX["analytics_tables"]["estatistica_descritiva"].iloc[:4])
rf.show(CTX,['G03_condicoes_ambientais', 'G04_temperatura_estimada'],TEMA_VISUALIZACAO)

## 2.2 Albedo e ambiente radiativo da superfície <a id="sec_2_2"></a>

<details>
<summary>Descrição e método</summary>

O albedo expressa refletância do entorno. No exercício de transposição, varia-se 0,10/0,20/0,40 mantendo as componentes reconstruídas na referência 0,20. O arranjo real é monofacial; bifacialidade exige caracterização traseira adicional.

**Entradas:** Albedo assumido e componentes estimadas da inversão.

$$G_{\mathrm{solo}}=\rho_g\,GHI\,\frac{1-\cos\beta}{2}.$$

**Evidência:** Cenário de sensibilidade. **Limite:** Expressão isotrópica para reflexão do solo. Não há albedo medido; a leitura POA principal não recebe uma segunda contribuição refletida.

**Saídas:** Sensibilidade de albedo e quadro de hipóteses. **Referências:** R06, R14, R29.

</details>

In [ ]:
display(CTX["environment_scenarios"]) if len(CTX["environment_scenarios"]) else print("Albedo: cenário omitido sem inversão.")
rf.show(CTX,['G22_albedo', 'F02_componentes_POA'],TEMA_VISUALIZACAO)

## 2.3 Sombreamento sujidade e obstruções locais <a id="sec_2_3"></a>

<details>
<summary>Descrição e método</summary>

A sujeira pode atenuar a luz incidente e diferir entre módulos e sensor POA. Os cenários são hipóteses de atenuação adicional à leitura. Sombreamento parcial pode envolver mismatch e diodos de bypass, que um redutor uniforme não reproduz.

**Entradas:** Cenários de atenuação adicional de 0%, 2% e 5%; alertas operacionais.

$$G_{\mathrm{ef}}=(1-s)G_{\mathrm{POA}}.$$

**Evidência:** Sensibilidade hipotética e fundamentos. **Limite:** Sem histórico de limpeza e mapa de sombras; o resíduo não identifica a perda real de sujidade ou sombra.

**Saídas:** Sensibilidade e alertas sem atribuição causal automática. **Referências:** R28, R25.

</details>

In [ ]:
display(CTX["environment_sensitivity"]); display(CTX["operational_flags"].head())
rf.show(CTX,['G20_sujidade_ventilacao'],TEMA_VISUALIZACAO)

## 2.4 Geometria de instalação e exposição do arranjo fotovoltaico <a id="sec_2_4"></a>

<details>
<summary>Descrição e método</summary>

A instalação define exposição e incidência. A inclinação é medida em relação à horizontal. Espaçamento entre fileiras e exposição traseira precisam de geometria adicional; em um exemplo de sombra coplanar, uma altura h gera projeção horizontal h/tan(altura solar).

**Entradas:** Inclinação de 25° e azimute de 180° documentados; fachadas hipotéticas.

$$\cos\theta_i=\cos\theta_z\cos\beta+\sin\theta_z\sin\beta\cos(\gamma_s-\gamma_a),\quad \ell=h/\tan\alpha_s.$$

**Evidência:** Geometria documentada e extensão didática. **Limite:** A expressão de projeção é um exemplo plano sem dimensão de fileira real; não calcula o sombreamento específico da usina.

**Saídas:** Convenções geométricas e limite de dados de espaçamento. **Referências:** R03, R09, R29.

</details>

In [ ]:
display(pd.DataFrame([CFG["site"]]))
rf.show(CTX,['F01_plano_angulos', 'F04_geometrias_BIPV'],TEMA_VISUALIZACAO)

# 3 Modelagem da conversão fotovoltaica

<details>
<summary>Descrição e método</summary>

Transformar as entradas em temperatura, potência e energia. **Responsável:** Gabriela e Giulia.

</details>

## 3.1 Transposição para o plano do arranjo <a id="sec_3_1"></a>

<details>
<summary>Descrição e método</summary>

A cadeia usual transpõe GHI/DNI/DHI para POA. Como a instalação já mede POA, o modelo principal começa dessa observação. A inversão opcional usa Driesse 2024, Erbs–Driesse e Perez–Driesse para exercícios de orientação/albedo.

**Entradas:** Componentes reconstruídas, posição solar e geometria.

$$G_{\mathrm{POA}}=DNI\max(\cos\theta_i,0)+G_{\mathrm{céu}}+G_{\mathrm{solo}}.$$

**Evidência:** Medição direta e inversão exploratória. **Limite:** Há possibilidade de soluções não únicas; o fechamento de POA é teste numérico, sem validação independente das componentes.

**Saídas:** Diagnóstico horário da inversão e critérios de exclusão. **Referências:** R06, R14, R15, R16, R18.

</details>

In [ ]:
display(pd.DataFrame([CTX.get("reverse_summary",{"executed":False})]))
rf.show(CTX,['G23_componentes_estimadas'],TEMA_VISUALIZACAO)

## 3.2 Modelagem da temperatura operacional <a id="sec_3_2"></a>

<details>
<summary>Descrição e método</summary>

A temperatura estimada entra no modelo elétrico. U₀=25 W/(m² K) e U₁=6,84 W s/(m³ K) são parâmetros genéricos de referência; a sensibilidade de ventilação usa 15 e 3, respectivamente.

**Entradas:** POA, temperatura do ar, vento e coeficientes Faiman.

$$T_{\mathrm{op}}=T_a+G_{\mathrm{ef}}/(U_0+U_1v).$$

**Evidência:** Modelo térmico. **Limite:** Ausência de temperatura de módulo/célula e altura de vento suficientemente documentada.

**Saídas:** Temperatura estimada e registro dos coeficientes. **Referências:** R07.

</details>

In [ ]:
display(A.loc[D.paired_daylight,["temp_cell_proxy_C"]].describe())
rf.show(CTX,['G04_temperatura_estimada'],TEMA_VISUALIZACAO)

## 3.3 Modelagem elétrica do módulo fotovoltaico <a id="sec_3_3"></a>

<details>
<summary>Descrição e método</summary>

O modelo de diodo único fornece curvas I–V, P–V e o ponto de máxima potência. Compare irradiâncias a 25 °C e temperaturas a 1.000 W/m². Registre divergências de classificação e parâmetros entre CEC e metadados.

**Entradas:** Parâmetros CEC, irradiância efetiva e temperatura estimada.

$$I=I_L-I_0[\exp((V+IR_s)/a)-1]-(V+IR_s)/R_{sh},\quad P=VI.$$

**Evidência:** Modelo elétrico e curvas calculadas. **Limite:** Sem varreduras I–V experimentais da instalação; as curvas não constituem validação do módulo.

**Saídas:** Curvas calculadas e identificação de MPP. **Referências:** R08, R09, R10.

</details>

In [ ]:
print("Curvas CEC calculadas; não são varreduras experimentais.")
rf.show(CTX,['G05_curvas_irradiancia', 'G06_curvas_temperatura'],TEMA_VISUALIZACAO)

## 3.4 Modelagem do arranjo fotovoltaico em corrente contínua <a id="sec_3_4"></a>

<details>
<summary>Descrição e método</summary>

Em uma associação ideal, as tensões somam em série e as correntes em paralelo. O exemplo 20 módulos × 6 strings por grupo corresponde a 120 módulos, mas não confirma o cabeamento real ou a ligação aos MPPTs.

**Entradas:** MPP do módulo, 2.880 módulos, 24 grupos/inversores e perdas DC assumidas.

$$V_a=N_sV_m,\quad I_a=N_pI_m,\quad P_a=N_sN_pP_m,\quad P_{\mathrm{DC}}=P_a\prod_j(1-\ell_j).$$

**Evidência:** Modelo agregado e equivalente didático. **Limite:** Strings/MPPTs reais incompletos; V×I disponível não foi adotado como potência DC observada certificada.

**Saídas:** Tensão, corrente e potência do equivalente em CSV. **Referências:** R08, R10, R13.

</details>

In [ ]:
display(CTX["analytics_tables"]["arranjo_equivalente_diurno"].head().round(3))
rf.show(CTX,['F03_arranjo_eletrico'],TEMA_VISUALIZACAO)

## 3.5 Conversão de CC para CA e modelagem do inversor <a id="sec_3_5"></a>

<details>
<summary>Descrição e método</summary>

A curva PVWatts estima conversão e clipping. O limite de referência é 30 kW por inversor; 27,6 kW permanece como sensibilidade documentada. Verificar a faixa real de MPPT exige topologia e configuração de campo adicionais.

**Entradas:** Potência DC calculada, eficiência e limites assumidos do inversor.

$$P_{\mathrm{AC}}=\min[\eta(P_{\mathrm{DC}})P_{\mathrm{DC}},P_{\mathrm{AC,max}}],\quad P_{\mathrm{dc0,inv}}=P_{\mathrm{AC,max}}/\eta_{\mathrm{nom}}.$$

**Evidência:** Modelo de conversão e sensibilidade. **Limite:** A simulação agregada não verifica todas as restrições reais de tensão/MPPT nem confirma o limite instalado.

**Saídas:** Curva de eficiência, clipping calculado e sensibilidade AC. **Referências:** R11, R13.

</details>

In [ ]:
display(CTX["capacity_sensitivity"].round(4))
rf.show(CTX,['G07_inversor'],TEMA_VISUALIZACAO)

## 3.6 Integração temporal e estimativa de energia <a id="sec_3_6"></a>

<details>
<summary>Descrição e método</summary>

Integre potência média por duração de cada intervalo. Compare séries e energias nas mesmas janelas. Apresente energia mensal e trimestral observada completa separadamente dos intervalos diurnos pareados.

**Entradas:** Potência média em intervalos de 15 minutos e máscaras temporais.

$$E=\sum_i P_i\Delta t_i,\qquad \Delta t=0,25\ \mathrm{h}.$$

**Evidência:** Integração calculada e geração observada. **Limite:** O recorte é trimestral, não uma medição anual; energia pareada não equivale ao total integral do dia.

**Saídas:** Energia diária/mensal, cobertura e totais integrais separados. **Referências:** R12.

</details>

In [ ]:
display(CTX["analytics_tables"]["energia_e_indicadores_mensais"].round(3))
rf.show(CTX,['G08_series_potencia', 'G09_energia_diaria', 'G10_energia_mensal'],TEMA_VISUALIZACAO)

# 4 Avaliação de desempenho perdas e validação

<details>
<summary>Descrição e método</summary>

Quantificar desempenho e confrontar a geração calculada com observações. **Responsável:** Júlia Ribeiro e Lucas Simão.

</details>

## 4.1 Balanço energético e cadeia de perdas <a id="sec_4_1"></a>

<details>
<summary>Descrição e método</summary>

Integre a cadeia óptica, térmica, elétrica DC, conversão e limite AC. A diferença entre estágios pertence ao modelo; um desvio do medidor pode envolver fenômenos e incertezas não identificados.

**Entradas:** Estágios do modelo e alertas de potência sob irradiância elevada.

$$\Delta E_j=E_j-E_{j+1},\qquad \sum_j\Delta E_j=E_1-E_{\mathrm{final}}.$$

**Evidência:** Decomposição calculada. **Limite:** Não se identificam perdas ópticas/operacionais reais isoladas sem medições adicionais; ganhos sob frio podem ocorrer.

**Saídas:** Energia por estágio e alertas operacionais. **Referências:** R08, R11, R12, R25.

</details>

In [ ]:
display(CTX["energy_stages"].round(3))
rf.show(CTX,['G11_cadeia_energetica'],TEMA_VISUALIZACAO)

## 4.2 Produtividades de referência do arranjo e final <a id="sec_4_2"></a>

<details>
<summary>Descrição e método</summary>

Yr é o recurso em horas equivalentes; Ya representa a energia DC por capacidade; Yf representa energia AC por capacidade. HSP/PSH corresponde ao mesmo conceito de recurso quando plano e janela são os mesmos.

**Entradas:** Irradiação POA, energia DC/AC e potência declarada de 893 kWp.

$$Y_r=H_{\mathrm{POA}}/G_{\mathrm{ref}},\quad Y_a=E_{\mathrm{DC}}/P_0,\quad Y_f=E_{\mathrm{AC}}/P_0,\quad G_{\mathrm{ref}}=1\ \mathrm{kW/m^2}.$$

**Evidência:** Indicadores sobre observações e modelo. **Limite:** Ya é modelada; HSP da janela pareada não é duração de brilho solar nem total integral diário.

**Saídas:** Yr, Ya, Yf e HSP na janela definida. **Referências:** R12, R30.

</details>

In [ ]:
display(CTX["yields"].round(4))
rf.show(CTX,['G12_produtividades_PR', 'G21_HSP_POA_pareada'],TEMA_VISUALIZACAO)

## 4.3 Razão de desempenho e decomposição normalizada das perdas <a id="sec_4_3"></a>

<details>
<summary>Descrição e método</summary>

Compare PR observado e modelado com irradiação e energia da mesma janela. Lc e Ls decompõem o modelo em perdas de captura e sistema. Informe a normalização e o papel da potência declarada.

**Entradas:** Produtividades calculadas na mesma máscara temporal.

$$PR=Y_f/Y_r,\quad L_c=Y_r-Y_a,\quad L_s=Y_a-Y_f.$$

**Evidência:** PR observado pareado e perdas calculadas. **Limite:** Não combinar energia integral do trimestre com POA parcial; perdas calculadas não são identificação experimental de cada causa.

**Saídas:** PR, Lc, Ls e janela de integração. **Referências:** R12, R30.

</details>

In [ ]:
display(CTX["yields"][["PR_model","PR_observed_paired","Lc_model_h","Ls_model_h"]].round(5))
rf.show(CTX,['G12_produtividades_PR'],TEMA_VISUALIZACAO)

## 4.4 Validação de modelos de geração fotovoltaica <a id="sec_4_4"></a>

<details>
<summary>Descrição e método</summary>

Avalie viés, magnitude do erro, correlação e determinação. Erro=simulado−observado. MBE, MAE e RMSE são em kW; nMBE/nRMSE são divididos pela média observada dos respectivos pares. O modelo físico é separado do fator empírico ajustado somente em abril.

**Entradas:** Mesmos pares de potência AC medida e simulada.

$$MBE=\bar e,\quad MAE=\overline{|e|},\quad RMSE=\sqrt{\overline{e^2}},\quad R^2=1-\frac{\sum e_i^2}{\sum(y_i-\bar y)^2}.$$

**Evidência:** Comparação observacional da saída AC. **Limite:** Junho é teste temporal separado, não instalação independente. Correlação elevada não prova exatidão; nenhum limiar universal de aceitação foi adotado.

**Saídas:** Métricas gerais, por mês/hora/POA, resíduos e consistência de medidores. **Referências:** R27, R25.

</details>

In [ ]:
display(CTX["metrics"].round(5)); display(CTX["analytics_tables"]["metricas_por_mes"].round(4)); display(pd.DataFrame([CTX["meter_inverter_consistency"]]).round(4))
rf.show(CTX,['G13_validacao', 'G14_residuos_temporais', 'G15_residuos_potencia', 'G16_distribuicao_residuos', 'G17_metricas_mensais', 'G18_vies_horario', 'G19_consistencia_medicoes'],TEMA_VISUALIZACAO)

## 4.5 Avaliação de desempenho de sistemas integrados à edificação <a id="sec_4_5"></a>

<details>
<summary>Descrição e método</summary>

Explore fachadas sul/leste/oeste e menor ventilação. O exemplo separado de 100 m² semitransparentes usa transmissão solar 0,25, reflexão 0,12, absorção 0,63 e transmissão visível 0,35. As propriedades não caracterizam o módulo real.

**Entradas:** Componentes reconstruídas, meteorologia local e coeficientes BIPV assumidos.

$$P_e=\eta GA,\quad Q_{\mathrm{abs}}=(\alpha-\eta)GA,\quad Q_{\mathrm{in}}=\tau_sGA+f_{\mathrm{in}}Q_{\mathrm{abs}},\quad SHGC_{\mathrm{eq}}=\tau_s+f_{\mathrm{in}}(\alpha-\eta).$$

**Evidência:** Cenário geométrico e balanço hipotético. **Limite:** Somente horas elegíveis da inversão; sem fachada medida, SHGC certificado, cálculo de lux ou balanço térmico completo do edifício.

**Saídas:** Produtividade/PR de cenários e conservação da partição solar. **Referências:** R14, R19, R20.

</details>

In [ ]:
display(CTX["bipv_comparison"].round(4)) if "bipv_comparison" in CTX else print("BIPV omitido sem inversão.")
rf.show(CTX,['G24_BIPV_orientacoes', 'G25_BIPV_balanco', 'F04_geometrias_BIPV'],TEMA_VISUALIZACAO)

# 5 Referências

<details>
<summary>Descrição e método</summary>

Inventário das fontes dos dados, modelos e limites. O levantamento teórico por tópico está no notebook bibliografia/02_Fundamentacao_Teorica_por_Subsecao.ipynb e em bibliografia/outputs/. Não substitui leitura crítica e redação final. Fontes NASA pertencem ao histórico/comparação de produtos, sem entrada na cadeia principal.

</details>

In [ ]:
REFERENCIAS=json.loads((ROOT/"references/referencias_projeto.json").read_text())
for r in REFERENCIAS:
    display(Markdown("**"+r["id"]+"** — "+r["citation"]+"\n\n"+r["url"]+"\n\nUso: "+r["role"]))


# 6 Documentos e relatório

<details>
<summary>Descrição e método</summary>

Seis DOCX: tabelas com três traços horizontais, quadros com grade, coleção completa de 30 gráficos e diagramas, arquivos separados de ilustrações e fluxograma e dossiê técnico por seção. Papel A4, margens superior/esquerda de 3 cm e inferior/direita de 2 cm; Times New Roman 12 em todo o texto nativo, com títulos de tabelas e figuras centralizados. O relatório HTML possui tema alternável e descrições recolhíveis. Conferência visual de Word requer renderizador disponível.

</details>

In [ ]:
if GERAR_DOCX:
    python_documentos=os.environ.get("CODEX_PRIMARY_RUNTIME_PYTHON",sys.executable)
    subprocess.run([python_documentos,str(ROOT/"scripts/gerar_documentos.py")],check=True)
    subprocess.run([python_documentos,str(ROOT/"scripts/gerar_dossie_tecnico.py")],check=True)
    display(pd.DataFrame([{"arquivo":str(p.relative_to(ROOT/"outputs")),"bytes":p.stat().st_size} for p in sorted((ROOT/"outputs").rglob("*.docx"))]))
print(rel.report(CTX))
print("Matriz:",ROOT/"outputs/relatorios/Matriz_de_Cobertura_do_Capitulo.md")


# 7 Pacote de resultados e exportação opcional ao Drive

<details>
<summary>Descrição e método</summary>

Ative SALVAR_RESULTADOS_NO_DRIVE para uma execução pessoal no Colab. A célula cria uma pasta datada com a hierarquia preservada; autenticação é pessoal, sem tokens no notebook. O pacote contém todas as saídas locais.

</details>

In [ ]:
ZIP_RESULTADOS=fv.zip_results(CTX)
print("Resultados compactados:",ZIP_RESULTADOS)
try:
    IS_COLAB=importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:IS_COLAB=False
if SALVAR_RESULTADOS_NO_DRIVE and IS_COLAB:
    display(fv.sync_outputs_drive(CTX,PASTA_RESULTADOS_ID))
elif SALVAR_RESULTADOS_NO_DRIVE:
    print("Exportação configurada para Colab; arquivos locais prontos.")
else:
    print("Saídas locais concluídas. Exportação pessoal ao Drive desativada por padrão.")
